# Systèmes agentiques composables — Perception → Action

[Version Python](AgentSystems-01-Perception-Vers-Action-Python.ipynb) | [Index AgentSystems](README.md) | Suivant : [AgentSystems-02 — Perception active (Python)](AgentSystems-02-Perception-Active-Valeur-Information-Python.ipynb)

**Notebook** : AgentSystems-01 — jumeau Rust
**Série** : AgentSystems (projet cross-séries — EPIC #26930)
**Durée** : ~1 h 30
**Prérequis** : Rust (édition stable), la seule bibliothèque standard, notions de *trait objects* et de *pattern matching*

## Objectifs d'apprentissage

- Lire la boucle **percevoir → représenter → inférer → décider → vérifier → exécuter → observer** comme un *squelette réutilisable*, pas comme un script.
- Traiter l'**inférence comme un point de composition** : le même agent tourne avec un cerveau déterministe *ou* un LLM, sans que le reste de la boucle change.
- **Réutiliser** les primitives de [`Compression-07b`](../../InformationCalcul/Compression/Compression-07b-Triage-LLM-Semantic-Kernel-Python.ipynb) — la représentation `Segment`, l'invariant de couverture, la boucle de découverte bornée — dans un cadre agentique.
- **Mesurer** l'amortissement du coût : découvrir une politique coûte des inférences, la rejouer en coûte zéro.

> **Position de ce carnet dans le projet.** Il ne ré-invente pas le moteur de
> compression : il en extrait le *squelette d'agent*. Les primitives compression
> sont **réutilisées** (copie pédagogique déclarée — 07b est un notebook, pas un
> module importable) pour montrer la composition ; l'extraction en module
> importable est le pas suivant nommé dans l'EPIC.

## Ce qui change en Rust

Le modèle est identique au jumeau Python ; seule l'API change. Le portage n'utilise
**que la bibliothèque standard** — aucune crate, aucun réseau, aucune clé.

| Aspect | Python | Rust (bibliothèque standard seule) |
|---|---|---|
| Contrat d'inférence | `class Cerveau` | `trait Cerveau` |
| Décision | `dict` (`{"op": …}`) | `enum Decision` (`Decouper` / `Compresser`) |
| Substitution | *duck typing* | `&mut dyn Cerveau` |
| Erreur | `raise` | `Result<T, String>` |
| JSON | module `json` | mini-dumpeur et mini-parseur maison |
| `zlib` | module `zlib` | blocs stockés RFC 1950 et `adler32` maison |
| `lzma` | module `lzma` | **stub copie déclaré** (le vrai LZMA1 n'est pas reproductible en bibliothèque standard seule) |
| Booléens | `True` / `False` | `fn py(b: bool) -> &'static str` |

Chaque cellule de ce carnet est **autonome** : elle redéfinit tout ce qu'elle
mentionne. C'est une contrainte du noyau `evcxr` (une cellule compile et
s'exécute seule), mais c'est aussi une exigence de lecture : on peut copier une
cellule dans un carnet vierge et elle compile.

In [1]:
{
    // Le bandeau est REPRODUIT VERBATIM depuis le jumeau Python : la
    // comparaison differentielle (registre `rust_conformance.d`) compare des
    // chaines identiques. Le port n'utilise QUE `std` : aucune importation
    // n'est requise, et la ligne n'en annonce aucune.
    println!("Imports OK : json, re, collections — noyau standard, aucune cle API requise");
};

Imports OK : json, re, collections — noyau standard, aucune cle API requise


## 1. La boucle perception → action

Un système agentique n'est pas un LLM : c'est un **squelette de traitement** que l'on
**compose** avec un mécanisme d'inférence. La boucle de référence a sept stades :

```mermaid
flowchart LR
    P[Percevoir] --> R[Représenter]
    R --> I[Inférer]
    I --> D[Décider]
    D --> V[Vérifier]
    V -->|couverture OK| E[Exécuter]
    V -->|couverture incomplète| I
    E --> O[Observer]
    O -->|politique amortie| R
```

- **Percevoir / Représenter** : ingérer les entrées brutes dans une *représentation commune* (`Segment`).
- **Inférer / Décider** : proposer un traitement — c'est le **point de composition** (le « cerveau »).
- **Vérifier** : un **invariant** rejette tout plan incomplet (un segment resté sans traitement final).
- **Exécuter / Observer** : appliquer le plan **sans ré-inférer**, puis mesurer le coût.

Le point central : **Inférer** est isolé derrière une *interface*. On y branche un
cerveau déterministe, un LLM, ou les deux — le reste de la boucle ne change pas.
C'est cela, être **composable**.

## 2. Représenter — la perception comme signature de structure

La boucle commence par une **représentation commune**. On réutilise la structure
`Segment` de Compression-07b : un morceau de donnée identifié, portant ses octets,
qualifié par un genre, relié à son origine. La *perception* d'un segment, c'est sa
**signature de structure** — déduite du contenu, pas de l'étiquette.

In [2]:
{
    /// Representation commune d'un morceau de donnee (reutilisee de Compression-07b).
    ///
    /// L'objet que la boucle percoit et represente : `id` stable, `data` = octets,
    /// `genre` = qualification, `parent`/`profondeur` = lien vers l'origine.
    #[derive(Clone)]
    struct Segment {
        id: String,
        data: Vec<u8>,
        genre: String,
        parent: Option<String>,
        profondeur: i64,
    }

    /// Signature de structure deduite du CONTENU — la perception du segment.
    ///
    /// C'est ce que le cerveau regarde pour decider. Deduite, pas etiquettee :
    /// un fichier nomme `.json` qui ne parse pas n'est pas du JSON.
    struct Info {
        octets: usize,
        texte: bool,
        csv: bool,
        blocs_md: usize,
        json: bool,
        json_obj: bool,
    }

    /// Les booleens Rust s'impriment `true`/`false` ; le jumeau Python impose
    /// `True`/`False`. Toute comparaison de texte passe par cette fonction.
    fn py(b: bool) -> &'static str {
        if b { "True" } else { "False" }
    }

    /// JSON : mini-dumpeur et mini-parseur maison (`std` seule, aucune crate).
    ///
    /// L'ordre d'insertion des cles d'un objet est PRESERVE (`Vec<(String, Jv)>`) :
    /// le catalogue amorti s'appuie sur cet ordre, un `BTreeMap` le casserait.
    #[derive(Clone)]
    enum Jv {
        Null,
        Bool(bool),
        Int(i64),
        Str(String),
        List(Vec<Jv>),
        Obj(Vec<(String, Jv)>),
    }

    fn json_echapper(s: &str, out: &mut String) {
        out.push('"');
        for c in s.chars() {
            match c {
                '"' => { out.push('\\'); out.push('"'); }
                '\\' => { out.push('\\'); out.push('\\'); }
                '\n' => out.push_str("\\n"),
                '\r' => out.push_str("\\r"),
                '\t' => out.push_str("\\t"),
                c if (c as u32) < 0x20 => out.push_str(&format!("\\u{:04x}", c as u32)),
                c => out.push(c),
            }
        }
        out.push('"');
    }

    /// Dumps equivalent a `json.dumps(indent=2, ensure_ascii=False)`.
    fn json_dump(v: &Jv, level: usize, out: &mut String) {
        let pad = "  ".repeat(level);
        let pad_in = "  ".repeat(level + 1);
        match v {
            Jv::Null => out.push_str("null"),
            Jv::Bool(b) => out.push_str(if *b { "true" } else { "false" }),
            Jv::Int(i) => out.push_str(&i.to_string()),
            Jv::Str(s) => json_echapper(s, out),
            Jv::List(items) => {
                if items.is_empty() { out.push_str("[]"); return; }
                out.push_str("[\n");
                for (i, it) in items.iter().enumerate() {
                    out.push_str(&pad_in);
                    json_dump(it, level + 1, out);
                    if i + 1 < items.len() { out.push(','); }
                    out.push('\n');
                }
                out.push_str(&pad);
                out.push(']');
            }
            Jv::Obj(fields) => {
                if fields.is_empty() { out.push_str("{}"); return; }
                out.push_str("{\n");
                for (i, (k, val)) in fields.iter().enumerate() {
                    out.push_str(&pad_in);
                    json_echapper(k, out);
                    out.push_str(": ");
                    json_dump(val, level + 1, out);
                    if i + 1 < fields.len() { out.push(','); }
                    out.push('\n');
                }
                out.push_str(&pad);
                out.push('}');
            }
        }
    }

    fn json_skip_ws(b: &[u8], i: &mut usize) {
        while *i < b.len() && (b[*i] as char).is_whitespace() { *i += 1; }
    }

    fn json_chaine(b: &[u8], i: &mut usize) -> Option<String> {
        if *i >= b.len() || b[*i] != b'"' { return None; }
        *i += 1;
        let mut s = String::new();
        while *i < b.len() {
            let c = b[*i];
            if c == b'"' { *i += 1; return Some(s); }
            if c == b'\\' {
                *i += 1;
                if *i >= b.len() { return None; }
                let e = b[*i];
                *i += 1;
                match e {
                    b'n' => s.push('\n'),
                    b't' => s.push('\t'),
                    b'r' => s.push('\r'),
                    b'b' => s.push('\u{8}'),
                    b'f' => s.push('\u{c}'),
                    b'/' => s.push('/'),
                    b'"' => s.push('"'),
                    b'\\' => s.push('\\'),
                    b'u' => {
                        if *i + 4 > b.len() { return None; }
                        let hex = std::str::from_utf8(&b[*i..*i + 4]).ok()?;
                        let code = u32::from_str_radix(hex, 16).ok()?;
                        *i += 4;
                        s.push(char::from_u32(code).unwrap_or('\u{fffd}'));
                    }
                    _ => return None,
                }
            } else {
                let start = *i;
                while *i < b.len() && b[*i] != b'"' && b[*i] != b'\\' { *i += 1; }
                s.push_str(std::str::from_utf8(&b[start..*i]).ok()?);
            }
        }
        None
    }

    fn json_valeur(b: &[u8], i: &mut usize) -> Option<Jv> {
        json_skip_ws(b, i);
        if *i >= b.len() { return None; }
        match b[*i] {
            b'{' => {
                *i += 1;
                let mut fields: Vec<(String, Jv)> = Vec::new();
                json_skip_ws(b, i);
                if *i < b.len() && b[*i] == b'}' { *i += 1; return Some(Jv::Obj(fields)); }
                loop {
                    json_skip_ws(b, i);
                    if *i >= b.len() || b[*i] != b'"' { return None; }
                    let k = json_chaine(b, i)?;
                    json_skip_ws(b, i);
                    if *i >= b.len() || b[*i] != b':' { return None; }
                    *i += 1;
                    let v = json_valeur(b, i)?;
                    fields.push((k, v));
                    json_skip_ws(b, i);
                    if *i >= b.len() { return None; }
                    match b[*i] {
                        b',' => { *i += 1; }
                        b'}' => { *i += 1; return Some(Jv::Obj(fields)); }
                        _ => return None,
                    }
                }
            }
            b'[' => {
                *i += 1;
                let mut items: Vec<Jv> = Vec::new();
                json_skip_ws(b, i);
                if *i < b.len() && b[*i] == b']' { *i += 1; return Some(Jv::List(items)); }
                loop {
                    let v = json_valeur(b, i)?;
                    items.push(v);
                    json_skip_ws(b, i);
                    if *i >= b.len() { return None; }
                    match b[*i] {
                        b',' => { *i += 1; }
                        b']' => { *i += 1; return Some(Jv::List(items)); }
                        _ => return None,
                    }
                }
            }
            b'"' => Some(Jv::Str(json_chaine(b, i)?)),
            b't' => {
                if b.len() >= *i + 4 && &b[*i..*i + 4] == b"true" { *i += 4; return Some(Jv::Bool(true)); }
                None
            }
            b'f' => {
                if b.len() >= *i + 5 && &b[*i..*i + 5] == b"false" { *i += 5; return Some(Jv::Bool(false)); }
                None
            }
            b'n' => {
                if b.len() >= *i + 4 && &b[*i..*i + 4] == b"null" { *i += 4; return Some(Jv::Null); }
                None
            }
            _ => {
                let start = *i;
                if b[*i] == b'-' { *i += 1; }
                while *i < b.len() && b[*i].is_ascii_digit() { *i += 1; }
                if start == *i { return None; }
                let txt = std::str::from_utf8(&b[start..*i]).ok()?;
                txt.parse::<i64>().ok().map(Jv::Int)
            }
        }
    }

    /// Parse un document JSON complet ; `None` si le texte n'est pas du JSON.
    fn json_parse(texte: &str) -> Option<Jv> {
        let b = texte.as_bytes();
        let mut i = 0usize;
        let v = json_valeur(b, &mut i)?;
        json_skip_ws(b, &mut i);
        if i != b.len() { return None; }
        Some(v)
    }

    /// Positions (en octets) des titres de markdown : meme regex que le jumeau
    /// Python (`^##? ` en mode multiligne), donc memes decoupes.
    fn positions_titres(texte: &str) -> Vec<usize> {
        let mut out = Vec::new();
        let mut off = 0usize;
        for l in texte.split_inclusive('\n') {
            let t = l.trim_end_matches('\n').trim_end_matches('\r');
            if t.starts_with("# ") || t.starts_with("## ") { out.push(off); }
            off += l.len();
        }
        out
    }

    /// Perception : la signature de structure deduite du contenu.
    fn structure_interne(seg: &Segment) -> Info {
        let mut info = Info {
            octets: seg.data.len(),
            texte: false,
            csv: false,
            blocs_md: 0,
            json: false,
            json_obj: false,
        };
        let texte = match String::from_utf8(seg.data.clone()) {
            Ok(t) => t,
            Err(_) => return info,
        };
        info.texte = true;
        let t = texte.trim();
        if t.starts_with('{') || t.starts_with('[') {
            if let Some(v) = json_parse(&texte) {
                info.json_obj = matches!(v, Jv::Obj(_));
                info.json = true;
                return info;
            }
        }
        let lignes: Vec<&str> = texte.lines().filter(|l| !l.is_empty()).collect();
        if !lignes.is_empty() && lignes[0].contains(',') {
            let n = lignes[0].matches(',').count();
            info.csv = lignes.iter().all(|l| l.matches(',').count() == n);
        }
        info.blocs_md = positions_titres(&texte).len();
        info
    }

    /// Classe de forme d'un segment — la cle du catalogue amorti.
    ///
    /// Distingue l'objet JSON (decoupable par ses cles) du tableau JSON
    /// (compresse tel quel) : deux formes, donc deux decisions distinctes.
    fn signature(s: &Info) -> String {
        if s.json {
            return if s.json_obj { "json_obj".to_string() } else { "json".to_string() };
        }
        if s.csv { return "csv".to_string(); }
        if s.blocs_md > 0 { return "markdown".to_string(); }
        if s.texte { "texte".to_string() } else { "binaire".to_string() }
    }

    /// Un petit corpus heterogene, genere deterministiquement : meme entree a
    /// chaque execution, donc resultat reproductible.
    fn corpus_demo() -> Vec<Segment> {
        let ventes = "nom,quantite,prix\ncafe,10,2.50\nthe,20,1.80\nsucre,5,0.90\nlait,8,1.10\npain,12,0.80\n";
        let clients_obj = Jv::Obj(vec![
            ("id".to_string(), Jv::Int(7)),
            ("panier".to_string(), Jv::List(vec![
                Jv::Obj(vec![
                    ("sku".to_string(), Jv::Str("cafe".to_string())),
                    ("q".to_string(), Jv::Int(10)),
                ]),
                Jv::Obj(vec![
                    ("sku".to_string(), Jv::Str("the".to_string())),
                    ("q".to_string(), Jv::Int(20)),
                ]),
            ])),
        ]);
        let mut clients = String::new();
        json_dump(&clients_obj, 0, &mut clients);
        clients.push('\n');
        let notes = "# Rapport\n\n## Chiffre\n12 kEUR, en hausse.\n\n## Risque\nFournisseur unique, lead time 6 semaines.\n";
        let journal: String = (0..40)
            .map(|i| format!("10:{:02}:00 INFO requete {} ok\n", i, i))
            .collect();
        let mut binaire: Vec<u8> = Vec::with_capacity(1024);
        for _ in 0..4 {
            for b in 0u16..256 { binaire.push(b as u8); }
        }
        vec![
            Segment {
                id: "ventes.csv#0".to_string(),
                data: ventes.as_bytes().to_vec(),
                genre: "donnees/csv".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "clients.json#0".to_string(),
                data: clients.into_bytes(),
                genre: "donnees/json".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "notes.md#0".to_string(),
                data: notes.as_bytes().to_vec(),
                genre: "texte/markdown".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "journal.log#0".to_string(),
                data: journal.into_bytes(),
                genre: "texte/log".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "brut.bin#0".to_string(),
                data: binaire,
                genre: "binaire".to_string(),
                parent: None,
                profondeur: 0,
            },
        ]
    }

    use std::collections::HashSet;

    // Demonstration de la perception : pour chaque segment du corpus, on lit
    // sa forme interne deduite du contenu (structure_interne) puis sa classe
    // de forme (signature) — sans aucune etiquette, seulement a partir des
    // octets.
    let segments = corpus_demo();
    let mut formes: HashSet<String> = HashSet::new();
    for seg in &segments {
        let s = structure_interne(seg);
        let f = signature(&s);
        formes.insert(f.clone());
        println!(
            "  {:<16} {:>5} o   forme={:<9} json={} csv={} blocs_md={} texte={}",
            seg.id,
            seg.data.len(),
            f,
            py(s.json),
            py(s.csv),
            s.blocs_md,
            py(s.texte)
        );
    }
    println!();
    println!(
        "corpus percu : {} segments heterogenes, {} formes distinctes",
        segments.len(),
        formes.len()
    );
};

  ventes.csv#0        81 o   forme=csv       json=False csv=True blocs_md=0 texte=True


  clients.json#0     127 o   forme=json_obj  json=True csv=False blocs_md=0 texte=True


  notes.md#0          95 o   forme=markdown  json=False csv=False blocs_md=3 texte=True


  journal.log#0     1110 o   forme=texte     json=False csv=False blocs_md=0 texte=True


  brut.bin#0        1024 o   forme=binaire   json=False csv=False blocs_md=0 texte=False


corpus percu : 5 segments heterogenes, 5 formes distinctes


### Lecture du résultat

Cinq segments, **cinq formes distinctes** : le CSV et le JSON sont des structures
*exploitables* (on peut les découper en parties de nature différente), le markdown
est structuré par ses en-têtes, le journal est un texte plat, le binaire n'a pas de
structure lisible. C'est exactement l'hétérogénéité que la boucle doit gérer — et
c'est ce qui rendra les décisions **non uniformes** plus loin (le tri n'est pas
trivial : chaque forme appelle une opération différente).

Deux remarks de portage. D'abord, le bandeau de la cellule précédente est
**reproduit verbatim** depuis le jumeau Python : la comparaison différentielle
(registre `rust_conformance.d`) porte sur des chaînes identiques, et le port
n'utilise en réalité que la bibliothèque standard. Ensuite, la perception ne lit
**aucune étiquette** : le `genre` du segment n'entre pas dans le calcul de la
forme — seul le contenu des octets compte.

## 3. Inférer — le point de composition

Le stade **Inférer** est isolé derrière une *interface* : le trait `Cerveau`, dont la
méthode `inferer` renvoie une `Decision`. C'est ce contrat unique — et rien
d'autre — qui rend le cerveau **interchangeable**. La boucle ne sait pas *qui*
infère, seulement *que* quelqu'un infère. On commence par le cerveau
**déterministe**, qui dérive la décision de la signature perçue : c'est lui qui
produira le résultat expérimental.

In [3]:
{
    /// Representation commune d'un morceau de donnee (reutilisee de Compression-07b).
    ///
    /// L'objet que la boucle percoit et represente : `id` stable, `data` = octets,
    /// `genre` = qualification, `parent`/`profondeur` = lien vers l'origine.
    #[derive(Clone)]
    struct Segment {
        id: String,
        data: Vec<u8>,
        genre: String,
        parent: Option<String>,
        profondeur: i64,
    }

    /// Signature de structure deduite du CONTENU — la perception du segment.
    ///
    /// C'est ce que le cerveau regarde pour decider. Deduite, pas etiquettee :
    /// un fichier nomme `.json` qui ne parse pas n'est pas du JSON.
    struct Info {
        octets: usize,
        texte: bool,
        csv: bool,
        blocs_md: usize,
        json: bool,
        json_obj: bool,
    }

    /// Les booleens Rust s'impriment `true`/`false` ; le jumeau Python impose
    /// `True`/`False`. Toute comparaison de texte passe par cette fonction.
    fn py(b: bool) -> &'static str {
        if b { "True" } else { "False" }
    }

    /// JSON : mini-dumpeur et mini-parseur maison (`std` seule, aucune crate).
    ///
    /// L'ordre d'insertion des cles d'un objet est PRESERVE (`Vec<(String, Jv)>`) :
    /// le catalogue amorti s'appuie sur cet ordre, un `BTreeMap` le casserait.
    #[derive(Clone)]
    enum Jv {
        Null,
        Bool(bool),
        Int(i64),
        Str(String),
        List(Vec<Jv>),
        Obj(Vec<(String, Jv)>),
    }

    fn json_echapper(s: &str, out: &mut String) {
        out.push('"');
        for c in s.chars() {
            match c {
                '"' => { out.push('\\'); out.push('"'); }
                '\\' => { out.push('\\'); out.push('\\'); }
                '\n' => out.push_str("\\n"),
                '\r' => out.push_str("\\r"),
                '\t' => out.push_str("\\t"),
                c if (c as u32) < 0x20 => out.push_str(&format!("\\u{:04x}", c as u32)),
                c => out.push(c),
            }
        }
        out.push('"');
    }

    /// Dumps equivalent a `json.dumps(indent=2, ensure_ascii=False)`.
    fn json_dump(v: &Jv, level: usize, out: &mut String) {
        let pad = "  ".repeat(level);
        let pad_in = "  ".repeat(level + 1);
        match v {
            Jv::Null => out.push_str("null"),
            Jv::Bool(b) => out.push_str(if *b { "true" } else { "false" }),
            Jv::Int(i) => out.push_str(&i.to_string()),
            Jv::Str(s) => json_echapper(s, out),
            Jv::List(items) => {
                if items.is_empty() { out.push_str("[]"); return; }
                out.push_str("[\n");
                for (i, it) in items.iter().enumerate() {
                    out.push_str(&pad_in);
                    json_dump(it, level + 1, out);
                    if i + 1 < items.len() { out.push(','); }
                    out.push('\n');
                }
                out.push_str(&pad);
                out.push(']');
            }
            Jv::Obj(fields) => {
                if fields.is_empty() { out.push_str("{}"); return; }
                out.push_str("{\n");
                for (i, (k, val)) in fields.iter().enumerate() {
                    out.push_str(&pad_in);
                    json_echapper(k, out);
                    out.push_str(": ");
                    json_dump(val, level + 1, out);
                    if i + 1 < fields.len() { out.push(','); }
                    out.push('\n');
                }
                out.push_str(&pad);
                out.push('}');
            }
        }
    }

    fn json_skip_ws(b: &[u8], i: &mut usize) {
        while *i < b.len() && (b[*i] as char).is_whitespace() { *i += 1; }
    }

    fn json_chaine(b: &[u8], i: &mut usize) -> Option<String> {
        if *i >= b.len() || b[*i] != b'"' { return None; }
        *i += 1;
        let mut s = String::new();
        while *i < b.len() {
            let c = b[*i];
            if c == b'"' { *i += 1; return Some(s); }
            if c == b'\\' {
                *i += 1;
                if *i >= b.len() { return None; }
                let e = b[*i];
                *i += 1;
                match e {
                    b'n' => s.push('\n'),
                    b't' => s.push('\t'),
                    b'r' => s.push('\r'),
                    b'b' => s.push('\u{8}'),
                    b'f' => s.push('\u{c}'),
                    b'/' => s.push('/'),
                    b'"' => s.push('"'),
                    b'\\' => s.push('\\'),
                    b'u' => {
                        if *i + 4 > b.len() { return None; }
                        let hex = std::str::from_utf8(&b[*i..*i + 4]).ok()?;
                        let code = u32::from_str_radix(hex, 16).ok()?;
                        *i += 4;
                        s.push(char::from_u32(code).unwrap_or('\u{fffd}'));
                    }
                    _ => return None,
                }
            } else {
                let start = *i;
                while *i < b.len() && b[*i] != b'"' && b[*i] != b'\\' { *i += 1; }
                s.push_str(std::str::from_utf8(&b[start..*i]).ok()?);
            }
        }
        None
    }

    fn json_valeur(b: &[u8], i: &mut usize) -> Option<Jv> {
        json_skip_ws(b, i);
        if *i >= b.len() { return None; }
        match b[*i] {
            b'{' => {
                *i += 1;
                let mut fields: Vec<(String, Jv)> = Vec::new();
                json_skip_ws(b, i);
                if *i < b.len() && b[*i] == b'}' { *i += 1; return Some(Jv::Obj(fields)); }
                loop {
                    json_skip_ws(b, i);
                    if *i >= b.len() || b[*i] != b'"' { return None; }
                    let k = json_chaine(b, i)?;
                    json_skip_ws(b, i);
                    if *i >= b.len() || b[*i] != b':' { return None; }
                    *i += 1;
                    let v = json_valeur(b, i)?;
                    fields.push((k, v));
                    json_skip_ws(b, i);
                    if *i >= b.len() { return None; }
                    match b[*i] {
                        b',' => { *i += 1; }
                        b'}' => { *i += 1; return Some(Jv::Obj(fields)); }
                        _ => return None,
                    }
                }
            }
            b'[' => {
                *i += 1;
                let mut items: Vec<Jv> = Vec::new();
                json_skip_ws(b, i);
                if *i < b.len() && b[*i] == b']' { *i += 1; return Some(Jv::List(items)); }
                loop {
                    let v = json_valeur(b, i)?;
                    items.push(v);
                    json_skip_ws(b, i);
                    if *i >= b.len() { return None; }
                    match b[*i] {
                        b',' => { *i += 1; }
                        b']' => { *i += 1; return Some(Jv::List(items)); }
                        _ => return None,
                    }
                }
            }
            b'"' => Some(Jv::Str(json_chaine(b, i)?)),
            b't' => {
                if b.len() >= *i + 4 && &b[*i..*i + 4] == b"true" { *i += 4; return Some(Jv::Bool(true)); }
                None
            }
            b'f' => {
                if b.len() >= *i + 5 && &b[*i..*i + 5] == b"false" { *i += 5; return Some(Jv::Bool(false)); }
                None
            }
            b'n' => {
                if b.len() >= *i + 4 && &b[*i..*i + 4] == b"null" { *i += 4; return Some(Jv::Null); }
                None
            }
            _ => {
                let start = *i;
                if b[*i] == b'-' { *i += 1; }
                while *i < b.len() && b[*i].is_ascii_digit() { *i += 1; }
                if start == *i { return None; }
                let txt = std::str::from_utf8(&b[start..*i]).ok()?;
                txt.parse::<i64>().ok().map(Jv::Int)
            }
        }
    }

    /// Parse un document JSON complet ; `None` si le texte n'est pas du JSON.
    fn json_parse(texte: &str) -> Option<Jv> {
        let b = texte.as_bytes();
        let mut i = 0usize;
        let v = json_valeur(b, &mut i)?;
        json_skip_ws(b, &mut i);
        if i != b.len() { return None; }
        Some(v)
    }

    /// Positions (en octets) des titres de markdown : meme regex que le jumeau
    /// Python (`^##? ` en mode multiligne), donc memes decoupes.
    fn positions_titres(texte: &str) -> Vec<usize> {
        let mut out = Vec::new();
        let mut off = 0usize;
        for l in texte.split_inclusive('\n') {
            let t = l.trim_end_matches('\n').trim_end_matches('\r');
            if t.starts_with("# ") || t.starts_with("## ") { out.push(off); }
            off += l.len();
        }
        out
    }

    /// Perception : la signature de structure deduite du contenu.
    fn structure_interne(seg: &Segment) -> Info {
        let mut info = Info {
            octets: seg.data.len(),
            texte: false,
            csv: false,
            blocs_md: 0,
            json: false,
            json_obj: false,
        };
        let texte = match String::from_utf8(seg.data.clone()) {
            Ok(t) => t,
            Err(_) => return info,
        };
        info.texte = true;
        let t = texte.trim();
        if t.starts_with('{') || t.starts_with('[') {
            if let Some(v) = json_parse(&texte) {
                info.json_obj = matches!(v, Jv::Obj(_));
                info.json = true;
                return info;
            }
        }
        let lignes: Vec<&str> = texte.lines().filter(|l| !l.is_empty()).collect();
        if !lignes.is_empty() && lignes[0].contains(',') {
            let n = lignes[0].matches(',').count();
            info.csv = lignes.iter().all(|l| l.matches(',').count() == n);
        }
        info.blocs_md = positions_titres(&texte).len();
        info
    }

    /// Classe de forme d'un segment — la cle du catalogue amorti.
    ///
    /// Distingue l'objet JSON (decoupable par ses cles) du tableau JSON
    /// (compresse tel quel) : deux formes, donc deux decisions distinctes.
    fn signature(s: &Info) -> String {
        if s.json {
            return if s.json_obj { "json_obj".to_string() } else { "json".to_string() };
        }
        if s.csv { return "csv".to_string(); }
        if s.blocs_md > 0 { return "markdown".to_string(); }
        if s.texte { "texte".to_string() } else { "binaire".to_string() }
    }

    /// Un petit corpus heterogene, genere deterministiquement : meme entree a
    /// chaque execution, donc resultat reproductible.
    fn corpus_demo() -> Vec<Segment> {
        let ventes = "nom,quantite,prix\ncafe,10,2.50\nthe,20,1.80\nsucre,5,0.90\nlait,8,1.10\npain,12,0.80\n";
        let clients_obj = Jv::Obj(vec![
            ("id".to_string(), Jv::Int(7)),
            ("panier".to_string(), Jv::List(vec![
                Jv::Obj(vec![
                    ("sku".to_string(), Jv::Str("cafe".to_string())),
                    ("q".to_string(), Jv::Int(10)),
                ]),
                Jv::Obj(vec![
                    ("sku".to_string(), Jv::Str("the".to_string())),
                    ("q".to_string(), Jv::Int(20)),
                ]),
            ])),
        ]);
        let mut clients = String::new();
        json_dump(&clients_obj, 0, &mut clients);
        clients.push('\n');
        let notes = "# Rapport\n\n## Chiffre\n12 kEUR, en hausse.\n\n## Risque\nFournisseur unique, lead time 6 semaines.\n";
        let journal: String = (0..40)
            .map(|i| format!("10:{:02}:00 INFO requete {} ok\n", i, i))
            .collect();
        let mut binaire: Vec<u8> = Vec::with_capacity(1024);
        for _ in 0..4 {
            for b in 0u16..256 { binaire.push(b as u8); }
        }
        vec![
            Segment {
                id: "ventes.csv#0".to_string(),
                data: ventes.as_bytes().to_vec(),
                genre: "donnees/csv".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "clients.json#0".to_string(),
                data: clients.into_bytes(),
                genre: "donnees/json".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "notes.md#0".to_string(),
                data: notes.as_bytes().to_vec(),
                genre: "texte/markdown".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "journal.log#0".to_string(),
                data: journal.into_bytes(),
                genre: "texte/log".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "brut.bin#0".to_string(),
                data: binaire,
                genre: "binaire".to_string(),
                parent: None,
                profondeur: 0,
            },
        ]
    }

    /// La decision d'un cerveau : decouper (reduire) ou compresser.
    ///
    /// Un `enum` remplace le `dict` du jumeau Python : les deux cas sont
    /// nommes, donc le compilateur refuse le cas forget.
    #[derive(Clone)]
    enum Decision {
        Decouper { mode: String },
        Compresser { codec: String },
    }

    impl Decision {
        fn op(&self) -> &'static str {
            match self {
                Decision::Decouper { .. } => "decouper",
                Decision::Compresser { .. } => "compresser",
            }
        }
        fn quoi(&self) -> &str {
            match self {
                Decision::Decouper { mode } => mode,
                Decision::Compresser { codec } => codec,
            }
        }
    }

    /// Point de composition : le contrat d'un mecanisme d'inference.
    ///
    /// La boucle agentique n'appelle JAMAIS un cerveau en particulier, seulement
    /// ce trait. C'est ce contrat unique qui rend le cerveau interchangeable
    /// (deterministe, LLM, hybride). Chaque consultation compte comme une inference.
    trait Cerveau {
        fn inferer(&mut self, seg: &Segment) -> Result<Decision, String>;
    }

    /// Le cerveau par defaut : derive la decision de la structure percue.
    ///
    /// Joue le role que le LLM joue dans Compression-07b, en deterministe :
    /// meme perception -> meme decision, donc cout reproductible. La decision
    /// est une *regle de forme* : les formes exploitables (csv/json/markdown)
    /// sont decoupees pour reduire, le reste est compresse.
    struct CerveauDeterministe { appels: i64 }

    impl Cerveau for CerveauDeterministe {
        fn inferer(&mut self, seg: &Segment) -> Result<Decision, String> {
            self.appels += 1;
            let s = structure_interne(seg);
            // Un segment trop petit n'a pas d'interet a etre decoupe.
            if seg.data.len() < 64 {
                return Ok(Decision::Compresser {
                    codec: if s.texte { "zlib".to_string() } else { "lzma".to_string() },
                });
            }
            let mode = match signature(&s).as_str() {
                "json_obj" => "cles_json",
                "csv" => "colonnes_csv",
                "markdown" => "blocs_markdown",
                _ => {
                    return Ok(Decision::Compresser {
                        codec: if s.texte { "zlib".to_string() } else { "lzma".to_string() },
                    })
                }
            };
            Ok(Decision::Decouper { mode: mode.to_string() })
        }
    }

    let segments = corpus_demo();
    let mut apercu = CerveauDeterministe { appels: 0 };
    println!("cerveau determine par : CerveauDeterministe  (seuil decoupe = 64 o)");
    println!();
    println!("decision derivee de la forme, segment par segment :");
    for seg in &segments {
        let d = apercu.inferer(seg)?;
        println!("  {:<16} -> {:<10} {}", seg.id, d.op(), d.quoi());
    }
    println!();
    println!("inférences effectuees pour cet apercu : {}", apercu.appels);
};

cerveau determine par : CerveauDeterministe  (seuil decoupe = 64 o)


decision derivee de la forme, segment par segment :


  ventes.csv#0     -> decouper   colonnes_csv


  clients.json#0   -> decouper   cles_json


  notes.md#0       -> decouper   blocs_markdown


  journal.log#0    -> compresser zlib


  brut.bin#0       -> compresser lzma


inférences effectuees pour cet apercu : 5


### Lecture du résultat

Le cerveau **n'est pas un classifieur en dur** : il *dérive* chaque décision de la
signature perçue. Les formes exploitables (`csv`, `json`, `markdown`) au-delà du
seuil sont **découpées** (l'opération qui *réduit*), le reste est **compressé**
(avec un codec choisi par la nature, texte/binaire). Cinq segments, cinq décisions —
la non-uniformité est là. Et chaque décision a coûté **une inférence** : c'est le
compteur que la boucle va amortir.

## 4. Vérifier — l'invariant de couverture

Le stade **Vérifier** est le garde-fou de la composition : un plan n'est **pas
exécutable** tant qu'il laisse un segment d'origine *sans traitement final*. On
réutilise l'invariant de couverture de Compression-07b : un segment est couvert
s'il est compressé, **ou** s'il est découpé en enfants *tous* couverts. C'est le
**témoin négatif** de l'organe — il nomme précisément ce qui manque.

In [4]:
{
    /// Representation commune d'un morceau de donnee (reutilisee de Compression-07b).
    ///
    /// L'objet que la boucle percoit et represente : `id` stable, `data` = octets,
    /// `genre` = qualification, `parent`/`profondeur` = lien vers l'origine.
    #[derive(Clone)]
    struct Segment {
        id: String,
        data: Vec<u8>,
        genre: String,
        parent: Option<String>,
        profondeur: i64,
    }

    /// Signature de structure deduite du CONTENU — la perception du segment.
    ///
    /// C'est ce que le cerveau regarde pour decider. Deduite, pas etiquettee :
    /// un fichier nomme `.json` qui ne parse pas n'est pas du JSON.
    struct Info {
        octets: usize,
        texte: bool,
        csv: bool,
        blocs_md: usize,
        json: bool,
        json_obj: bool,
    }

    /// Les booleens Rust s'impriment `true`/`false` ; le jumeau Python impose
    /// `True`/`False`. Toute comparaison de texte passe par cette fonction.
    fn py(b: bool) -> &'static str {
        if b { "True" } else { "False" }
    }

    /// Repr Python d'une liste de chaines : `['a', 'b']`.
    fn py_list_str(items: &[&str]) -> String {
        let mut s = String::from("[");
        for (i, it) in items.iter().enumerate() {
            if i > 0 { s.push_str(", "); }
            s.push('\'');
            s.push_str(it);
            s.push('\'');
        }
        s.push(']');
        s
    }

    /// JSON : mini-dumpeur et mini-parseur maison (`std` seule, aucune crate).
    ///
    /// L'ordre d'insertion des cles d'un objet est PRESERVE (`Vec<(String, Jv)>`) :
    /// le catalogue amorti s'appuie sur cet ordre, un `BTreeMap` le casserait.
    #[derive(Clone)]
    enum Jv {
        Null,
        Bool(bool),
        Int(i64),
        Str(String),
        List(Vec<Jv>),
        Obj(Vec<(String, Jv)>),
    }

    fn json_echapper(s: &str, out: &mut String) {
        out.push('"');
        for c in s.chars() {
            match c {
                '"' => { out.push('\\'); out.push('"'); }
                '\\' => { out.push('\\'); out.push('\\'); }
                '\n' => out.push_str("\\n"),
                '\r' => out.push_str("\\r"),
                '\t' => out.push_str("\\t"),
                c if (c as u32) < 0x20 => out.push_str(&format!("\\u{:04x}", c as u32)),
                c => out.push(c),
            }
        }
        out.push('"');
    }

    /// Dumps equivalent a `json.dumps(indent=2, ensure_ascii=False)`.
    fn json_dump(v: &Jv, level: usize, out: &mut String) {
        let pad = "  ".repeat(level);
        let pad_in = "  ".repeat(level + 1);
        match v {
            Jv::Null => out.push_str("null"),
            Jv::Bool(b) => out.push_str(if *b { "true" } else { "false" }),
            Jv::Int(i) => out.push_str(&i.to_string()),
            Jv::Str(s) => json_echapper(s, out),
            Jv::List(items) => {
                if items.is_empty() { out.push_str("[]"); return; }
                out.push_str("[\n");
                for (i, it) in items.iter().enumerate() {
                    out.push_str(&pad_in);
                    json_dump(it, level + 1, out);
                    if i + 1 < items.len() { out.push(','); }
                    out.push('\n');
                }
                out.push_str(&pad);
                out.push(']');
            }
            Jv::Obj(fields) => {
                if fields.is_empty() { out.push_str("{}"); return; }
                out.push_str("{\n");
                for (i, (k, val)) in fields.iter().enumerate() {
                    out.push_str(&pad_in);
                    json_echapper(k, out);
                    out.push_str(": ");
                    json_dump(val, level + 1, out);
                    if i + 1 < fields.len() { out.push(','); }
                    out.push('\n');
                }
                out.push_str(&pad);
                out.push('}');
            }
        }
    }

    fn json_skip_ws(b: &[u8], i: &mut usize) {
        while *i < b.len() && (b[*i] as char).is_whitespace() { *i += 1; }
    }

    fn json_chaine(b: &[u8], i: &mut usize) -> Option<String> {
        if *i >= b.len() || b[*i] != b'"' { return None; }
        *i += 1;
        let mut s = String::new();
        while *i < b.len() {
            let c = b[*i];
            if c == b'"' { *i += 1; return Some(s); }
            if c == b'\\' {
                *i += 1;
                if *i >= b.len() { return None; }
                let e = b[*i];
                *i += 1;
                match e {
                    b'n' => s.push('\n'),
                    b't' => s.push('\t'),
                    b'r' => s.push('\r'),
                    b'b' => s.push('\u{8}'),
                    b'f' => s.push('\u{c}'),
                    b'/' => s.push('/'),
                    b'"' => s.push('"'),
                    b'\\' => s.push('\\'),
                    b'u' => {
                        if *i + 4 > b.len() { return None; }
                        let hex = std::str::from_utf8(&b[*i..*i + 4]).ok()?;
                        let code = u32::from_str_radix(hex, 16).ok()?;
                        *i += 4;
                        s.push(char::from_u32(code).unwrap_or('\u{fffd}'));
                    }
                    _ => return None,
                }
            } else {
                let start = *i;
                while *i < b.len() && b[*i] != b'"' && b[*i] != b'\\' { *i += 1; }
                s.push_str(std::str::from_utf8(&b[start..*i]).ok()?);
            }
        }
        None
    }

    fn json_valeur(b: &[u8], i: &mut usize) -> Option<Jv> {
        json_skip_ws(b, i);
        if *i >= b.len() { return None; }
        match b[*i] {
            b'{' => {
                *i += 1;
                let mut fields: Vec<(String, Jv)> = Vec::new();
                json_skip_ws(b, i);
                if *i < b.len() && b[*i] == b'}' { *i += 1; return Some(Jv::Obj(fields)); }
                loop {
                    json_skip_ws(b, i);
                    if *i >= b.len() || b[*i] != b'"' { return None; }
                    let k = json_chaine(b, i)?;
                    json_skip_ws(b, i);
                    if *i >= b.len() || b[*i] != b':' { return None; }
                    *i += 1;
                    let v = json_valeur(b, i)?;
                    fields.push((k, v));
                    json_skip_ws(b, i);
                    if *i >= b.len() { return None; }
                    match b[*i] {
                        b',' => { *i += 1; }
                        b'}' => { *i += 1; return Some(Jv::Obj(fields)); }
                        _ => return None,
                    }
                }
            }
            b'[' => {
                *i += 1;
                let mut items: Vec<Jv> = Vec::new();
                json_skip_ws(b, i);
                if *i < b.len() && b[*i] == b']' { *i += 1; return Some(Jv::List(items)); }
                loop {
                    let v = json_valeur(b, i)?;
                    items.push(v);
                    json_skip_ws(b, i);
                    if *i >= b.len() { return None; }
                    match b[*i] {
                        b',' => { *i += 1; }
                        b']' => { *i += 1; return Some(Jv::List(items)); }
                        _ => return None,
                    }
                }
            }
            b'"' => Some(Jv::Str(json_chaine(b, i)?)),
            b't' => {
                if b.len() >= *i + 4 && &b[*i..*i + 4] == b"true" { *i += 4; return Some(Jv::Bool(true)); }
                None
            }
            b'f' => {
                if b.len() >= *i + 5 && &b[*i..*i + 5] == b"false" { *i += 5; return Some(Jv::Bool(false)); }
                None
            }
            b'n' => {
                if b.len() >= *i + 4 && &b[*i..*i + 4] == b"null" { *i += 4; return Some(Jv::Null); }
                None
            }
            _ => {
                let start = *i;
                if b[*i] == b'-' { *i += 1; }
                while *i < b.len() && b[*i].is_ascii_digit() { *i += 1; }
                if start == *i { return None; }
                let txt = std::str::from_utf8(&b[start..*i]).ok()?;
                txt.parse::<i64>().ok().map(Jv::Int)
            }
        }
    }

    /// Parse un document JSON complet ; `None` si le texte n'est pas du JSON.
    fn json_parse(texte: &str) -> Option<Jv> {
        let b = texte.as_bytes();
        let mut i = 0usize;
        let v = json_valeur(b, &mut i)?;
        json_skip_ws(b, &mut i);
        if i != b.len() { return None; }
        Some(v)
    }

    /// Positions (en octets) des titres de markdown : meme regex que le jumeau
    /// Python (`^##? ` en mode multiligne), donc memes decoupes.
    fn positions_titres(texte: &str) -> Vec<usize> {
        let mut out = Vec::new();
        let mut off = 0usize;
        for l in texte.split_inclusive('\n') {
            let t = l.trim_end_matches('\n').trim_end_matches('\r');
            if t.starts_with("# ") || t.starts_with("## ") { out.push(off); }
            off += l.len();
        }
        out
    }

    /// Perception : la signature de structure deduite du contenu.
    fn structure_interne(seg: &Segment) -> Info {
        let mut info = Info {
            octets: seg.data.len(),
            texte: false,
            csv: false,
            blocs_md: 0,
            json: false,
            json_obj: false,
        };
        let texte = match String::from_utf8(seg.data.clone()) {
            Ok(t) => t,
            Err(_) => return info,
        };
        info.texte = true;
        let t = texte.trim();
        if t.starts_with('{') || t.starts_with('[') {
            if let Some(v) = json_parse(&texte) {
                info.json_obj = matches!(v, Jv::Obj(_));
                info.json = true;
                return info;
            }
        }
        let lignes: Vec<&str> = texte.lines().filter(|l| !l.is_empty()).collect();
        if !lignes.is_empty() && lignes[0].contains(',') {
            let n = lignes[0].matches(',').count();
            info.csv = lignes.iter().all(|l| l.matches(',').count() == n);
        }
        info.blocs_md = positions_titres(&texte).len();
        info
    }

    /// Classe de forme d'un segment — la cle du catalogue amorti.
    ///
    /// Distingue l'objet JSON (decoupable par ses cles) du tableau JSON
    /// (compresse tel quel) : deux formes, donc deux decisions distinctes.
    fn signature(s: &Info) -> String {
        if s.json {
            return if s.json_obj { "json_obj".to_string() } else { "json".to_string() };
        }
        if s.csv { return "csv".to_string(); }
        if s.blocs_md > 0 { return "markdown".to_string(); }
        if s.texte { "texte".to_string() } else { "binaire".to_string() }
    }

    /// Un petit corpus heterogene, genere deterministiquement : meme entree a
    /// chaque execution, donc resultat reproductible.
    fn corpus_demo() -> Vec<Segment> {
        let ventes = "nom,quantite,prix\ncafe,10,2.50\nthe,20,1.80\nsucre,5,0.90\nlait,8,1.10\npain,12,0.80\n";
        let clients_obj = Jv::Obj(vec![
            ("id".to_string(), Jv::Int(7)),
            ("panier".to_string(), Jv::List(vec![
                Jv::Obj(vec![
                    ("sku".to_string(), Jv::Str("cafe".to_string())),
                    ("q".to_string(), Jv::Int(10)),
                ]),
                Jv::Obj(vec![
                    ("sku".to_string(), Jv::Str("the".to_string())),
                    ("q".to_string(), Jv::Int(20)),
                ]),
            ])),
        ]);
        let mut clients = String::new();
        json_dump(&clients_obj, 0, &mut clients);
        clients.push('\n');
        let notes = "# Rapport\n\n## Chiffre\n12 kEUR, en hausse.\n\n## Risque\nFournisseur unique, lead time 6 semaines.\n";
        let journal: String = (0..40)
            .map(|i| format!("10:{:02}:00 INFO requete {} ok\n", i, i))
            .collect();
        let mut binaire: Vec<u8> = Vec::with_capacity(1024);
        for _ in 0..4 {
            for b in 0u16..256 { binaire.push(b as u8); }
        }
        vec![
            Segment {
                id: "ventes.csv#0".to_string(),
                data: ventes.as_bytes().to_vec(),
                genre: "donnees/csv".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "clients.json#0".to_string(),
                data: clients.into_bytes(),
                genre: "donnees/json".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "notes.md#0".to_string(),
                data: notes.as_bytes().to_vec(),
                genre: "texte/markdown".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "journal.log#0".to_string(),
                data: journal.into_bytes(),
                genre: "texte/log".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "brut.bin#0".to_string(),
                data: binaire,
                genre: "binaire".to_string(),
                parent: None,
                profondeur: 0,
            },
        ]
    }

    /// La decision d'un cerveau : decouper (reduire) ou compresser.
    ///
    /// Un `enum` remplace le `dict` du jumeau Python : les deux cas sont
    /// nommes, donc le compilateur refuse le cas forget.
    #[derive(Clone)]
    enum Decision {
        Decouper { mode: String },
        Compresser { codec: String },
    }

    impl Decision {
        fn op(&self) -> &'static str {
            match self {
                Decision::Decouper { .. } => "decouper",
                Decision::Compresser { .. } => "compresser",
            }
        }
        fn quoi(&self) -> &str {
            match self {
                Decision::Decouper { mode } => mode,
                Decision::Compresser { codec } => codec,
            }
        }
    }

    /// La recette : le traitement final retenu pour un segment.
    ///
    /// Un segment est couvert s'il est compresse, ou decoupe en enfants tous
    /// couverts — c'est l'invariant que `verifier_couverture` fait respecter.
    enum Recette {
        Decouper { mode: String, enfants: Vec<String> },
        Compresser { codec: String },
    }

    /// Un segment est couvert s'il est compresse, ou decoupe en enfants tous
    /// couverts. Le cycle est detecte : la recette ne doit pas boucler.
    fn couverture(
        sid: &str,
        recette: &[(String, Recette)],
        vus: &mut Vec<String>,
    ) -> Result<Vec<String>, String> {
        if vus.iter().any(|v| v == sid) {
            return Err(format!("cycle dans la recette autour de {}", sid));
        }
        vus.push(sid.to_string());
        let entree = recette.iter().find(|(k, _)| k == sid).map(|(_, v)| v);
        let res = match entree {
            None => Ok(vec![sid.to_string()]),
            Some(Recette::Compresser { .. }) => Ok(Vec::new()),
            Some(Recette::Decouper { enfants, .. }) => {
                let mut manquants: Vec<String> = Vec::new();
                for enfant in enfants {
                    match couverture(enfant, recette, vus) {
                        Ok(abs) => manquants.extend(abs),
                        Err(msg) => {
                            vus.pop();
                            return Err(msg);
                        }
                    }
                }
                Ok(manquants)
            }
        };
        vus.pop();
        res
    }

    /// Invariant : leve en nommant tout segment original non traite.
    fn verifier_couverture(racines: &[&str], recette: &[(String, Recette)]) -> Result<(), String> {
        let mut manquants: Vec<String> = Vec::new();
        for sid in racines {
            match couverture(sid, recette, &mut Vec::new()) {
                Ok(abs) => manquants.extend(abs),
                Err(msg) => return Err(msg),
            }
        }
        if !manquants.is_empty() {
            let liste: Vec<&str> = manquants.iter().take(6).map(|s| s.as_str()).collect();
            return Err(format!(
                "{} segment(s) sans traitement final : {}",
                manquants.len(),
                py_list_str(&liste)
            ));
        }
        Ok(())
    }

    /// Operation inversible : un segment macro -> des fragments qualifies.
    fn decouper(seg: &Segment, mode: &str) -> Vec<Segment> {
        let texte = match String::from_utf8(seg.data.clone()) {
            Ok(t) => t,
            Err(_) => return Vec::new(),
        };
        let mut kids: Vec<Segment> = Vec::new();
        match mode {
            "blocs_markdown" => {
                let mut bornes: Vec<usize> = vec![0];
                bornes.extend(positions_titres(&texte));
                bornes.push(texte.len());
                for i in 0..bornes.len() - 1 {
                    let m = &texte[bornes[i]..bornes[i + 1]];
                    if !m.trim().is_empty() {
                        kids.push(Segment {
                            id: format!("{}.b{}", seg.id, kids.len()),
                            data: m.as_bytes().to_vec(),
                            genre: format!("{}/bloc", seg.genre),
                            parent: Some(seg.id.clone()),
                            profondeur: seg.profondeur + 1,
                        });
                    }
                }
            }
            "colonnes_csv" => {
                let lignes: Vec<&str> = texte.lines().filter(|l| !l.is_empty()).collect();
                if lignes.is_empty() { return kids; }
                let cols: Vec<&str> = lignes[0].split(',').collect();
                let mat: Vec<Vec<&str>> = lignes.iter().map(|l| l.split(',').collect()).collect();
                for j in 0..cols.len() {
                    let mut s = String::new();
                    for (r, row) in mat.iter().enumerate() {
                        if r > 0 { s.push('\n'); }
                        s.push_str(row[j]);
                    }
                    kids.push(Segment {
                        id: format!("{}.c{}", seg.id, j),
                        data: s.into_bytes(),
                        genre: format!("{}/col:{}", seg.genre, cols[j]),
                        parent: Some(seg.id.clone()),
                        profondeur: seg.profondeur + 1,
                    });
                }
            }
            "cles_json" => {
                let champs = match json_parse(&texte) {
                    Some(Jv::Obj(f)) => f,
                    _ => return kids,
                };
                for (k, v) in &champs {
                    let mut s = String::new();
                    json_dump(v, 0, &mut s);
                    kids.push(Segment {
                        id: format!("{}.k{}", seg.id, kids.len()),
                        data: s.into_bytes(),
                        genre: format!("{}/cle:{}", seg.genre, k),
                        parent: Some(seg.id.clone()),
                        profondeur: seg.profondeur + 1,
                    });
                }
            }
            _ => {}
        }
        kids
    }

    /// L'inverse d'un decoupage : reconstruit les octets du segment parent.
    fn recoller(mode: &str, enfants: &[Segment]) -> Vec<u8> {
        match mode {
            "blocs_markdown" => {
                let mut out = Vec::new();
                for e in enfants { out.extend_from_slice(&e.data); }
                out
            }
            "colonnes_csv" => {
                let mat: Vec<Vec<String>> = enfants
                    .iter()
                    .map(|e| String::from_utf8_lossy(&e.data).split('\n').map(|s| s.to_string()).collect())
                    .collect();
                if mat.is_empty() { return Vec::new(); }
                let n = mat[0].len();
                let mut lignes: Vec<String> = Vec::new();
                for row in 0..n {
                    let parts: Vec<&str> = mat.iter().map(|c| c[row].as_str()).collect();
                    lignes.push(parts.join(","));
                }
                let mut s = lignes.join("\n");
                s.push('\n');
                s.into_bytes()
            }
            "cles_json" => {
                let mut champs: Vec<(String, Jv)> = Vec::new();
                for e in enfants {
                    let k = e.genre.rsplit(':').next().unwrap_or("").to_string();
                    let v = json_parse(&String::from_utf8_lossy(&e.data)).unwrap_or(Jv::Null);
                    champs.push((k, v));
                }
                let mut s = String::new();
                json_dump(&Jv::Obj(champs), 0, &mut s);
                s.push('\n');
                s.into_bytes()
            }
            _ => Vec::new(),
        }
    }

    /// Adler-32 (RFC 1950) — le controle d'integrite du flux zlib.
    fn adler32(data: &[u8]) -> u32 {
        let mut a: u32 = 1;
        let mut b: u32 = 0;
        for &x in data {
            a = (a + x as u32) % 65521;
            b = (b + a) % 65521;
        }
        (b << 16) | a
    }

    /// zlib en blocs STORES (RFC 1950/1951) : l'entete `0x78 0x9C`, un bloc
    /// deflate non compresse (BTYPE=00), puis l'adler32.
    ///
    /// Le flux produit est un zlib valide : il se decompresse par le
    /// `zlib.decompress` de CPython. On ne reimplemente pas deflate
    /// compresse : le carnet teste l'INVERSIBILITE, pas le taux.
    fn zlib_store(data: &[u8]) -> Vec<u8> {
        let mut out: Vec<u8> = vec![0x78, 0x9C];
        let n = (data.len() as u16).to_le_bytes();
        out.push(0x01); // BFINAL=1, BTYPE=00 (stocke)
        out.push(n[0]);
        out.push(n[1]);
        out.push(!n[0]);
        out.push(!n[1]);
        out.extend_from_slice(data);
        let ad = adler32(data).to_be_bytes();
        out.extend_from_slice(&ad);
        out
    }

    /// Le miroir exact de `zlib_store`.
    fn zlib_unstore(data: &[u8]) -> Option<Vec<u8>> {
        if data.len() < 11 { return None; }
        if data[0] != 0x78 || data[1] != 0x9C { return None; }
        if data[2] & 0x06 != 0x00 { return None; }
        let len = u16::from_le_bytes([data[3], data[4]]) as usize;
        if u16::from_le_bytes([data[5], data[6]]) != !(len as u16) { return None; }
        if data.len() < 7 + len + 4 { return None; }
        let payload = &data[7..7 + len];
        let ad = u32::from_be_bytes([
            data[7 + len], data[8 + len], data[9 + len], data[10 + len],
        ]);
        if ad != adler32(payload) { return None; }
        Some(payload.to_vec())
    }

    /// STUB DECLARE — le codec lzma.
    ///
    /// Le vrai LZMA1 (range coding, arbre de probabilites litterales, etats
    /// IsMatch) n'est PAS reproductible en `std` seule : il faudrait une crate
    /// (`lzma-rs`), exclue par le cahier des charges de ce portage. Ce stub
    /// garde le format LZMA-alone (entete de treize octets) et recopie le
    /// payload tel quel ; il remplit donc le contrat teste par le carnet, qui
    /// est l'INVERSIBILITE, et rien de plus.
    fn lzma_lit(data: &[u8]) -> Vec<u8> {
        let mut out: Vec<u8> = vec![0x5D];
        out.extend_from_slice(&0x0010_0000u32.to_le_bytes()); // taille de dictionnaire
        out.extend_from_slice(&(data.len() as u64).to_le_bytes());
        out.extend_from_slice(data);
        out
    }

    /// Le miroir du stub lzma.
    fn lzma_unlit(data: &[u8]) -> Option<Vec<u8>> {
        if data.len() < 13 { return None; }
        Some(data[13..].to_vec())
    }

    // Contrat d'inversibilite (temoin de l'organe) : decouper puis recoller
    // redonne les octets d'origine, pour chaque mode present dans le corpus.
    let segments = corpus_demo();
    for seg in &segments {
        for mode in ["blocs_markdown", "colonnes_csv", "cles_json"] {
            let s = structure_interne(seg);
            if mode == "colonnes_csv" && !s.csv { continue; }
            if mode == "cles_json" && !s.json { continue; }
            if mode == "blocs_markdown" && s.blocs_md == 0 { continue; }
            let enfants = decouper(seg, mode);
            assert!(!enfants.is_empty(), "decoupe vide : {}", mode);
            assert_eq!(recoller(mode, &enfants), seg.data, "non inversible : {}", mode);
        }
    }
    let codecs: Vec<&str> = vec!["lzma", "store", "zlib"];
    println!(
        "codecs : {} ; decoupes/recolles verifiés inversibles sur le corpus",
        py_list_str(&codecs)
    );

    // Témoin des codecs : le contrat verifie est l'inversibilite.
    let temoin = "temoin de codec ".repeat(5);
    let t = temoin.as_bytes();
    assert_eq!(zlib_unstore(&zlib_store(t)).unwrap_or_default(), t.to_vec(), "codec non inversible : zlib");
    assert_eq!(lzma_unlit(&lzma_lit(t)).unwrap_or_default(), t.to_vec(), "codec non inversible : lzma");
    assert_eq!(t.to_vec(), t.to_vec(), "codec non inversible : store");
    println!("codecs inversibles : OK");
};

codecs : ['lzma', 'store', 'zlib'] ; decoupes/recolles verifiés inversibles sur le corpus


codecs inversibles : OK


### Lecture du résultat

Deux familles d'opérations, toutes **inversibles** (le témoin négatif passe) : les
*découpes* (csv/json/markdown) et les *codecs* (zlib/lzma/store). L'invariant de
couverture qui vient d'être défini est le **garde-fou de la composition** : il sera
déclenché plus bas sur un plan *délibérément* incomplet, pour montrer que la boucle
**rejette** un plan qu'elle ne peut pas exécuter en toute sécurité.

Une note d'honnêteté sur les codecs. Le codec `zlib` est écrit en blocs **stockés**
(entête RFC 1950, `adler32` maison) : le flux produit est un zlib valide, il se
décompresse par n'importe quel décompresseur, mais il ne **compresse** pas — le
carnet teste l'inversibilité, pas le taux. Le codec `lzma` est un **stub copie
déclaré** : le vrai LZMA1 (range coding) n'est pas reproductible avec la seule
bibliothèque standard, il faudrait une crate que le cahier des charges exclut. Le
stub garde le format LZMA-alone et recopie la charge utile ; il remplit donc le
contrat testé par le carnet, qui est l'inversibilité, et rien de plus.

## 5. Le squelette — `percevoir → … → observer`

On assemble maintenant les stades en un **squelette** `AgentTriage`. Il compose :
la représentation (`Segment`), un cerveau (le point de composition), l'invariant
de couverture (la vérification), et l'application de la recette (l'exécution). Le
résultat clé est dans `decouvrir` / `rejouer` : la découverte **paie** le
catalogue `forme → décision` en inférences ; le rejeu le **consomme** sans
ré-inférer.

Le catalogue est une `Vec` de paires, pas une table triée : une forme déjà connue
est **remplacée en place**, ce qui conserve l'ordre de *première insertion*. C'est
cet ordre que l'on lira dans le résultat de la découverte.

In [5]:
/// Representation commune d'un morceau de donnee (reutilisee de Compression-07b).
///
/// L'objet que la boucle percoit et represente : `id` stable, `data` = octets,
/// `genre` = qualification, `parent`/`profondeur` = lien vers l'origine.
#[derive(Clone)]
struct Segment {
    id: String,
    data: Vec<u8>,
    genre: String,
    parent: Option<String>,
    profondeur: i64,
}
/// La decision d'un cerveau : decouper (reduire) ou compresser.
///
/// Un `enum` remplace le `dict` du jumeau Python : les deux cas sont
/// nommes, donc le compilateur refuse le cas forget.
#[derive(Clone)]
enum Decision {
    Decouper { mode: String },
    Compresser { codec: String },
}

impl Decision {
    fn op(&self) -> &'static str {
        match self {
            Decision::Decouper { .. } => "decouper",
            Decision::Compresser { .. } => "compresser",
        }
    }
    fn quoi(&self) -> &str {
        match self {
            Decision::Decouper { mode } => mode,
            Decision::Compresser { codec } => codec,
        }
    }
}
/// La recette : le traitement final retenu pour un segment.
///
/// Un segment est couvert s'il est compresse, ou decoupe en enfants tous
/// couverts — c'est l'invariant que `verifier_couverture` fait respecter.
enum Recette {
    Decouper { mode: String, enfants: Vec<String> },
    Compresser { codec: String },
}
/// Point de composition : le contrat d'un mecanisme d'inference.
///
/// La boucle agentique n'appelle JAMAIS un cerveau en particulier, seulement
/// ce trait. C'est ce contrat unique qui rend le cerveau interchangeable
/// (deterministe, LLM, hybride). Chaque consultation compte comme une inference.
trait Cerveau {
    fn inferer(&mut self, seg: &Segment) -> Result<Decision, String>;
}
// Le squelette est defini ici ; il est instancie dans les cellules
// suivantes, qui reconstruisent ce dont elles ont besoin. Aucune sortie :
// ce sont des definitions seules.
let _ = ();

### Lecture du résultat

Le squelette tient en une structure. Notez la place exacte de chaque stade :
`decouvrir` fait **percevoir→représenter→inférer→décider→vérifier** d'un bloc (la
vérification est *dans* la découverte — un plan incomplet n'est jamais accepté), et
`rejouer` fait **exécuter** le savoir acquis **sans passer par l'inférence**.
`observer`, c'est le compteur `appel_decouverte` qu'on va lire dans la section
suivante.

## 6. Exécuter + Observer — l'amortissement du coût

On laisse le squelette **découvrir** la politique sur le corpus, puis on le fait
**rejouer** sur un *deuxième* corpus de même forme. L'observation est le contraste
de coût entre les deux phases.

In [6]:
{
    /// Representation commune d'un morceau de donnee (reutilisee de Compression-07b).
    ///
    /// L'objet que la boucle percoit et represente : `id` stable, `data` = octets,
    /// `genre` = qualification, `parent`/`profondeur` = lien vers l'origine.
    #[derive(Clone)]
    struct Segment {
        id: String,
        data: Vec<u8>,
        genre: String,
        parent: Option<String>,
        profondeur: i64,
    }

    /// Signature de structure deduite du CONTENU — la perception du segment.
    ///
    /// C'est ce que le cerveau regarde pour decider. Deduite, pas etiquettee :
    /// un fichier nomme `.json` qui ne parse pas n'est pas du JSON.
    struct Info {
        octets: usize,
        texte: bool,
        csv: bool,
        blocs_md: usize,
        json: bool,
        json_obj: bool,
    }

    /// Les booleens Rust s'impriment `true`/`false` ; le jumeau Python impose
    /// `True`/`False`. Toute comparaison de texte passe par cette fonction.
    fn py(b: bool) -> &'static str {
        if b { "True" } else { "False" }
    }

    /// Repr Python d'une liste de chaines : `['a', 'b']`.
    fn py_list_str(items: &[&str]) -> String {
        let mut s = String::from("[");
        for (i, it) in items.iter().enumerate() {
            if i > 0 { s.push_str(", "); }
            s.push('\'');
            s.push_str(it);
            s.push('\'');
        }
        s.push(']');
        s
    }

    /// JSON : mini-dumpeur et mini-parseur maison (`std` seule, aucune crate).
    ///
    /// L'ordre d'insertion des cles d'un objet est PRESERVE (`Vec<(String, Jv)>`) :
    /// le catalogue amorti s'appuie sur cet ordre, un `BTreeMap` le casserait.
    #[derive(Clone)]
    enum Jv {
        Null,
        Bool(bool),
        Int(i64),
        Str(String),
        List(Vec<Jv>),
        Obj(Vec<(String, Jv)>),
    }

    fn json_echapper(s: &str, out: &mut String) {
        out.push('"');
        for c in s.chars() {
            match c {
                '"' => { out.push('\\'); out.push('"'); }
                '\\' => { out.push('\\'); out.push('\\'); }
                '\n' => out.push_str("\\n"),
                '\r' => out.push_str("\\r"),
                '\t' => out.push_str("\\t"),
                c if (c as u32) < 0x20 => out.push_str(&format!("\\u{:04x}", c as u32)),
                c => out.push(c),
            }
        }
        out.push('"');
    }

    /// Dumps equivalent a `json.dumps(indent=2, ensure_ascii=False)`.
    fn json_dump(v: &Jv, level: usize, out: &mut String) {
        let pad = "  ".repeat(level);
        let pad_in = "  ".repeat(level + 1);
        match v {
            Jv::Null => out.push_str("null"),
            Jv::Bool(b) => out.push_str(if *b { "true" } else { "false" }),
            Jv::Int(i) => out.push_str(&i.to_string()),
            Jv::Str(s) => json_echapper(s, out),
            Jv::List(items) => {
                if items.is_empty() { out.push_str("[]"); return; }
                out.push_str("[\n");
                for (i, it) in items.iter().enumerate() {
                    out.push_str(&pad_in);
                    json_dump(it, level + 1, out);
                    if i + 1 < items.len() { out.push(','); }
                    out.push('\n');
                }
                out.push_str(&pad);
                out.push(']');
            }
            Jv::Obj(fields) => {
                if fields.is_empty() { out.push_str("{}"); return; }
                out.push_str("{\n");
                for (i, (k, val)) in fields.iter().enumerate() {
                    out.push_str(&pad_in);
                    json_echapper(k, out);
                    out.push_str(": ");
                    json_dump(val, level + 1, out);
                    if i + 1 < fields.len() { out.push(','); }
                    out.push('\n');
                }
                out.push_str(&pad);
                out.push('}');
            }
        }
    }

    fn json_skip_ws(b: &[u8], i: &mut usize) {
        while *i < b.len() && (b[*i] as char).is_whitespace() { *i += 1; }
    }

    fn json_chaine(b: &[u8], i: &mut usize) -> Option<String> {
        if *i >= b.len() || b[*i] != b'"' { return None; }
        *i += 1;
        let mut s = String::new();
        while *i < b.len() {
            let c = b[*i];
            if c == b'"' { *i += 1; return Some(s); }
            if c == b'\\' {
                *i += 1;
                if *i >= b.len() { return None; }
                let e = b[*i];
                *i += 1;
                match e {
                    b'n' => s.push('\n'),
                    b't' => s.push('\t'),
                    b'r' => s.push('\r'),
                    b'b' => s.push('\u{8}'),
                    b'f' => s.push('\u{c}'),
                    b'/' => s.push('/'),
                    b'"' => s.push('"'),
                    b'\\' => s.push('\\'),
                    b'u' => {
                        if *i + 4 > b.len() { return None; }
                        let hex = std::str::from_utf8(&b[*i..*i + 4]).ok()?;
                        let code = u32::from_str_radix(hex, 16).ok()?;
                        *i += 4;
                        s.push(char::from_u32(code).unwrap_or('\u{fffd}'));
                    }
                    _ => return None,
                }
            } else {
                let start = *i;
                while *i < b.len() && b[*i] != b'"' && b[*i] != b'\\' { *i += 1; }
                s.push_str(std::str::from_utf8(&b[start..*i]).ok()?);
            }
        }
        None
    }

    fn json_valeur(b: &[u8], i: &mut usize) -> Option<Jv> {
        json_skip_ws(b, i);
        if *i >= b.len() { return None; }
        match b[*i] {
            b'{' => {
                *i += 1;
                let mut fields: Vec<(String, Jv)> = Vec::new();
                json_skip_ws(b, i);
                if *i < b.len() && b[*i] == b'}' { *i += 1; return Some(Jv::Obj(fields)); }
                loop {
                    json_skip_ws(b, i);
                    if *i >= b.len() || b[*i] != b'"' { return None; }
                    let k = json_chaine(b, i)?;
                    json_skip_ws(b, i);
                    if *i >= b.len() || b[*i] != b':' { return None; }
                    *i += 1;
                    let v = json_valeur(b, i)?;
                    fields.push((k, v));
                    json_skip_ws(b, i);
                    if *i >= b.len() { return None; }
                    match b[*i] {
                        b',' => { *i += 1; }
                        b'}' => { *i += 1; return Some(Jv::Obj(fields)); }
                        _ => return None,
                    }
                }
            }
            b'[' => {
                *i += 1;
                let mut items: Vec<Jv> = Vec::new();
                json_skip_ws(b, i);
                if *i < b.len() && b[*i] == b']' { *i += 1; return Some(Jv::List(items)); }
                loop {
                    let v = json_valeur(b, i)?;
                    items.push(v);
                    json_skip_ws(b, i);
                    if *i >= b.len() { return None; }
                    match b[*i] {
                        b',' => { *i += 1; }
                        b']' => { *i += 1; return Some(Jv::List(items)); }
                        _ => return None,
                    }
                }
            }
            b'"' => Some(Jv::Str(json_chaine(b, i)?)),
            b't' => {
                if b.len() >= *i + 4 && &b[*i..*i + 4] == b"true" { *i += 4; return Some(Jv::Bool(true)); }
                None
            }
            b'f' => {
                if b.len() >= *i + 5 && &b[*i..*i + 5] == b"false" { *i += 5; return Some(Jv::Bool(false)); }
                None
            }
            b'n' => {
                if b.len() >= *i + 4 && &b[*i..*i + 4] == b"null" { *i += 4; return Some(Jv::Null); }
                None
            }
            _ => {
                let start = *i;
                if b[*i] == b'-' { *i += 1; }
                while *i < b.len() && b[*i].is_ascii_digit() { *i += 1; }
                if start == *i { return None; }
                let txt = std::str::from_utf8(&b[start..*i]).ok()?;
                txt.parse::<i64>().ok().map(Jv::Int)
            }
        }
    }

    /// Parse un document JSON complet ; `None` si le texte n'est pas du JSON.
    fn json_parse(texte: &str) -> Option<Jv> {
        let b = texte.as_bytes();
        let mut i = 0usize;
        let v = json_valeur(b, &mut i)?;
        json_skip_ws(b, &mut i);
        if i != b.len() { return None; }
        Some(v)
    }

    /// Positions (en octets) des titres de markdown : meme regex que le jumeau
    /// Python (`^##? ` en mode multiligne), donc memes decoupes.
    fn positions_titres(texte: &str) -> Vec<usize> {
        let mut out = Vec::new();
        let mut off = 0usize;
        for l in texte.split_inclusive('\n') {
            let t = l.trim_end_matches('\n').trim_end_matches('\r');
            if t.starts_with("# ") || t.starts_with("## ") { out.push(off); }
            off += l.len();
        }
        out
    }

    /// Perception : la signature de structure deduite du contenu.
    fn structure_interne(seg: &Segment) -> Info {
        let mut info = Info {
            octets: seg.data.len(),
            texte: false,
            csv: false,
            blocs_md: 0,
            json: false,
            json_obj: false,
        };
        let texte = match String::from_utf8(seg.data.clone()) {
            Ok(t) => t,
            Err(_) => return info,
        };
        info.texte = true;
        let t = texte.trim();
        if t.starts_with('{') || t.starts_with('[') {
            if let Some(v) = json_parse(&texte) {
                info.json_obj = matches!(v, Jv::Obj(_));
                info.json = true;
                return info;
            }
        }
        let lignes: Vec<&str> = texte.lines().filter(|l| !l.is_empty()).collect();
        if !lignes.is_empty() && lignes[0].contains(',') {
            let n = lignes[0].matches(',').count();
            info.csv = lignes.iter().all(|l| l.matches(',').count() == n);
        }
        info.blocs_md = positions_titres(&texte).len();
        info
    }

    /// Classe de forme d'un segment — la cle du catalogue amorti.
    ///
    /// Distingue l'objet JSON (decoupable par ses cles) du tableau JSON
    /// (compresse tel quel) : deux formes, donc deux decisions distinctes.
    fn signature(s: &Info) -> String {
        if s.json {
            return if s.json_obj { "json_obj".to_string() } else { "json".to_string() };
        }
        if s.csv { return "csv".to_string(); }
        if s.blocs_md > 0 { return "markdown".to_string(); }
        if s.texte { "texte".to_string() } else { "binaire".to_string() }
    }

    /// Un petit corpus heterogene, genere deterministiquement : meme entree a
    /// chaque execution, donc resultat reproductible.
    fn corpus_demo() -> Vec<Segment> {
        let ventes = "nom,quantite,prix\ncafe,10,2.50\nthe,20,1.80\nsucre,5,0.90\nlait,8,1.10\npain,12,0.80\n";
        let clients_obj = Jv::Obj(vec![
            ("id".to_string(), Jv::Int(7)),
            ("panier".to_string(), Jv::List(vec![
                Jv::Obj(vec![
                    ("sku".to_string(), Jv::Str("cafe".to_string())),
                    ("q".to_string(), Jv::Int(10)),
                ]),
                Jv::Obj(vec![
                    ("sku".to_string(), Jv::Str("the".to_string())),
                    ("q".to_string(), Jv::Int(20)),
                ]),
            ])),
        ]);
        let mut clients = String::new();
        json_dump(&clients_obj, 0, &mut clients);
        clients.push('\n');
        let notes = "# Rapport\n\n## Chiffre\n12 kEUR, en hausse.\n\n## Risque\nFournisseur unique, lead time 6 semaines.\n";
        let journal: String = (0..40)
            .map(|i| format!("10:{:02}:00 INFO requete {} ok\n", i, i))
            .collect();
        let mut binaire: Vec<u8> = Vec::with_capacity(1024);
        for _ in 0..4 {
            for b in 0u16..256 { binaire.push(b as u8); }
        }
        vec![
            Segment {
                id: "ventes.csv#0".to_string(),
                data: ventes.as_bytes().to_vec(),
                genre: "donnees/csv".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "clients.json#0".to_string(),
                data: clients.into_bytes(),
                genre: "donnees/json".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "notes.md#0".to_string(),
                data: notes.as_bytes().to_vec(),
                genre: "texte/markdown".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "journal.log#0".to_string(),
                data: journal.into_bytes(),
                genre: "texte/log".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "brut.bin#0".to_string(),
                data: binaire,
                genre: "binaire".to_string(),
                parent: None,
                profondeur: 0,
            },
        ]
    }

    /// La decision d'un cerveau : decouper (reduire) ou compresser.
    ///
    /// Un `enum` remplace le `dict` du jumeau Python : les deux cas sont
    /// nommes, donc le compilateur refuse le cas forget.
    #[derive(Clone)]
    enum Decision {
        Decouper { mode: String },
        Compresser { codec: String },
    }

    impl Decision {
        fn op(&self) -> &'static str {
            match self {
                Decision::Decouper { .. } => "decouper",
                Decision::Compresser { .. } => "compresser",
            }
        }
        fn quoi(&self) -> &str {
            match self {
                Decision::Decouper { mode } => mode,
                Decision::Compresser { codec } => codec,
            }
        }
    }

    /// Point de composition : le contrat d'un mecanisme d'inference.
    ///
    /// La boucle agentique n'appelle JAMAIS un cerveau en particulier, seulement
    /// ce trait. C'est ce contrat unique qui rend le cerveau interchangeable
    /// (deterministe, LLM, hybride). Chaque consultation compte comme une inference.
    trait Cerveau {
        fn inferer(&mut self, seg: &Segment) -> Result<Decision, String>;
    }

    /// La recette : le traitement final retenu pour un segment.
    ///
    /// Un segment est couvert s'il est compresse, ou decoupe en enfants tous
    /// couverts — c'est l'invariant que `verifier_couverture` fait respecter.
    enum Recette {
        Decouper { mode: String, enfants: Vec<String> },
        Compresser { codec: String },
    }

    /// Le squelette agentique : percevoir -> representer -> inferer -> decider
    /// -> verifier -> executer -> observer.
    ///
    /// Compose la representation (Segment), un cerveau (point de composition),
    /// l'invariant de couverture (verification) et l'application de la recette
    /// (execution). L'inference est isolee derriere le trait `Cerveau`.
    struct AgentTriage<'a> {
        cerveau: &'a mut dyn Cerveau,
        borne: i64,
        catalogue: Vec<(String, Decision)>,
        recette: Vec<(String, Recette)>,
        appel_decouverte: i64,
    }

    impl<'a> AgentTriage<'a> {
        /// Apprend le catalogue forme->decision en parcourant le corpus.
        ///
        /// Chaque consultation du cerveau compte. Si le parcours atteint la borne
        /// sans epuiser la file, la decouverte ECHOUE (safety bound) : on ne
        /// laisse jamais l'agent tourner a l'infini.
        fn decouvrir(&mut self, segments: &[Segment]) -> Result<(), String> {
            self.catalogue = Vec::new();
            self.recette = Vec::new();
            self.appel_decouverte = 0;
            let mut file: Vec<Segment> = segments.to_vec();
            while !file.is_empty() {
                // La borne est verifiee AVANT le retrait : c'est elle qui
                // distingue une decouverte bornee d'une decouverte divergente.
                if self.appel_decouverte >= self.borne {
                    return Err(format!(
                        "decouverte interrompue a {} inférences, {} segment(s) en attente — safety bound.",
                        self.appel_decouverte,
                        file.len()
                    ));
                }
                let seg = file.remove(0);
                let dec = self.cerveau.inferer(&seg)?; // inferer + decider (compte)
                self.appel_decouverte += 1;
                let forme = signature(&structure_interne(&seg));
                self.cat_set(&forme, dec.clone());
                if let Decision::Decouper { mode } = &dec {
                    let enfants = decouper(&seg, mode);
                    // La recette porte les ENFANTS reels, pas seulement la
                    // decision : c'est cette liste que l'invariant parcourt.
                    let ids: Vec<String> = enfants.iter().map(|e| e.id.clone()).collect();
                    self.recette.push((
                        seg.id.clone(),
                        Recette::Decouper { mode: mode.clone(), enfants: ids },
                    ));
                    file.extend(enfants);
                } else {
                    self.recette.push((seg.id.clone(), Recette::Compresser { codec: dec.quoi().to_string() }));
                }
            }
            let racines: Vec<&str> = segments.iter().map(|s| s.id.as_str()).collect();
            verifier_couverture(&racines, &self.recette)?; // verifier
            Ok(())
        }

        /// Le catalogue garde l'ordre de PREMIERE insertion : une forme deja
        /// connue est remplacee en place, pas appendue.
        fn cat_set(&mut self, forme: &str, dec: Decision) {
            for (k, v) in self.catalogue.iter_mut() {
                if k == forme {
                    *v = dec;
                    return;
                }
            }
            self.catalogue.push((forme.to_string(), dec));
        }

        /// Applique le catalogue appris a de nouveaux segments, SANS re-inferer.
        ///
        /// C'est l'amortissement : la decouverte a paye le catalogue, le rejeu le
        /// consomme a cout d'inference NUL.
        fn rejouer(&self, segments: &[Segment]) -> Vec<(String, Decision)> {
            segments
                .iter()
                .map(|seg| {
                    let forme = signature(&structure_interne(seg));
                    let dec = self
                        .catalogue
                        .iter()
                        .find(|(k, _)| *k == forme)
                        .map(|(_, v)| v.clone())
                        .unwrap_or(Decision::Compresser { codec: "zlib".to_string() });
                    (seg.id.clone(), dec)
                })
                .collect()
        }
    }

    /// Un segment est couvert s'il est compresse, ou decoupe en enfants tous
    /// couverts. Le cycle est detecte : la recette ne doit pas boucler.
    fn couverture(
        sid: &str,
        recette: &[(String, Recette)],
        vus: &mut Vec<String>,
    ) -> Result<Vec<String>, String> {
        if vus.iter().any(|v| v == sid) {
            return Err(format!("cycle dans la recette autour de {}", sid));
        }
        vus.push(sid.to_string());
        let entree = recette.iter().find(|(k, _)| k == sid).map(|(_, v)| v);
        let res = match entree {
            None => Ok(vec![sid.to_string()]),
            Some(Recette::Compresser { .. }) => Ok(Vec::new()),
            Some(Recette::Decouper { enfants, .. }) => {
                let mut manquants: Vec<String> = Vec::new();
                for enfant in enfants {
                    match couverture(enfant, recette, vus) {
                        Ok(abs) => manquants.extend(abs),
                        Err(msg) => {
                            vus.pop();
                            return Err(msg);
                        }
                    }
                }
                Ok(manquants)
            }
        };
        vus.pop();
        res
    }

    /// Invariant : leve en nommant tout segment original non traite.
    fn verifier_couverture(racines: &[&str], recette: &[(String, Recette)]) -> Result<(), String> {
        let mut manquants: Vec<String> = Vec::new();
        for sid in racines {
            match couverture(sid, recette, &mut Vec::new()) {
                Ok(abs) => manquants.extend(abs),
                Err(msg) => return Err(msg),
            }
        }
        if !manquants.is_empty() {
            let liste: Vec<&str> = manquants.iter().take(6).map(|s| s.as_str()).collect();
            return Err(format!(
                "{} segment(s) sans traitement final : {}",
                manquants.len(),
                py_list_str(&liste)
            ));
        }
        Ok(())
    }

    /// Operation inversible : un segment macro -> des fragments qualifies.
    fn decouper(seg: &Segment, mode: &str) -> Vec<Segment> {
        let texte = match String::from_utf8(seg.data.clone()) {
            Ok(t) => t,
            Err(_) => return Vec::new(),
        };
        let mut kids: Vec<Segment> = Vec::new();
        match mode {
            "blocs_markdown" => {
                let mut bornes: Vec<usize> = vec![0];
                bornes.extend(positions_titres(&texte));
                bornes.push(texte.len());
                for i in 0..bornes.len() - 1 {
                    let m = &texte[bornes[i]..bornes[i + 1]];
                    if !m.trim().is_empty() {
                        kids.push(Segment {
                            id: format!("{}.b{}", seg.id, kids.len()),
                            data: m.as_bytes().to_vec(),
                            genre: format!("{}/bloc", seg.genre),
                            parent: Some(seg.id.clone()),
                            profondeur: seg.profondeur + 1,
                        });
                    }
                }
            }
            "colonnes_csv" => {
                let lignes: Vec<&str> = texte.lines().filter(|l| !l.is_empty()).collect();
                if lignes.is_empty() { return kids; }
                let cols: Vec<&str> = lignes[0].split(',').collect();
                let mat: Vec<Vec<&str>> = lignes.iter().map(|l| l.split(',').collect()).collect();
                for j in 0..cols.len() {
                    let mut s = String::new();
                    for (r, row) in mat.iter().enumerate() {
                        if r > 0 { s.push('\n'); }
                        s.push_str(row[j]);
                    }
                    kids.push(Segment {
                        id: format!("{}.c{}", seg.id, j),
                        data: s.into_bytes(),
                        genre: format!("{}/col:{}", seg.genre, cols[j]),
                        parent: Some(seg.id.clone()),
                        profondeur: seg.profondeur + 1,
                    });
                }
            }
            "cles_json" => {
                let champs = match json_parse(&texte) {
                    Some(Jv::Obj(f)) => f,
                    _ => return kids,
                };
                for (k, v) in &champs {
                    let mut s = String::new();
                    json_dump(v, 0, &mut s);
                    kids.push(Segment {
                        id: format!("{}.k{}", seg.id, kids.len()),
                        data: s.into_bytes(),
                        genre: format!("{}/cle:{}", seg.genre, k),
                        parent: Some(seg.id.clone()),
                        profondeur: seg.profondeur + 1,
                    });
                }
            }
            _ => {}
        }
        kids
    }

    /// L'inverse d'un decoupage : reconstruit les octets du segment parent.
    fn recoller(mode: &str, enfants: &[Segment]) -> Vec<u8> {
        match mode {
            "blocs_markdown" => {
                let mut out = Vec::new();
                for e in enfants { out.extend_from_slice(&e.data); }
                out
            }
            "colonnes_csv" => {
                let mat: Vec<Vec<String>> = enfants
                    .iter()
                    .map(|e| String::from_utf8_lossy(&e.data).split('\n').map(|s| s.to_string()).collect())
                    .collect();
                if mat.is_empty() { return Vec::new(); }
                let n = mat[0].len();
                let mut lignes: Vec<String> = Vec::new();
                for row in 0..n {
                    let parts: Vec<&str> = mat.iter().map(|c| c[row].as_str()).collect();
                    lignes.push(parts.join(","));
                }
                let mut s = lignes.join("\n");
                s.push('\n');
                s.into_bytes()
            }
            "cles_json" => {
                let mut champs: Vec<(String, Jv)> = Vec::new();
                for e in enfants {
                    let k = e.genre.rsplit(':').next().unwrap_or("").to_string();
                    let v = json_parse(&String::from_utf8_lossy(&e.data)).unwrap_or(Jv::Null);
                    champs.push((k, v));
                }
                let mut s = String::new();
                json_dump(&Jv::Obj(champs), 0, &mut s);
                s.push('\n');
                s.into_bytes()
            }
            _ => Vec::new(),
        }
    }

    /// Le cerveau par defaut : derive la decision de la structure percue.
    struct CerveauDeterministe { appels: i64 }

    impl Cerveau for CerveauDeterministe {
        fn inferer(&mut self, seg: &Segment) -> Result<Decision, String> {
            self.appels += 1;
            let s = structure_interne(seg);
            if seg.data.len() < 64 {
                return Ok(Decision::Compresser {
                    codec: if s.texte { "zlib".to_string() } else { "lzma".to_string() },
                });
            }
            let mode = match signature(&s).as_str() {
                "json_obj" => "cles_json",
                "csv" => "colonnes_csv",
                "markdown" => "blocs_markdown",
                _ => {
                    return Ok(Decision::Compresser {
                        codec: if s.texte { "zlib".to_string() } else { "lzma".to_string() },
                    })
                }
            };
            Ok(Decision::Decouper { mode: mode.to_string() })
        }
    }

    let segments = corpus_demo();
    let mut cerveau = CerveauDeterministe { appels: 0 };
    let mut agent = AgentTriage {
        cerveau: &mut cerveau,
        borne: 64,
        catalogue: Vec::new(),
        recette: Vec::new(),
        appel_decouverte: 0,
    };
    // Decouverte : on instancie l'agent (cerveau deterministe, borne 64
    // octets), on lui fait percevoir le corpus, puis on lit le catalogue
    // amorti de formes et la recette — ce que l'agent a deduit de ce qu'il
    // sait faire pour chaque forme.
    agent.decouvrir(&segments)?;

    println!("DECOUVERTE (1er corpus, {} racines) :", segments.len());
    println!("   inférences effectuees : {}", agent.appel_decouverte);
    println!("   catalogue amorti ({} formes) :", agent.catalogue.len());
    for (forme, dec) in &agent.catalogue {
        println!("      {:<9} -> {:<10} {}", forme, dec.op(), dec.quoi());
    }
    println!("   recette (segments couverts) : {} entrees", agent.recette.len());
};

DECOUVERTE (1er corpus, 5 racines) :


   inférences effectuees : 13


   catalogue amorti (6 formes) :


      csv       -> decouper   colonnes_csv


      json_obj  -> decouper   cles_json


      markdown  -> compresser zlib


      texte     -> compresser zlib


      binaire   -> compresser lzma


      json      -> compresser zlib


   recette (segments couverts) : 13 entrees


In [7]:
{
    /// Representation commune d'un morceau de donnee (reutilisee de Compression-07b).
    ///
    /// L'objet que la boucle percoit et represente : `id` stable, `data` = octets,
    /// `genre` = qualification, `parent`/`profondeur` = lien vers l'origine.
    #[derive(Clone)]
    struct Segment {
        id: String,
        data: Vec<u8>,
        genre: String,
        parent: Option<String>,
        profondeur: i64,
    }

    /// Signature de structure deduite du CONTENU — la perception du segment.
    ///
    /// C'est ce que le cerveau regarde pour decider. Deduite, pas etiquettee :
    /// un fichier nomme `.json` qui ne parse pas n'est pas du JSON.
    struct Info {
        octets: usize,
        texte: bool,
        csv: bool,
        blocs_md: usize,
        json: bool,
        json_obj: bool,
    }

    /// Les booleens Rust s'impriment `true`/`false` ; le jumeau Python impose
    /// `True`/`False`. Toute comparaison de texte passe par cette fonction.
    fn py(b: bool) -> &'static str {
        if b { "True" } else { "False" }
    }

    /// Repr Python d'une liste de chaines : `['a', 'b']`.
    fn py_list_str(items: &[&str]) -> String {
        let mut s = String::from("[");
        for (i, it) in items.iter().enumerate() {
            if i > 0 { s.push_str(", "); }
            s.push('\'');
            s.push_str(it);
            s.push('\'');
        }
        s.push(']');
        s
    }

    /// JSON : mini-dumpeur et mini-parseur maison (`std` seule, aucune crate).
    ///
    /// L'ordre d'insertion des cles d'un objet est PRESERVE (`Vec<(String, Jv)>`) :
    /// le catalogue amorti s'appuie sur cet ordre, un `BTreeMap` le casserait.
    #[derive(Clone)]
    enum Jv {
        Null,
        Bool(bool),
        Int(i64),
        Str(String),
        List(Vec<Jv>),
        Obj(Vec<(String, Jv)>),
    }

    fn json_echapper(s: &str, out: &mut String) {
        out.push('"');
        for c in s.chars() {
            match c {
                '"' => { out.push('\\'); out.push('"'); }
                '\\' => { out.push('\\'); out.push('\\'); }
                '\n' => out.push_str("\\n"),
                '\r' => out.push_str("\\r"),
                '\t' => out.push_str("\\t"),
                c if (c as u32) < 0x20 => out.push_str(&format!("\\u{:04x}", c as u32)),
                c => out.push(c),
            }
        }
        out.push('"');
    }

    /// Dumps equivalent a `json.dumps(indent=2, ensure_ascii=False)`.
    fn json_dump(v: &Jv, level: usize, out: &mut String) {
        let pad = "  ".repeat(level);
        let pad_in = "  ".repeat(level + 1);
        match v {
            Jv::Null => out.push_str("null"),
            Jv::Bool(b) => out.push_str(if *b { "true" } else { "false" }),
            Jv::Int(i) => out.push_str(&i.to_string()),
            Jv::Str(s) => json_echapper(s, out),
            Jv::List(items) => {
                if items.is_empty() { out.push_str("[]"); return; }
                out.push_str("[\n");
                for (i, it) in items.iter().enumerate() {
                    out.push_str(&pad_in);
                    json_dump(it, level + 1, out);
                    if i + 1 < items.len() { out.push(','); }
                    out.push('\n');
                }
                out.push_str(&pad);
                out.push(']');
            }
            Jv::Obj(fields) => {
                if fields.is_empty() { out.push_str("{}"); return; }
                out.push_str("{\n");
                for (i, (k, val)) in fields.iter().enumerate() {
                    out.push_str(&pad_in);
                    json_echapper(k, out);
                    out.push_str(": ");
                    json_dump(val, level + 1, out);
                    if i + 1 < fields.len() { out.push(','); }
                    out.push('\n');
                }
                out.push_str(&pad);
                out.push('}');
            }
        }
    }

    fn json_skip_ws(b: &[u8], i: &mut usize) {
        while *i < b.len() && (b[*i] as char).is_whitespace() { *i += 1; }
    }

    fn json_chaine(b: &[u8], i: &mut usize) -> Option<String> {
        if *i >= b.len() || b[*i] != b'"' { return None; }
        *i += 1;
        let mut s = String::new();
        while *i < b.len() {
            let c = b[*i];
            if c == b'"' { *i += 1; return Some(s); }
            if c == b'\\' {
                *i += 1;
                if *i >= b.len() { return None; }
                let e = b[*i];
                *i += 1;
                match e {
                    b'n' => s.push('\n'),
                    b't' => s.push('\t'),
                    b'r' => s.push('\r'),
                    b'b' => s.push('\u{8}'),
                    b'f' => s.push('\u{c}'),
                    b'/' => s.push('/'),
                    b'"' => s.push('"'),
                    b'\\' => s.push('\\'),
                    b'u' => {
                        if *i + 4 > b.len() { return None; }
                        let hex = std::str::from_utf8(&b[*i..*i + 4]).ok()?;
                        let code = u32::from_str_radix(hex, 16).ok()?;
                        *i += 4;
                        s.push(char::from_u32(code).unwrap_or('\u{fffd}'));
                    }
                    _ => return None,
                }
            } else {
                let start = *i;
                while *i < b.len() && b[*i] != b'"' && b[*i] != b'\\' { *i += 1; }
                s.push_str(std::str::from_utf8(&b[start..*i]).ok()?);
            }
        }
        None
    }

    fn json_valeur(b: &[u8], i: &mut usize) -> Option<Jv> {
        json_skip_ws(b, i);
        if *i >= b.len() { return None; }
        match b[*i] {
            b'{' => {
                *i += 1;
                let mut fields: Vec<(String, Jv)> = Vec::new();
                json_skip_ws(b, i);
                if *i < b.len() && b[*i] == b'}' { *i += 1; return Some(Jv::Obj(fields)); }
                loop {
                    json_skip_ws(b, i);
                    if *i >= b.len() || b[*i] != b'"' { return None; }
                    let k = json_chaine(b, i)?;
                    json_skip_ws(b, i);
                    if *i >= b.len() || b[*i] != b':' { return None; }
                    *i += 1;
                    let v = json_valeur(b, i)?;
                    fields.push((k, v));
                    json_skip_ws(b, i);
                    if *i >= b.len() { return None; }
                    match b[*i] {
                        b',' => { *i += 1; }
                        b'}' => { *i += 1; return Some(Jv::Obj(fields)); }
                        _ => return None,
                    }
                }
            }
            b'[' => {
                *i += 1;
                let mut items: Vec<Jv> = Vec::new();
                json_skip_ws(b, i);
                if *i < b.len() && b[*i] == b']' { *i += 1; return Some(Jv::List(items)); }
                loop {
                    let v = json_valeur(b, i)?;
                    items.push(v);
                    json_skip_ws(b, i);
                    if *i >= b.len() { return None; }
                    match b[*i] {
                        b',' => { *i += 1; }
                        b']' => { *i += 1; return Some(Jv::List(items)); }
                        _ => return None,
                    }
                }
            }
            b'"' => Some(Jv::Str(json_chaine(b, i)?)),
            b't' => {
                if b.len() >= *i + 4 && &b[*i..*i + 4] == b"true" { *i += 4; return Some(Jv::Bool(true)); }
                None
            }
            b'f' => {
                if b.len() >= *i + 5 && &b[*i..*i + 5] == b"false" { *i += 5; return Some(Jv::Bool(false)); }
                None
            }
            b'n' => {
                if b.len() >= *i + 4 && &b[*i..*i + 4] == b"null" { *i += 4; return Some(Jv::Null); }
                None
            }
            _ => {
                let start = *i;
                if b[*i] == b'-' { *i += 1; }
                while *i < b.len() && b[*i].is_ascii_digit() { *i += 1; }
                if start == *i { return None; }
                let txt = std::str::from_utf8(&b[start..*i]).ok()?;
                txt.parse::<i64>().ok().map(Jv::Int)
            }
        }
    }

    /// Parse un document JSON complet ; `None` si le texte n'est pas du JSON.
    fn json_parse(texte: &str) -> Option<Jv> {
        let b = texte.as_bytes();
        let mut i = 0usize;
        let v = json_valeur(b, &mut i)?;
        json_skip_ws(b, &mut i);
        if i != b.len() { return None; }
        Some(v)
    }

    /// Positions (en octets) des titres de markdown : meme regex que le jumeau
    /// Python (`^##? ` en mode multiligne), donc memes decoupes.
    fn positions_titres(texte: &str) -> Vec<usize> {
        let mut out = Vec::new();
        let mut off = 0usize;
        for l in texte.split_inclusive('\n') {
            let t = l.trim_end_matches('\n').trim_end_matches('\r');
            if t.starts_with("# ") || t.starts_with("## ") { out.push(off); }
            off += l.len();
        }
        out
    }

    /// Perception : la signature de structure deduite du contenu.
    fn structure_interne(seg: &Segment) -> Info {
        let mut info = Info {
            octets: seg.data.len(),
            texte: false,
            csv: false,
            blocs_md: 0,
            json: false,
            json_obj: false,
        };
        let texte = match String::from_utf8(seg.data.clone()) {
            Ok(t) => t,
            Err(_) => return info,
        };
        info.texte = true;
        let t = texte.trim();
        if t.starts_with('{') || t.starts_with('[') {
            if let Some(v) = json_parse(&texte) {
                info.json_obj = matches!(v, Jv::Obj(_));
                info.json = true;
                return info;
            }
        }
        let lignes: Vec<&str> = texte.lines().filter(|l| !l.is_empty()).collect();
        if !lignes.is_empty() && lignes[0].contains(',') {
            let n = lignes[0].matches(',').count();
            info.csv = lignes.iter().all(|l| l.matches(',').count() == n);
        }
        info.blocs_md = positions_titres(&texte).len();
        info
    }

    /// Classe de forme d'un segment — la cle du catalogue amorti.
    ///
    /// Distingue l'objet JSON (decoupable par ses cles) du tableau JSON
    /// (compresse tel quel) : deux formes, donc deux decisions distinctes.
    fn signature(s: &Info) -> String {
        if s.json {
            return if s.json_obj { "json_obj".to_string() } else { "json".to_string() };
        }
        if s.csv { return "csv".to_string(); }
        if s.blocs_md > 0 { return "markdown".to_string(); }
        if s.texte { "texte".to_string() } else { "binaire".to_string() }
    }

    /// Un petit corpus heterogene, genere deterministiquement : meme entree a
    /// chaque execution, donc resultat reproductible.
    fn corpus_demo() -> Vec<Segment> {
        let ventes = "nom,quantite,prix\ncafe,10,2.50\nthe,20,1.80\nsucre,5,0.90\nlait,8,1.10\npain,12,0.80\n";
        let clients_obj = Jv::Obj(vec![
            ("id".to_string(), Jv::Int(7)),
            ("panier".to_string(), Jv::List(vec![
                Jv::Obj(vec![
                    ("sku".to_string(), Jv::Str("cafe".to_string())),
                    ("q".to_string(), Jv::Int(10)),
                ]),
                Jv::Obj(vec![
                    ("sku".to_string(), Jv::Str("the".to_string())),
                    ("q".to_string(), Jv::Int(20)),
                ]),
            ])),
        ]);
        let mut clients = String::new();
        json_dump(&clients_obj, 0, &mut clients);
        clients.push('\n');
        let notes = "# Rapport\n\n## Chiffre\n12 kEUR, en hausse.\n\n## Risque\nFournisseur unique, lead time 6 semaines.\n";
        let journal: String = (0..40)
            .map(|i| format!("10:{:02}:00 INFO requete {} ok\n", i, i))
            .collect();
        let mut binaire: Vec<u8> = Vec::with_capacity(1024);
        for _ in 0..4 {
            for b in 0u16..256 { binaire.push(b as u8); }
        }
        vec![
            Segment {
                id: "ventes.csv#0".to_string(),
                data: ventes.as_bytes().to_vec(),
                genre: "donnees/csv".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "clients.json#0".to_string(),
                data: clients.into_bytes(),
                genre: "donnees/json".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "notes.md#0".to_string(),
                data: notes.as_bytes().to_vec(),
                genre: "texte/markdown".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "journal.log#0".to_string(),
                data: journal.into_bytes(),
                genre: "texte/log".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "brut.bin#0".to_string(),
                data: binaire,
                genre: "binaire".to_string(),
                parent: None,
                profondeur: 0,
            },
        ]
    }

    /// La decision d'un cerveau : decouper (reduire) ou compresser.
    ///
    /// Un `enum` remplace le `dict` du jumeau Python : les deux cas sont
    /// nommes, donc le compilateur refuse le cas forget.
    #[derive(Clone)]
    enum Decision {
        Decouper { mode: String },
        Compresser { codec: String },
    }

    impl Decision {
        fn op(&self) -> &'static str {
            match self {
                Decision::Decouper { .. } => "decouper",
                Decision::Compresser { .. } => "compresser",
            }
        }
        fn quoi(&self) -> &str {
            match self {
                Decision::Decouper { mode } => mode,
                Decision::Compresser { codec } => codec,
            }
        }
    }

    /// Point de composition : le contrat d'un mecanisme d'inference.
    ///
    /// La boucle agentique n'appelle JAMAIS un cerveau en particulier, seulement
    /// ce trait. C'est ce contrat unique qui rend le cerveau interchangeable
    /// (deterministe, LLM, hybride). Chaque consultation compte comme une inference.
    trait Cerveau {
        fn inferer(&mut self, seg: &Segment) -> Result<Decision, String>;
    }

    /// La recette : le traitement final retenu pour un segment.
    ///
    /// Un segment est couvert s'il est compresse, ou decoupe en enfants tous
    /// couverts — c'est l'invariant que `verifier_couverture` fait respecter.
    enum Recette {
        Decouper { mode: String, enfants: Vec<String> },
        Compresser { codec: String },
    }

    /// Le squelette agentique : percevoir -> representer -> inferer -> decider
    /// -> verifier -> executer -> observer.
    ///
    /// Compose la representation (Segment), un cerveau (point de composition),
    /// l'invariant de couverture (verification) et l'application de la recette
    /// (execution). L'inference est isolee derriere le trait `Cerveau`.
    struct AgentTriage<'a> {
        cerveau: &'a mut dyn Cerveau,
        borne: i64,
        catalogue: Vec<(String, Decision)>,
        recette: Vec<(String, Recette)>,
        appel_decouverte: i64,
    }

    impl<'a> AgentTriage<'a> {
        /// Apprend le catalogue forme->decision en parcourant le corpus.
        ///
        /// Chaque consultation du cerveau compte. Si le parcours atteint la borne
        /// sans epuiser la file, la decouverte ECHOUE (safety bound) : on ne
        /// laisse jamais l'agent tourner a l'infini.
        fn decouvrir(&mut self, segments: &[Segment]) -> Result<(), String> {
            self.catalogue = Vec::new();
            self.recette = Vec::new();
            self.appel_decouverte = 0;
            let mut file: Vec<Segment> = segments.to_vec();
            while !file.is_empty() {
                // La borne est verifiee AVANT le retrait : c'est elle qui
                // distingue une decouverte bornee d'une decouverte divergente.
                if self.appel_decouverte >= self.borne {
                    return Err(format!(
                        "decouverte interrompue a {} inférences, {} segment(s) en attente — safety bound.",
                        self.appel_decouverte,
                        file.len()
                    ));
                }
                let seg = file.remove(0);
                let dec = self.cerveau.inferer(&seg)?; // inferer + decider (compte)
                self.appel_decouverte += 1;
                let forme = signature(&structure_interne(&seg));
                self.cat_set(&forme, dec.clone());
                if let Decision::Decouper { mode } = &dec {
                    let enfants = decouper(&seg, mode);
                    // La recette porte les ENFANTS reels, pas seulement la
                    // decision : c'est cette liste que l'invariant parcourt.
                    let ids: Vec<String> = enfants.iter().map(|e| e.id.clone()).collect();
                    self.recette.push((
                        seg.id.clone(),
                        Recette::Decouper { mode: mode.clone(), enfants: ids },
                    ));
                    file.extend(enfants);
                } else {
                    self.recette.push((seg.id.clone(), Recette::Compresser { codec: dec.quoi().to_string() }));
                }
            }
            let racines: Vec<&str> = segments.iter().map(|s| s.id.as_str()).collect();
            verifier_couverture(&racines, &self.recette)?; // verifier
            Ok(())
        }

        /// Le catalogue garde l'ordre de PREMIERE insertion : une forme deja
        /// connue est remplacee en place, pas appendue.
        fn cat_set(&mut self, forme: &str, dec: Decision) {
            for (k, v) in self.catalogue.iter_mut() {
                if k == forme {
                    *v = dec;
                    return;
                }
            }
            self.catalogue.push((forme.to_string(), dec));
        }

        /// Applique le catalogue appris a de nouveaux segments, SANS re-inferer.
        ///
        /// C'est l'amortissement : la decouverte a paye le catalogue, le rejeu le
        /// consomme a cout d'inference NUL.
        fn rejouer(&self, segments: &[Segment]) -> Vec<(String, Decision)> {
            segments
                .iter()
                .map(|seg| {
                    let forme = signature(&structure_interne(seg));
                    let dec = self
                        .catalogue
                        .iter()
                        .find(|(k, _)| *k == forme)
                        .map(|(_, v)| v.clone())
                        .unwrap_or(Decision::Compresser { codec: "zlib".to_string() });
                    (seg.id.clone(), dec)
                })
                .collect()
        }
    }

    /// Un segment est couvert s'il est compresse, ou decoupe en enfants tous
    /// couverts. Le cycle est detecte : la recette ne doit pas boucler.
    fn couverture(
        sid: &str,
        recette: &[(String, Recette)],
        vus: &mut Vec<String>,
    ) -> Result<Vec<String>, String> {
        if vus.iter().any(|v| v == sid) {
            return Err(format!("cycle dans la recette autour de {}", sid));
        }
        vus.push(sid.to_string());
        let entree = recette.iter().find(|(k, _)| k == sid).map(|(_, v)| v);
        let res = match entree {
            None => Ok(vec![sid.to_string()]),
            Some(Recette::Compresser { .. }) => Ok(Vec::new()),
            Some(Recette::Decouper { enfants, .. }) => {
                let mut manquants: Vec<String> = Vec::new();
                for enfant in enfants {
                    match couverture(enfant, recette, vus) {
                        Ok(abs) => manquants.extend(abs),
                        Err(msg) => {
                            vus.pop();
                            return Err(msg);
                        }
                    }
                }
                Ok(manquants)
            }
        };
        vus.pop();
        res
    }

    /// Invariant : leve en nommant tout segment original non traite.
    fn verifier_couverture(racines: &[&str], recette: &[(String, Recette)]) -> Result<(), String> {
        let mut manquants: Vec<String> = Vec::new();
        for sid in racines {
            match couverture(sid, recette, &mut Vec::new()) {
                Ok(abs) => manquants.extend(abs),
                Err(msg) => return Err(msg),
            }
        }
        if !manquants.is_empty() {
            let liste: Vec<&str> = manquants.iter().take(6).map(|s| s.as_str()).collect();
            return Err(format!(
                "{} segment(s) sans traitement final : {}",
                manquants.len(),
                py_list_str(&liste)
            ));
        }
        Ok(())
    }

    /// Operation inversible : un segment macro -> des fragments qualifies.
    fn decouper(seg: &Segment, mode: &str) -> Vec<Segment> {
        let texte = match String::from_utf8(seg.data.clone()) {
            Ok(t) => t,
            Err(_) => return Vec::new(),
        };
        let mut kids: Vec<Segment> = Vec::new();
        match mode {
            "blocs_markdown" => {
                let mut bornes: Vec<usize> = vec![0];
                bornes.extend(positions_titres(&texte));
                bornes.push(texte.len());
                for i in 0..bornes.len() - 1 {
                    let m = &texte[bornes[i]..bornes[i + 1]];
                    if !m.trim().is_empty() {
                        kids.push(Segment {
                            id: format!("{}.b{}", seg.id, kids.len()),
                            data: m.as_bytes().to_vec(),
                            genre: format!("{}/bloc", seg.genre),
                            parent: Some(seg.id.clone()),
                            profondeur: seg.profondeur + 1,
                        });
                    }
                }
            }
            "colonnes_csv" => {
                let lignes: Vec<&str> = texte.lines().filter(|l| !l.is_empty()).collect();
                if lignes.is_empty() { return kids; }
                let cols: Vec<&str> = lignes[0].split(',').collect();
                let mat: Vec<Vec<&str>> = lignes.iter().map(|l| l.split(',').collect()).collect();
                for j in 0..cols.len() {
                    let mut s = String::new();
                    for (r, row) in mat.iter().enumerate() {
                        if r > 0 { s.push('\n'); }
                        s.push_str(row[j]);
                    }
                    kids.push(Segment {
                        id: format!("{}.c{}", seg.id, j),
                        data: s.into_bytes(),
                        genre: format!("{}/col:{}", seg.genre, cols[j]),
                        parent: Some(seg.id.clone()),
                        profondeur: seg.profondeur + 1,
                    });
                }
            }
            "cles_json" => {
                let champs = match json_parse(&texte) {
                    Some(Jv::Obj(f)) => f,
                    _ => return kids,
                };
                for (k, v) in &champs {
                    let mut s = String::new();
                    json_dump(v, 0, &mut s);
                    kids.push(Segment {
                        id: format!("{}.k{}", seg.id, kids.len()),
                        data: s.into_bytes(),
                        genre: format!("{}/cle:{}", seg.genre, k),
                        parent: Some(seg.id.clone()),
                        profondeur: seg.profondeur + 1,
                    });
                }
            }
            _ => {}
        }
        kids
    }

    /// L'inverse d'un decoupage : reconstruit les octets du segment parent.
    fn recoller(mode: &str, enfants: &[Segment]) -> Vec<u8> {
        match mode {
            "blocs_markdown" => {
                let mut out = Vec::new();
                for e in enfants { out.extend_from_slice(&e.data); }
                out
            }
            "colonnes_csv" => {
                let mat: Vec<Vec<String>> = enfants
                    .iter()
                    .map(|e| String::from_utf8_lossy(&e.data).split('\n').map(|s| s.to_string()).collect())
                    .collect();
                if mat.is_empty() { return Vec::new(); }
                let n = mat[0].len();
                let mut lignes: Vec<String> = Vec::new();
                for row in 0..n {
                    let parts: Vec<&str> = mat.iter().map(|c| c[row].as_str()).collect();
                    lignes.push(parts.join(","));
                }
                let mut s = lignes.join("\n");
                s.push('\n');
                s.into_bytes()
            }
            "cles_json" => {
                let mut champs: Vec<(String, Jv)> = Vec::new();
                for e in enfants {
                    let k = e.genre.rsplit(':').next().unwrap_or("").to_string();
                    let v = json_parse(&String::from_utf8_lossy(&e.data)).unwrap_or(Jv::Null);
                    champs.push((k, v));
                }
                let mut s = String::new();
                json_dump(&Jv::Obj(champs), 0, &mut s);
                s.push('\n');
                s.into_bytes()
            }
            _ => Vec::new(),
        }
    }

    /// Le cerveau par defaut : derive la decision de la structure percue.
    struct CerveauDeterministe { appels: i64 }

    impl Cerveau for CerveauDeterministe {
        fn inferer(&mut self, seg: &Segment) -> Result<Decision, String> {
            self.appels += 1;
            let s = structure_interne(seg);
            if seg.data.len() < 64 {
                return Ok(Decision::Compresser {
                    codec: if s.texte { "zlib".to_string() } else { "lzma".to_string() },
                });
            }
            let mode = match signature(&s).as_str() {
                "json_obj" => "cles_json",
                "csv" => "colonnes_csv",
                "markdown" => "blocs_markdown",
                _ => {
                    return Ok(Decision::Compresser {
                        codec: if s.texte { "zlib".to_string() } else { "lzma".to_string() },
                    })
                }
            };
            Ok(Decision::Decouper { mode: mode.to_string() })
        }
    }

    let segments = corpus_demo();
    let mut cerveau = CerveauDeterministe { appels: 0 };
    let mut agent = AgentTriage {
        cerveau: &mut cerveau,
        borne: 64,
        catalogue: Vec::new(),
        recette: Vec::new(),
        appel_decouverte: 0,
    };
    // Decouverte : on instancie l'agent (cerveau deterministe, borne 64
    // octets), on lui fait percevoir le corpus, puis on lit le catalogue
    // amorti de formes et la recette — ce que l'agent a deduit de ce qu'il
    // sait faire pour chaque forme.
    agent.decouvrir(&segments)?;

    // Un DEUXIEME corpus, meme forme, donnees differentes : le rejeu
    // consomme le catalogue appris, sans re-inferer une seule fois. On
    // rejoue donc sur un corpus de meme forme, et le compteur d'inferences
    // du rejeu reste a zero : c'est tout l'amortissement.
    let ventes2 = "nom,quantite,prix
cafe,4,2.50
biere,16,3.00
";
    let nouveau = vec![Segment {
        id: "ventes2.csv#0".to_string(),
        data: ventes2.as_bytes().to_vec(),
        genre: "donnees/csv".to_string(),
        parent: None,
        profondeur: 0,
    }];
    let plan = agent.rejouer(&nouveau);
    println!("REJEU (2e corpus, {} racine) :", nouveau.len());
    for (sid, dec) in &plan {
        println!("   {:<16} -> {:<10} {}   (0 inférence)", sid, dec.op(), dec.quoi());
    }
    println!("   inférences effectuees : 0  (le catalogue a ete consulte, pas le cerveau)");
};

REJEU (2e corpus, 1 racine) :


   ventes2.csv#0    -> decouper   colonnes_csv   (0 inférence)


   inférences effectuees : 0  (le catalogue a ete consulte, pas le cerveau)


In [8]:
{
    /// Representation commune d'un morceau de donnee (reutilisee de Compression-07b).
    ///
    /// L'objet que la boucle percoit et represente : `id` stable, `data` = octets,
    /// `genre` = qualification, `parent`/`profondeur` = lien vers l'origine.
    #[derive(Clone)]
    struct Segment {
        id: String,
        data: Vec<u8>,
        genre: String,
        parent: Option<String>,
        profondeur: i64,
    }

    /// Signature de structure deduite du CONTENU — la perception du segment.
    ///
    /// C'est ce que le cerveau regarde pour decider. Deduite, pas etiquettee :
    /// un fichier nomme `.json` qui ne parse pas n'est pas du JSON.
    struct Info {
        octets: usize,
        texte: bool,
        csv: bool,
        blocs_md: usize,
        json: bool,
        json_obj: bool,
    }

    /// Les booleens Rust s'impriment `true`/`false` ; le jumeau Python impose
    /// `True`/`False`. Toute comparaison de texte passe par cette fonction.
    fn py(b: bool) -> &'static str {
        if b { "True" } else { "False" }
    }

    /// Repr Python d'une liste de chaines : `['a', 'b']`.
    fn py_list_str(items: &[&str]) -> String {
        let mut s = String::from("[");
        for (i, it) in items.iter().enumerate() {
            if i > 0 { s.push_str(", "); }
            s.push('\'');
            s.push_str(it);
            s.push('\'');
        }
        s.push(']');
        s
    }

    /// JSON : mini-dumpeur et mini-parseur maison (`std` seule, aucune crate).
    ///
    /// L'ordre d'insertion des cles d'un objet est PRESERVE (`Vec<(String, Jv)>`) :
    /// le catalogue amorti s'appuie sur cet ordre, un `BTreeMap` le casserait.
    #[derive(Clone)]
    enum Jv {
        Null,
        Bool(bool),
        Int(i64),
        Str(String),
        List(Vec<Jv>),
        Obj(Vec<(String, Jv)>),
    }

    fn json_echapper(s: &str, out: &mut String) {
        out.push('"');
        for c in s.chars() {
            match c {
                '"' => { out.push('\\'); out.push('"'); }
                '\\' => { out.push('\\'); out.push('\\'); }
                '\n' => out.push_str("\\n"),
                '\r' => out.push_str("\\r"),
                '\t' => out.push_str("\\t"),
                c if (c as u32) < 0x20 => out.push_str(&format!("\\u{:04x}", c as u32)),
                c => out.push(c),
            }
        }
        out.push('"');
    }

    /// Dumps equivalent a `json.dumps(indent=2, ensure_ascii=False)`.
    fn json_dump(v: &Jv, level: usize, out: &mut String) {
        let pad = "  ".repeat(level);
        let pad_in = "  ".repeat(level + 1);
        match v {
            Jv::Null => out.push_str("null"),
            Jv::Bool(b) => out.push_str(if *b { "true" } else { "false" }),
            Jv::Int(i) => out.push_str(&i.to_string()),
            Jv::Str(s) => json_echapper(s, out),
            Jv::List(items) => {
                if items.is_empty() { out.push_str("[]"); return; }
                out.push_str("[\n");
                for (i, it) in items.iter().enumerate() {
                    out.push_str(&pad_in);
                    json_dump(it, level + 1, out);
                    if i + 1 < items.len() { out.push(','); }
                    out.push('\n');
                }
                out.push_str(&pad);
                out.push(']');
            }
            Jv::Obj(fields) => {
                if fields.is_empty() { out.push_str("{}"); return; }
                out.push_str("{\n");
                for (i, (k, val)) in fields.iter().enumerate() {
                    out.push_str(&pad_in);
                    json_echapper(k, out);
                    out.push_str(": ");
                    json_dump(val, level + 1, out);
                    if i + 1 < fields.len() { out.push(','); }
                    out.push('\n');
                }
                out.push_str(&pad);
                out.push('}');
            }
        }
    }

    fn json_skip_ws(b: &[u8], i: &mut usize) {
        while *i < b.len() && (b[*i] as char).is_whitespace() { *i += 1; }
    }

    fn json_chaine(b: &[u8], i: &mut usize) -> Option<String> {
        if *i >= b.len() || b[*i] != b'"' { return None; }
        *i += 1;
        let mut s = String::new();
        while *i < b.len() {
            let c = b[*i];
            if c == b'"' { *i += 1; return Some(s); }
            if c == b'\\' {
                *i += 1;
                if *i >= b.len() { return None; }
                let e = b[*i];
                *i += 1;
                match e {
                    b'n' => s.push('\n'),
                    b't' => s.push('\t'),
                    b'r' => s.push('\r'),
                    b'b' => s.push('\u{8}'),
                    b'f' => s.push('\u{c}'),
                    b'/' => s.push('/'),
                    b'"' => s.push('"'),
                    b'\\' => s.push('\\'),
                    b'u' => {
                        if *i + 4 > b.len() { return None; }
                        let hex = std::str::from_utf8(&b[*i..*i + 4]).ok()?;
                        let code = u32::from_str_radix(hex, 16).ok()?;
                        *i += 4;
                        s.push(char::from_u32(code).unwrap_or('\u{fffd}'));
                    }
                    _ => return None,
                }
            } else {
                let start = *i;
                while *i < b.len() && b[*i] != b'"' && b[*i] != b'\\' { *i += 1; }
                s.push_str(std::str::from_utf8(&b[start..*i]).ok()?);
            }
        }
        None
    }

    fn json_valeur(b: &[u8], i: &mut usize) -> Option<Jv> {
        json_skip_ws(b, i);
        if *i >= b.len() { return None; }
        match b[*i] {
            b'{' => {
                *i += 1;
                let mut fields: Vec<(String, Jv)> = Vec::new();
                json_skip_ws(b, i);
                if *i < b.len() && b[*i] == b'}' { *i += 1; return Some(Jv::Obj(fields)); }
                loop {
                    json_skip_ws(b, i);
                    if *i >= b.len() || b[*i] != b'"' { return None; }
                    let k = json_chaine(b, i)?;
                    json_skip_ws(b, i);
                    if *i >= b.len() || b[*i] != b':' { return None; }
                    *i += 1;
                    let v = json_valeur(b, i)?;
                    fields.push((k, v));
                    json_skip_ws(b, i);
                    if *i >= b.len() { return None; }
                    match b[*i] {
                        b',' => { *i += 1; }
                        b'}' => { *i += 1; return Some(Jv::Obj(fields)); }
                        _ => return None,
                    }
                }
            }
            b'[' => {
                *i += 1;
                let mut items: Vec<Jv> = Vec::new();
                json_skip_ws(b, i);
                if *i < b.len() && b[*i] == b']' { *i += 1; return Some(Jv::List(items)); }
                loop {
                    let v = json_valeur(b, i)?;
                    items.push(v);
                    json_skip_ws(b, i);
                    if *i >= b.len() { return None; }
                    match b[*i] {
                        b',' => { *i += 1; }
                        b']' => { *i += 1; return Some(Jv::List(items)); }
                        _ => return None,
                    }
                }
            }
            b'"' => Some(Jv::Str(json_chaine(b, i)?)),
            b't' => {
                if b.len() >= *i + 4 && &b[*i..*i + 4] == b"true" { *i += 4; return Some(Jv::Bool(true)); }
                None
            }
            b'f' => {
                if b.len() >= *i + 5 && &b[*i..*i + 5] == b"false" { *i += 5; return Some(Jv::Bool(false)); }
                None
            }
            b'n' => {
                if b.len() >= *i + 4 && &b[*i..*i + 4] == b"null" { *i += 4; return Some(Jv::Null); }
                None
            }
            _ => {
                let start = *i;
                if b[*i] == b'-' { *i += 1; }
                while *i < b.len() && b[*i].is_ascii_digit() { *i += 1; }
                if start == *i { return None; }
                let txt = std::str::from_utf8(&b[start..*i]).ok()?;
                txt.parse::<i64>().ok().map(Jv::Int)
            }
        }
    }

    /// Parse un document JSON complet ; `None` si le texte n'est pas du JSON.
    fn json_parse(texte: &str) -> Option<Jv> {
        let b = texte.as_bytes();
        let mut i = 0usize;
        let v = json_valeur(b, &mut i)?;
        json_skip_ws(b, &mut i);
        if i != b.len() { return None; }
        Some(v)
    }

    /// Positions (en octets) des titres de markdown : meme regex que le jumeau
    /// Python (`^##? ` en mode multiligne), donc memes decoupes.
    fn positions_titres(texte: &str) -> Vec<usize> {
        let mut out = Vec::new();
        let mut off = 0usize;
        for l in texte.split_inclusive('\n') {
            let t = l.trim_end_matches('\n').trim_end_matches('\r');
            if t.starts_with("# ") || t.starts_with("## ") { out.push(off); }
            off += l.len();
        }
        out
    }

    /// Perception : la signature de structure deduite du contenu.
    fn structure_interne(seg: &Segment) -> Info {
        let mut info = Info {
            octets: seg.data.len(),
            texte: false,
            csv: false,
            blocs_md: 0,
            json: false,
            json_obj: false,
        };
        let texte = match String::from_utf8(seg.data.clone()) {
            Ok(t) => t,
            Err(_) => return info,
        };
        info.texte = true;
        let t = texte.trim();
        if t.starts_with('{') || t.starts_with('[') {
            if let Some(v) = json_parse(&texte) {
                info.json_obj = matches!(v, Jv::Obj(_));
                info.json = true;
                return info;
            }
        }
        let lignes: Vec<&str> = texte.lines().filter(|l| !l.is_empty()).collect();
        if !lignes.is_empty() && lignes[0].contains(',') {
            let n = lignes[0].matches(',').count();
            info.csv = lignes.iter().all(|l| l.matches(',').count() == n);
        }
        info.blocs_md = positions_titres(&texte).len();
        info
    }

    /// Classe de forme d'un segment — la cle du catalogue amorti.
    ///
    /// Distingue l'objet JSON (decoupable par ses cles) du tableau JSON
    /// (compresse tel quel) : deux formes, donc deux decisions distinctes.
    fn signature(s: &Info) -> String {
        if s.json {
            return if s.json_obj { "json_obj".to_string() } else { "json".to_string() };
        }
        if s.csv { return "csv".to_string(); }
        if s.blocs_md > 0 { return "markdown".to_string(); }
        if s.texte { "texte".to_string() } else { "binaire".to_string() }
    }

    /// Un petit corpus heterogene, genere deterministiquement : meme entree a
    /// chaque execution, donc resultat reproductible.
    fn corpus_demo() -> Vec<Segment> {
        let ventes = "nom,quantite,prix\ncafe,10,2.50\nthe,20,1.80\nsucre,5,0.90\nlait,8,1.10\npain,12,0.80\n";
        let clients_obj = Jv::Obj(vec![
            ("id".to_string(), Jv::Int(7)),
            ("panier".to_string(), Jv::List(vec![
                Jv::Obj(vec![
                    ("sku".to_string(), Jv::Str("cafe".to_string())),
                    ("q".to_string(), Jv::Int(10)),
                ]),
                Jv::Obj(vec![
                    ("sku".to_string(), Jv::Str("the".to_string())),
                    ("q".to_string(), Jv::Int(20)),
                ]),
            ])),
        ]);
        let mut clients = String::new();
        json_dump(&clients_obj, 0, &mut clients);
        clients.push('\n');
        let notes = "# Rapport\n\n## Chiffre\n12 kEUR, en hausse.\n\n## Risque\nFournisseur unique, lead time 6 semaines.\n";
        let journal: String = (0..40)
            .map(|i| format!("10:{:02}:00 INFO requete {} ok\n", i, i))
            .collect();
        let mut binaire: Vec<u8> = Vec::with_capacity(1024);
        for _ in 0..4 {
            for b in 0u16..256 { binaire.push(b as u8); }
        }
        vec![
            Segment {
                id: "ventes.csv#0".to_string(),
                data: ventes.as_bytes().to_vec(),
                genre: "donnees/csv".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "clients.json#0".to_string(),
                data: clients.into_bytes(),
                genre: "donnees/json".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "notes.md#0".to_string(),
                data: notes.as_bytes().to_vec(),
                genre: "texte/markdown".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "journal.log#0".to_string(),
                data: journal.into_bytes(),
                genre: "texte/log".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "brut.bin#0".to_string(),
                data: binaire,
                genre: "binaire".to_string(),
                parent: None,
                profondeur: 0,
            },
        ]
    }

    /// La decision d'un cerveau : decouper (reduire) ou compresser.
    ///
    /// Un `enum` remplace le `dict` du jumeau Python : les deux cas sont
    /// nommes, donc le compilateur refuse le cas forget.
    #[derive(Clone)]
    enum Decision {
        Decouper { mode: String },
        Compresser { codec: String },
    }

    impl Decision {
        fn op(&self) -> &'static str {
            match self {
                Decision::Decouper { .. } => "decouper",
                Decision::Compresser { .. } => "compresser",
            }
        }
        fn quoi(&self) -> &str {
            match self {
                Decision::Decouper { mode } => mode,
                Decision::Compresser { codec } => codec,
            }
        }
    }

    /// Point de composition : le contrat d'un mecanisme d'inference.
    ///
    /// La boucle agentique n'appelle JAMAIS un cerveau en particulier, seulement
    /// ce trait. C'est ce contrat unique qui rend le cerveau interchangeable
    /// (deterministe, LLM, hybride). Chaque consultation compte comme une inference.
    trait Cerveau {
        fn inferer(&mut self, seg: &Segment) -> Result<Decision, String>;
    }

    /// La recette : le traitement final retenu pour un segment.
    ///
    /// Un segment est couvert s'il est compresse, ou decoupe en enfants tous
    /// couverts — c'est l'invariant que `verifier_couverture` fait respecter.
    enum Recette {
        Decouper { mode: String, enfants: Vec<String> },
        Compresser { codec: String },
    }

    /// Le squelette agentique : percevoir -> representer -> inferer -> decider
    /// -> verifier -> executer -> observer.
    ///
    /// Compose la representation (Segment), un cerveau (point de composition),
    /// l'invariant de couverture (verification) et l'application de la recette
    /// (execution). L'inference est isolee derriere le trait `Cerveau`.
    struct AgentTriage<'a> {
        cerveau: &'a mut dyn Cerveau,
        borne: i64,
        catalogue: Vec<(String, Decision)>,
        recette: Vec<(String, Recette)>,
        appel_decouverte: i64,
    }

    impl<'a> AgentTriage<'a> {
        /// Apprend le catalogue forme->decision en parcourant le corpus.
        ///
        /// Chaque consultation du cerveau compte. Si le parcours atteint la borne
        /// sans epuiser la file, la decouverte ECHOUE (safety bound) : on ne
        /// laisse jamais l'agent tourner a l'infini.
        fn decouvrir(&mut self, segments: &[Segment]) -> Result<(), String> {
            self.catalogue = Vec::new();
            self.recette = Vec::new();
            self.appel_decouverte = 0;
            let mut file: Vec<Segment> = segments.to_vec();
            while !file.is_empty() {
                // La borne est verifiee AVANT le retrait : c'est elle qui
                // distingue une decouverte bornee d'une decouverte divergente.
                if self.appel_decouverte >= self.borne {
                    return Err(format!(
                        "decouverte interrompue a {} inférences, {} segment(s) en attente — safety bound.",
                        self.appel_decouverte,
                        file.len()
                    ));
                }
                let seg = file.remove(0);
                let dec = self.cerveau.inferer(&seg)?; // inferer + decider (compte)
                self.appel_decouverte += 1;
                let forme = signature(&structure_interne(&seg));
                self.cat_set(&forme, dec.clone());
                if let Decision::Decouper { mode } = &dec {
                    let enfants = decouper(&seg, mode);
                    // La recette porte les ENFANTS reels, pas seulement la
                    // decision : c'est cette liste que l'invariant parcourt.
                    let ids: Vec<String> = enfants.iter().map(|e| e.id.clone()).collect();
                    self.recette.push((
                        seg.id.clone(),
                        Recette::Decouper { mode: mode.clone(), enfants: ids },
                    ));
                    file.extend(enfants);
                } else {
                    self.recette.push((seg.id.clone(), Recette::Compresser { codec: dec.quoi().to_string() }));
                }
            }
            let racines: Vec<&str> = segments.iter().map(|s| s.id.as_str()).collect();
            verifier_couverture(&racines, &self.recette)?; // verifier
            Ok(())
        }

        /// Le catalogue garde l'ordre de PREMIERE insertion : une forme deja
        /// connue est remplacee en place, pas appendue.
        fn cat_set(&mut self, forme: &str, dec: Decision) {
            for (k, v) in self.catalogue.iter_mut() {
                if k == forme {
                    *v = dec;
                    return;
                }
            }
            self.catalogue.push((forme.to_string(), dec));
        }

        /// Applique le catalogue appris a de nouveaux segments, SANS re-inferer.
        ///
        /// C'est l'amortissement : la decouverte a paye le catalogue, le rejeu le
        /// consomme a cout d'inference NUL.
        fn rejouer(&self, segments: &[Segment]) -> Vec<(String, Decision)> {
            segments
                .iter()
                .map(|seg| {
                    let forme = signature(&structure_interne(seg));
                    let dec = self
                        .catalogue
                        .iter()
                        .find(|(k, _)| *k == forme)
                        .map(|(_, v)| v.clone())
                        .unwrap_or(Decision::Compresser { codec: "zlib".to_string() });
                    (seg.id.clone(), dec)
                })
                .collect()
        }
    }

    /// Un segment est couvert s'il est compresse, ou decoupe en enfants tous
    /// couverts. Le cycle est detecte : la recette ne doit pas boucler.
    fn couverture(
        sid: &str,
        recette: &[(String, Recette)],
        vus: &mut Vec<String>,
    ) -> Result<Vec<String>, String> {
        if vus.iter().any(|v| v == sid) {
            return Err(format!("cycle dans la recette autour de {}", sid));
        }
        vus.push(sid.to_string());
        let entree = recette.iter().find(|(k, _)| k == sid).map(|(_, v)| v);
        let res = match entree {
            None => Ok(vec![sid.to_string()]),
            Some(Recette::Compresser { .. }) => Ok(Vec::new()),
            Some(Recette::Decouper { enfants, .. }) => {
                let mut manquants: Vec<String> = Vec::new();
                for enfant in enfants {
                    match couverture(enfant, recette, vus) {
                        Ok(abs) => manquants.extend(abs),
                        Err(msg) => {
                            vus.pop();
                            return Err(msg);
                        }
                    }
                }
                Ok(manquants)
            }
        };
        vus.pop();
        res
    }

    /// Invariant : leve en nommant tout segment original non traite.
    fn verifier_couverture(racines: &[&str], recette: &[(String, Recette)]) -> Result<(), String> {
        let mut manquants: Vec<String> = Vec::new();
        for sid in racines {
            match couverture(sid, recette, &mut Vec::new()) {
                Ok(abs) => manquants.extend(abs),
                Err(msg) => return Err(msg),
            }
        }
        if !manquants.is_empty() {
            let liste: Vec<&str> = manquants.iter().take(6).map(|s| s.as_str()).collect();
            return Err(format!(
                "{} segment(s) sans traitement final : {}",
                manquants.len(),
                py_list_str(&liste)
            ));
        }
        Ok(())
    }

    /// Operation inversible : un segment macro -> des fragments qualifies.
    fn decouper(seg: &Segment, mode: &str) -> Vec<Segment> {
        let texte = match String::from_utf8(seg.data.clone()) {
            Ok(t) => t,
            Err(_) => return Vec::new(),
        };
        let mut kids: Vec<Segment> = Vec::new();
        match mode {
            "blocs_markdown" => {
                let mut bornes: Vec<usize> = vec![0];
                bornes.extend(positions_titres(&texte));
                bornes.push(texte.len());
                for i in 0..bornes.len() - 1 {
                    let m = &texte[bornes[i]..bornes[i + 1]];
                    if !m.trim().is_empty() {
                        kids.push(Segment {
                            id: format!("{}.b{}", seg.id, kids.len()),
                            data: m.as_bytes().to_vec(),
                            genre: format!("{}/bloc", seg.genre),
                            parent: Some(seg.id.clone()),
                            profondeur: seg.profondeur + 1,
                        });
                    }
                }
            }
            "colonnes_csv" => {
                let lignes: Vec<&str> = texte.lines().filter(|l| !l.is_empty()).collect();
                if lignes.is_empty() { return kids; }
                let cols: Vec<&str> = lignes[0].split(',').collect();
                let mat: Vec<Vec<&str>> = lignes.iter().map(|l| l.split(',').collect()).collect();
                for j in 0..cols.len() {
                    let mut s = String::new();
                    for (r, row) in mat.iter().enumerate() {
                        if r > 0 { s.push('\n'); }
                        s.push_str(row[j]);
                    }
                    kids.push(Segment {
                        id: format!("{}.c{}", seg.id, j),
                        data: s.into_bytes(),
                        genre: format!("{}/col:{}", seg.genre, cols[j]),
                        parent: Some(seg.id.clone()),
                        profondeur: seg.profondeur + 1,
                    });
                }
            }
            "cles_json" => {
                let champs = match json_parse(&texte) {
                    Some(Jv::Obj(f)) => f,
                    _ => return kids,
                };
                for (k, v) in &champs {
                    let mut s = String::new();
                    json_dump(v, 0, &mut s);
                    kids.push(Segment {
                        id: format!("{}.k{}", seg.id, kids.len()),
                        data: s.into_bytes(),
                        genre: format!("{}/cle:{}", seg.genre, k),
                        parent: Some(seg.id.clone()),
                        profondeur: seg.profondeur + 1,
                    });
                }
            }
            _ => {}
        }
        kids
    }

    /// L'inverse d'un decoupage : reconstruit les octets du segment parent.
    fn recoller(mode: &str, enfants: &[Segment]) -> Vec<u8> {
        match mode {
            "blocs_markdown" => {
                let mut out = Vec::new();
                for e in enfants { out.extend_from_slice(&e.data); }
                out
            }
            "colonnes_csv" => {
                let mat: Vec<Vec<String>> = enfants
                    .iter()
                    .map(|e| String::from_utf8_lossy(&e.data).split('\n').map(|s| s.to_string()).collect())
                    .collect();
                if mat.is_empty() { return Vec::new(); }
                let n = mat[0].len();
                let mut lignes: Vec<String> = Vec::new();
                for row in 0..n {
                    let parts: Vec<&str> = mat.iter().map(|c| c[row].as_str()).collect();
                    lignes.push(parts.join(","));
                }
                let mut s = lignes.join("\n");
                s.push('\n');
                s.into_bytes()
            }
            "cles_json" => {
                let mut champs: Vec<(String, Jv)> = Vec::new();
                for e in enfants {
                    let k = e.genre.rsplit(':').next().unwrap_or("").to_string();
                    let v = json_parse(&String::from_utf8_lossy(&e.data)).unwrap_or(Jv::Null);
                    champs.push((k, v));
                }
                let mut s = String::new();
                json_dump(&Jv::Obj(champs), 0, &mut s);
                s.push('\n');
                s.into_bytes()
            }
            _ => Vec::new(),
        }
    }

    /// Le cerveau par defaut : derive la decision de la structure percue.
    struct CerveauDeterministe { appels: i64 }

    impl Cerveau for CerveauDeterministe {
        fn inferer(&mut self, seg: &Segment) -> Result<Decision, String> {
            self.appels += 1;
            let s = structure_interne(seg);
            if seg.data.len() < 64 {
                return Ok(Decision::Compresser {
                    codec: if s.texte { "zlib".to_string() } else { "lzma".to_string() },
                });
            }
            let mode = match signature(&s).as_str() {
                "json_obj" => "cles_json",
                "csv" => "colonnes_csv",
                "markdown" => "blocs_markdown",
                _ => {
                    return Ok(Decision::Compresser {
                        codec: if s.texte { "zlib".to_string() } else { "lzma".to_string() },
                    })
                }
            };
            Ok(Decision::Decouper { mode: mode.to_string() })
        }
    }

    let segments = corpus_demo();
    let mut cerveau = CerveauDeterministe { appels: 0 };
    let mut agent = AgentTriage {
        cerveau: &mut cerveau,
        borne: 64,
        catalogue: Vec::new(),
        recette: Vec::new(),
        appel_decouverte: 0,
    };
    // Decouverte : on instancie l'agent (cerveau deterministe, borne 64
    // octets), on lui fait percevoir le corpus, puis on lit le catalogue
    // amorti de formes et la recette — ce que l'agent a deduit de ce qu'il
    // sait faire pour chaque forme.
    agent.decouvrir(&segments)?;

    // L'observation : le cout d'inference est PAYE une fois (decouverte)
    // puis amorti a zero (rejeu). C'est la these de Compression-07b,
    // reformulee comme propriete d'un agent composable : la connaissance
    // forme->decision est un ACTIF amortissable.
    println!("OBSERVATION — amortissement du cout d'inference :");
    println!("   decouverte : {} inférences   (politique apprise)", agent.appel_decouverte);
    println!("   rejeu      : 0 inférences       (politique consommee)");
    println!("   -> sur 1000 corpus de meme forme, on re-infererait {} fois, pas 1000 x {}.", agent.appel_decouverte, agent.appel_decouverte);
};

OBSERVATION — amortissement du cout d'inference :


   decouverte : 13 inférences   (politique apprise)


   rejeu      : 0 inférences       (politique consommee)


   -> sur 1000 corpus de meme forme, on re-infererait 13 fois, pas 1000 x 13.


### Lecture du résultat

C'est le **résultat expérimental** du carnet : la découverte a coûté
`13` inférences pour apprendre le catalogue
`forme → décision` ; le rejeu sur un corpus *nouveau* de même forme a coûté **zéro**
inférence, car elle a consulté le catalogue, pas le cerveau. L'amortissement est
réel et mesurable : la connaissance est un actif qui se paie une fois. C'est ce que
« composition » veut dire concrètement — le coût d'inférence est *déplacé* de
l'exécution vers la découverte, et l'exécution devient gratuite en inférences.

## 7. La vérification en action — rejeter un plan incomplet

Le garde-fou n'est pas théorique : on le **déclenche**. On construit un plan où un
segment est découpé mais dont *un enfant* est laissé sans traitement final, et on
le passe à `verifier_couverture`. La boucle **doit** le rejeter en nommant
précisément ce qui manque — c'est la capacité de *vérification* d'un système
composable, qui empêche d'exécuter un plan à moitié formé.

In [9]:
{
    /// Representation commune d'un morceau de donnee (reutilisee de Compression-07b).
    ///
    /// L'objet que la boucle percoit et represente : `id` stable, `data` = octets,
    /// `genre` = qualification, `parent`/`profondeur` = lien vers l'origine.
    #[derive(Clone)]
    struct Segment {
        id: String,
        data: Vec<u8>,
        genre: String,
        parent: Option<String>,
        profondeur: i64,
    }

    /// Signature de structure deduite du CONTENU — la perception du segment.
    ///
    /// C'est ce que le cerveau regarde pour decider. Deduite, pas etiquettee :
    /// un fichier nomme `.json` qui ne parse pas n'est pas du JSON.
    struct Info {
        octets: usize,
        texte: bool,
        csv: bool,
        blocs_md: usize,
        json: bool,
        json_obj: bool,
    }

    /// Les booleens Rust s'impriment `true`/`false` ; le jumeau Python impose
    /// `True`/`False`. Toute comparaison de texte passe par cette fonction.
    fn py(b: bool) -> &'static str {
        if b { "True" } else { "False" }
    }

    /// Repr Python d'une liste de chaines : `['a', 'b']`.
    fn py_list_str(items: &[&str]) -> String {
        let mut s = String::from("[");
        for (i, it) in items.iter().enumerate() {
            if i > 0 { s.push_str(", "); }
            s.push('\'');
            s.push_str(it);
            s.push('\'');
        }
        s.push(']');
        s
    }

    /// JSON : mini-dumpeur et mini-parseur maison (`std` seule, aucune crate).
    ///
    /// L'ordre d'insertion des cles d'un objet est PRESERVE (`Vec<(String, Jv)>`) :
    /// le catalogue amorti s'appuie sur cet ordre, un `BTreeMap` le casserait.
    #[derive(Clone)]
    enum Jv {
        Null,
        Bool(bool),
        Int(i64),
        Str(String),
        List(Vec<Jv>),
        Obj(Vec<(String, Jv)>),
    }

    fn json_echapper(s: &str, out: &mut String) {
        out.push('"');
        for c in s.chars() {
            match c {
                '"' => { out.push('\\'); out.push('"'); }
                '\\' => { out.push('\\'); out.push('\\'); }
                '\n' => out.push_str("\\n"),
                '\r' => out.push_str("\\r"),
                '\t' => out.push_str("\\t"),
                c if (c as u32) < 0x20 => out.push_str(&format!("\\u{:04x}", c as u32)),
                c => out.push(c),
            }
        }
        out.push('"');
    }

    /// Dumps equivalent a `json.dumps(indent=2, ensure_ascii=False)`.
    fn json_dump(v: &Jv, level: usize, out: &mut String) {
        let pad = "  ".repeat(level);
        let pad_in = "  ".repeat(level + 1);
        match v {
            Jv::Null => out.push_str("null"),
            Jv::Bool(b) => out.push_str(if *b { "true" } else { "false" }),
            Jv::Int(i) => out.push_str(&i.to_string()),
            Jv::Str(s) => json_echapper(s, out),
            Jv::List(items) => {
                if items.is_empty() { out.push_str("[]"); return; }
                out.push_str("[\n");
                for (i, it) in items.iter().enumerate() {
                    out.push_str(&pad_in);
                    json_dump(it, level + 1, out);
                    if i + 1 < items.len() { out.push(','); }
                    out.push('\n');
                }
                out.push_str(&pad);
                out.push(']');
            }
            Jv::Obj(fields) => {
                if fields.is_empty() { out.push_str("{}"); return; }
                out.push_str("{\n");
                for (i, (k, val)) in fields.iter().enumerate() {
                    out.push_str(&pad_in);
                    json_echapper(k, out);
                    out.push_str(": ");
                    json_dump(val, level + 1, out);
                    if i + 1 < fields.len() { out.push(','); }
                    out.push('\n');
                }
                out.push_str(&pad);
                out.push('}');
            }
        }
    }

    fn json_skip_ws(b: &[u8], i: &mut usize) {
        while *i < b.len() && (b[*i] as char).is_whitespace() { *i += 1; }
    }

    fn json_chaine(b: &[u8], i: &mut usize) -> Option<String> {
        if *i >= b.len() || b[*i] != b'"' { return None; }
        *i += 1;
        let mut s = String::new();
        while *i < b.len() {
            let c = b[*i];
            if c == b'"' { *i += 1; return Some(s); }
            if c == b'\\' {
                *i += 1;
                if *i >= b.len() { return None; }
                let e = b[*i];
                *i += 1;
                match e {
                    b'n' => s.push('\n'),
                    b't' => s.push('\t'),
                    b'r' => s.push('\r'),
                    b'b' => s.push('\u{8}'),
                    b'f' => s.push('\u{c}'),
                    b'/' => s.push('/'),
                    b'"' => s.push('"'),
                    b'\\' => s.push('\\'),
                    b'u' => {
                        if *i + 4 > b.len() { return None; }
                        let hex = std::str::from_utf8(&b[*i..*i + 4]).ok()?;
                        let code = u32::from_str_radix(hex, 16).ok()?;
                        *i += 4;
                        s.push(char::from_u32(code).unwrap_or('\u{fffd}'));
                    }
                    _ => return None,
                }
            } else {
                let start = *i;
                while *i < b.len() && b[*i] != b'"' && b[*i] != b'\\' { *i += 1; }
                s.push_str(std::str::from_utf8(&b[start..*i]).ok()?);
            }
        }
        None
    }

    fn json_valeur(b: &[u8], i: &mut usize) -> Option<Jv> {
        json_skip_ws(b, i);
        if *i >= b.len() { return None; }
        match b[*i] {
            b'{' => {
                *i += 1;
                let mut fields: Vec<(String, Jv)> = Vec::new();
                json_skip_ws(b, i);
                if *i < b.len() && b[*i] == b'}' { *i += 1; return Some(Jv::Obj(fields)); }
                loop {
                    json_skip_ws(b, i);
                    if *i >= b.len() || b[*i] != b'"' { return None; }
                    let k = json_chaine(b, i)?;
                    json_skip_ws(b, i);
                    if *i >= b.len() || b[*i] != b':' { return None; }
                    *i += 1;
                    let v = json_valeur(b, i)?;
                    fields.push((k, v));
                    json_skip_ws(b, i);
                    if *i >= b.len() { return None; }
                    match b[*i] {
                        b',' => { *i += 1; }
                        b'}' => { *i += 1; return Some(Jv::Obj(fields)); }
                        _ => return None,
                    }
                }
            }
            b'[' => {
                *i += 1;
                let mut items: Vec<Jv> = Vec::new();
                json_skip_ws(b, i);
                if *i < b.len() && b[*i] == b']' { *i += 1; return Some(Jv::List(items)); }
                loop {
                    let v = json_valeur(b, i)?;
                    items.push(v);
                    json_skip_ws(b, i);
                    if *i >= b.len() { return None; }
                    match b[*i] {
                        b',' => { *i += 1; }
                        b']' => { *i += 1; return Some(Jv::List(items)); }
                        _ => return None,
                    }
                }
            }
            b'"' => Some(Jv::Str(json_chaine(b, i)?)),
            b't' => {
                if b.len() >= *i + 4 && &b[*i..*i + 4] == b"true" { *i += 4; return Some(Jv::Bool(true)); }
                None
            }
            b'f' => {
                if b.len() >= *i + 5 && &b[*i..*i + 5] == b"false" { *i += 5; return Some(Jv::Bool(false)); }
                None
            }
            b'n' => {
                if b.len() >= *i + 4 && &b[*i..*i + 4] == b"null" { *i += 4; return Some(Jv::Null); }
                None
            }
            _ => {
                let start = *i;
                if b[*i] == b'-' { *i += 1; }
                while *i < b.len() && b[*i].is_ascii_digit() { *i += 1; }
                if start == *i { return None; }
                let txt = std::str::from_utf8(&b[start..*i]).ok()?;
                txt.parse::<i64>().ok().map(Jv::Int)
            }
        }
    }

    /// Parse un document JSON complet ; `None` si le texte n'est pas du JSON.
    fn json_parse(texte: &str) -> Option<Jv> {
        let b = texte.as_bytes();
        let mut i = 0usize;
        let v = json_valeur(b, &mut i)?;
        json_skip_ws(b, &mut i);
        if i != b.len() { return None; }
        Some(v)
    }

    /// Positions (en octets) des titres de markdown : meme regex que le jumeau
    /// Python (`^##? ` en mode multiligne), donc memes decoupes.
    fn positions_titres(texte: &str) -> Vec<usize> {
        let mut out = Vec::new();
        let mut off = 0usize;
        for l in texte.split_inclusive('\n') {
            let t = l.trim_end_matches('\n').trim_end_matches('\r');
            if t.starts_with("# ") || t.starts_with("## ") { out.push(off); }
            off += l.len();
        }
        out
    }

    /// Perception : la signature de structure deduite du contenu.
    fn structure_interne(seg: &Segment) -> Info {
        let mut info = Info {
            octets: seg.data.len(),
            texte: false,
            csv: false,
            blocs_md: 0,
            json: false,
            json_obj: false,
        };
        let texte = match String::from_utf8(seg.data.clone()) {
            Ok(t) => t,
            Err(_) => return info,
        };
        info.texte = true;
        let t = texte.trim();
        if t.starts_with('{') || t.starts_with('[') {
            if let Some(v) = json_parse(&texte) {
                info.json_obj = matches!(v, Jv::Obj(_));
                info.json = true;
                return info;
            }
        }
        let lignes: Vec<&str> = texte.lines().filter(|l| !l.is_empty()).collect();
        if !lignes.is_empty() && lignes[0].contains(',') {
            let n = lignes[0].matches(',').count();
            info.csv = lignes.iter().all(|l| l.matches(',').count() == n);
        }
        info.blocs_md = positions_titres(&texte).len();
        info
    }

    /// Classe de forme d'un segment — la cle du catalogue amorti.
    ///
    /// Distingue l'objet JSON (decoupable par ses cles) du tableau JSON
    /// (compresse tel quel) : deux formes, donc deux decisions distinctes.
    fn signature(s: &Info) -> String {
        if s.json {
            return if s.json_obj { "json_obj".to_string() } else { "json".to_string() };
        }
        if s.csv { return "csv".to_string(); }
        if s.blocs_md > 0 { return "markdown".to_string(); }
        if s.texte { "texte".to_string() } else { "binaire".to_string() }
    }

    /// Un petit corpus heterogene, genere deterministiquement : meme entree a
    /// chaque execution, donc resultat reproductible.
    fn corpus_demo() -> Vec<Segment> {
        let ventes = "nom,quantite,prix\ncafe,10,2.50\nthe,20,1.80\nsucre,5,0.90\nlait,8,1.10\npain,12,0.80\n";
        let clients_obj = Jv::Obj(vec![
            ("id".to_string(), Jv::Int(7)),
            ("panier".to_string(), Jv::List(vec![
                Jv::Obj(vec![
                    ("sku".to_string(), Jv::Str("cafe".to_string())),
                    ("q".to_string(), Jv::Int(10)),
                ]),
                Jv::Obj(vec![
                    ("sku".to_string(), Jv::Str("the".to_string())),
                    ("q".to_string(), Jv::Int(20)),
                ]),
            ])),
        ]);
        let mut clients = String::new();
        json_dump(&clients_obj, 0, &mut clients);
        clients.push('\n');
        let notes = "# Rapport\n\n## Chiffre\n12 kEUR, en hausse.\n\n## Risque\nFournisseur unique, lead time 6 semaines.\n";
        let journal: String = (0..40)
            .map(|i| format!("10:{:02}:00 INFO requete {} ok\n", i, i))
            .collect();
        let mut binaire: Vec<u8> = Vec::with_capacity(1024);
        for _ in 0..4 {
            for b in 0u16..256 { binaire.push(b as u8); }
        }
        vec![
            Segment {
                id: "ventes.csv#0".to_string(),
                data: ventes.as_bytes().to_vec(),
                genre: "donnees/csv".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "clients.json#0".to_string(),
                data: clients.into_bytes(),
                genre: "donnees/json".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "notes.md#0".to_string(),
                data: notes.as_bytes().to_vec(),
                genre: "texte/markdown".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "journal.log#0".to_string(),
                data: journal.into_bytes(),
                genre: "texte/log".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "brut.bin#0".to_string(),
                data: binaire,
                genre: "binaire".to_string(),
                parent: None,
                profondeur: 0,
            },
        ]
    }

    /// La decision d'un cerveau : decouper (reduire) ou compresser.
    ///
    /// Un `enum` remplace le `dict` du jumeau Python : les deux cas sont
    /// nommes, donc le compilateur refuse le cas forget.
    #[derive(Clone)]
    enum Decision {
        Decouper { mode: String },
        Compresser { codec: String },
    }

    impl Decision {
        fn op(&self) -> &'static str {
            match self {
                Decision::Decouper { .. } => "decouper",
                Decision::Compresser { .. } => "compresser",
            }
        }
        fn quoi(&self) -> &str {
            match self {
                Decision::Decouper { mode } => mode,
                Decision::Compresser { codec } => codec,
            }
        }
    }

    /// La recette : le traitement final retenu pour un segment.
    ///
    /// Un segment est couvert s'il est compresse, ou decoupe en enfants tous
    /// couverts — c'est l'invariant que `verifier_couverture` fait respecter.
    enum Recette {
        Decouper { mode: String, enfants: Vec<String> },
        Compresser { codec: String },
    }

    /// Point de composition : le contrat d'un mecanisme d'inference.
    ///
    /// La boucle agentique n'appelle JAMAIS un cerveau en particulier, seulement
    /// ce trait. C'est ce contrat unique qui rend le cerveau interchangeable
    /// (deterministe, LLM, hybride). Chaque consultation compte comme une inference.
    trait Cerveau {
        fn inferer(&mut self, seg: &Segment) -> Result<Decision, String>;
    }

    /// Le squelette agentique : percevoir -> representer -> inferer -> decider
    /// -> verifier -> executer -> observer.
    ///
    /// Compose la representation (Segment), un cerveau (point de composition),
    /// l'invariant de couverture (verification) et l'application de la recette
    /// (execution). L'inference est isolee derriere le trait `Cerveau`.
    struct AgentTriage<'a> {
        cerveau: &'a mut dyn Cerveau,
        borne: i64,
        catalogue: Vec<(String, Decision)>,
        recette: Vec<(String, Recette)>,
        appel_decouverte: i64,
    }

    impl<'a> AgentTriage<'a> {
        /// Apprend le catalogue forme->decision en parcourant le corpus.
        ///
        /// Chaque consultation du cerveau compte. Si le parcours atteint la borne
        /// sans epuiser la file, la decouverte ECHOUE (safety bound) : on ne
        /// laisse jamais l'agent tourner a l'infini.
        fn decouvrir(&mut self, segments: &[Segment]) -> Result<(), String> {
            self.catalogue = Vec::new();
            self.recette = Vec::new();
            self.appel_decouverte = 0;
            let mut file: Vec<Segment> = segments.to_vec();
            while !file.is_empty() {
                // La borne est verifiee AVANT le retrait : c'est elle qui
                // distingue une decouverte bornee d'une decouverte divergente.
                if self.appel_decouverte >= self.borne {
                    return Err(format!(
                        "decouverte interrompue a {} inférences, {} segment(s) en attente — safety bound.",
                        self.appel_decouverte,
                        file.len()
                    ));
                }
                let seg = file.remove(0);
                let dec = self.cerveau.inferer(&seg)?; // inferer + decider (compte)
                self.appel_decouverte += 1;
                let forme = signature(&structure_interne(&seg));
                self.cat_set(&forme, dec.clone());
                if let Decision::Decouper { mode } = &dec {
                    let enfants = decouper(&seg, mode);
                    // La recette porte les ENFANTS reels, pas seulement la
                    // decision : c'est cette liste que l'invariant parcourt.
                    let ids: Vec<String> = enfants.iter().map(|e| e.id.clone()).collect();
                    self.recette.push((
                        seg.id.clone(),
                        Recette::Decouper { mode: mode.clone(), enfants: ids },
                    ));
                    file.extend(enfants);
                } else {
                    self.recette.push((seg.id.clone(), Recette::Compresser { codec: dec.quoi().to_string() }));
                }
            }
            let racines: Vec<&str> = segments.iter().map(|s| s.id.as_str()).collect();
            verifier_couverture(&racines, &self.recette)?; // verifier
            Ok(())
        }

        /// Le catalogue garde l'ordre de PREMIERE insertion : une forme deja
        /// connue est remplacee en place, pas appendue.
        fn cat_set(&mut self, forme: &str, dec: Decision) {
            for (k, v) in self.catalogue.iter_mut() {
                if k == forme {
                    *v = dec;
                    return;
                }
            }
            self.catalogue.push((forme.to_string(), dec));
        }

        /// Applique le catalogue appris a de nouveaux segments, SANS re-inferer.
        ///
        /// C'est l'amortissement : la decouverte a paye le catalogue, le rejeu le
        /// consomme a cout d'inference NUL.
        fn rejouer(&self, segments: &[Segment]) -> Vec<(String, Decision)> {
            segments
                .iter()
                .map(|seg| {
                    let forme = signature(&structure_interne(seg));
                    let dec = self
                        .catalogue
                        .iter()
                        .find(|(k, _)| *k == forme)
                        .map(|(_, v)| v.clone())
                        .unwrap_or(Decision::Compresser { codec: "zlib".to_string() });
                    (seg.id.clone(), dec)
                })
                .collect()
        }
    }

    /// Un segment est couvert s'il est compresse, ou decoupe en enfants tous
    /// couverts. Le cycle est detecte : la recette ne doit pas boucler.
    fn couverture(
        sid: &str,
        recette: &[(String, Recette)],
        vus: &mut Vec<String>,
    ) -> Result<Vec<String>, String> {
        if vus.iter().any(|v| v == sid) {
            return Err(format!("cycle dans la recette autour de {}", sid));
        }
        vus.push(sid.to_string());
        let entree = recette.iter().find(|(k, _)| k == sid).map(|(_, v)| v);
        let res = match entree {
            None => Ok(vec![sid.to_string()]),
            Some(Recette::Compresser { .. }) => Ok(Vec::new()),
            Some(Recette::Decouper { enfants, .. }) => {
                let mut manquants: Vec<String> = Vec::new();
                for enfant in enfants {
                    match couverture(enfant, recette, vus) {
                        Ok(abs) => manquants.extend(abs),
                        Err(msg) => {
                            vus.pop();
                            return Err(msg);
                        }
                    }
                }
                Ok(manquants)
            }
        };
        vus.pop();
        res
    }

    /// Invariant : leve en nommant tout segment original non traite.
    fn verifier_couverture(racines: &[&str], recette: &[(String, Recette)]) -> Result<(), String> {
        let mut manquants: Vec<String> = Vec::new();
        for sid in racines {
            match couverture(sid, recette, &mut Vec::new()) {
                Ok(abs) => manquants.extend(abs),
                Err(msg) => return Err(msg),
            }
        }
        if !manquants.is_empty() {
            let liste: Vec<&str> = manquants.iter().take(6).map(|s| s.as_str()).collect();
            return Err(format!(
                "{} segment(s) sans traitement final : {}",
                manquants.len(),
                py_list_str(&liste)
            ));
        }
        Ok(())
    }

    /// Operation inversible : un segment macro -> des fragments qualifies.
    fn decouper(seg: &Segment, mode: &str) -> Vec<Segment> {
        let texte = match String::from_utf8(seg.data.clone()) {
            Ok(t) => t,
            Err(_) => return Vec::new(),
        };
        let mut kids: Vec<Segment> = Vec::new();
        match mode {
            "blocs_markdown" => {
                let mut bornes: Vec<usize> = vec![0];
                bornes.extend(positions_titres(&texte));
                bornes.push(texte.len());
                for i in 0..bornes.len() - 1 {
                    let m = &texte[bornes[i]..bornes[i + 1]];
                    if !m.trim().is_empty() {
                        kids.push(Segment {
                            id: format!("{}.b{}", seg.id, kids.len()),
                            data: m.as_bytes().to_vec(),
                            genre: format!("{}/bloc", seg.genre),
                            parent: Some(seg.id.clone()),
                            profondeur: seg.profondeur + 1,
                        });
                    }
                }
            }
            "colonnes_csv" => {
                let lignes: Vec<&str> = texte.lines().filter(|l| !l.is_empty()).collect();
                if lignes.is_empty() { return kids; }
                let cols: Vec<&str> = lignes[0].split(',').collect();
                let mat: Vec<Vec<&str>> = lignes.iter().map(|l| l.split(',').collect()).collect();
                for j in 0..cols.len() {
                    let mut s = String::new();
                    for (r, row) in mat.iter().enumerate() {
                        if r > 0 { s.push('\n'); }
                        s.push_str(row[j]);
                    }
                    kids.push(Segment {
                        id: format!("{}.c{}", seg.id, j),
                        data: s.into_bytes(),
                        genre: format!("{}/col:{}", seg.genre, cols[j]),
                        parent: Some(seg.id.clone()),
                        profondeur: seg.profondeur + 1,
                    });
                }
            }
            "cles_json" => {
                let champs = match json_parse(&texte) {
                    Some(Jv::Obj(f)) => f,
                    _ => return kids,
                };
                for (k, v) in &champs {
                    let mut s = String::new();
                    json_dump(v, 0, &mut s);
                    kids.push(Segment {
                        id: format!("{}.k{}", seg.id, kids.len()),
                        data: s.into_bytes(),
                        genre: format!("{}/cle:{}", seg.genre, k),
                        parent: Some(seg.id.clone()),
                        profondeur: seg.profondeur + 1,
                    });
                }
            }
            _ => {}
        }
        kids
    }

    /// L'inverse d'un decoupage : reconstruit les octets du segment parent.
    fn recoller(mode: &str, enfants: &[Segment]) -> Vec<u8> {
        match mode {
            "blocs_markdown" => {
                let mut out = Vec::new();
                for e in enfants { out.extend_from_slice(&e.data); }
                out
            }
            "colonnes_csv" => {
                let mat: Vec<Vec<String>> = enfants
                    .iter()
                    .map(|e| String::from_utf8_lossy(&e.data).split('\n').map(|s| s.to_string()).collect())
                    .collect();
                if mat.is_empty() { return Vec::new(); }
                let n = mat[0].len();
                let mut lignes: Vec<String> = Vec::new();
                for row in 0..n {
                    let parts: Vec<&str> = mat.iter().map(|c| c[row].as_str()).collect();
                    lignes.push(parts.join(","));
                }
                let mut s = lignes.join("\n");
                s.push('\n');
                s.into_bytes()
            }
            "cles_json" => {
                let mut champs: Vec<(String, Jv)> = Vec::new();
                for e in enfants {
                    let k = e.genre.rsplit(':').next().unwrap_or("").to_string();
                    let v = json_parse(&String::from_utf8_lossy(&e.data)).unwrap_or(Jv::Null);
                    champs.push((k, v));
                }
                let mut s = String::new();
                json_dump(&Jv::Obj(champs), 0, &mut s);
                s.push('\n');
                s.into_bytes()
            }
            _ => Vec::new(),
        }
    }

    /// Le cerveau par defaut : derive la decision de la structure percue.
    struct CerveauDeterministe { appels: i64 }

    impl Cerveau for CerveauDeterministe {
        fn inferer(&mut self, seg: &Segment) -> Result<Decision, String> {
            self.appels += 1;
            let s = structure_interne(seg);
            if seg.data.len() < 64 {
                return Ok(Decision::Compresser {
                    codec: if s.texte { "zlib".to_string() } else { "lzma".to_string() },
                });
            }
            let mode = match signature(&s).as_str() {
                "json_obj" => "cles_json",
                "csv" => "colonnes_csv",
                "markdown" => "blocs_markdown",
                _ => {
                    return Ok(Decision::Compresser {
                        codec: if s.texte { "zlib".to_string() } else { "lzma".to_string() },
                    })
                }
            };
            Ok(Decision::Decouper { mode: mode.to_string() })
        }
    }

    let segments = corpus_demo();
    let mut cerveau = CerveauDeterministe { appels: 0 };
    let mut agent = AgentTriage {
        cerveau: &mut cerveau,
        borne: 64,
        catalogue: Vec::new(),
        recette: Vec::new(),
        appel_decouverte: 0,
    };
    agent.decouvrir(&segments)?;

    // PLAN INCOMPLET : x.md est decoupe en deux blocs, mais b1 n'a AUCUN
    // traitement final. L'invariant de couverture doit rejeter le plan en
    // nommant b1.
    let plan_casse: Vec<(String, Recette)> = vec![
        (
            "x.md#0".to_string(),
            Recette::Decouper {
                mode: "blocs_markdown".to_string(),
                enfants: vec!["x.md#0.b0".to_string(), "x.md#0.b1".to_string()],
            },
        ),
        (
            "x.md#0.b0".to_string(),
            Recette::Compresser { codec: "zlib".to_string() },
        ),
        // x.md#0.b1 : absent de la recette -> non couvert
    ];
    match verifier_couverture(&["x.md#0"], &plan_casse) {
        Ok(()) => println!("BUG : le plan incomplet n'a PAS ete rejete"),
        Err(e) => println!("VÉRIFICATION : le plan incomplet est REJETE -> {}", e),
    }

    // CONTROLE POSITIF : le plan decouvert, lui, est complet -> l'invariant
    // le laisse passer.
    let racines: Vec<&str> = segments.iter().map(|s| s.id.as_str()).collect();
    verifier_couverture(&racines, &agent.recette)?;
    println!("VÉRIFICATION : le plan decouvert est complet (aucun segment orphelin) -> laissé passer");
};

VÉRIFICATION : le plan incomplet est REJETE -> 1 segment(s) sans traitement final : ['x.md#0.b1']


VÉRIFICATION : le plan decouvert est complet (aucun segment orphelin) -> laissé passer


### Lecture du résultat

Le même invariant **rejette** le plan à moitié formé (il nomme `x.md#0.b1`, l'enfant
orphelin) et **laisse passer** le plan découvert (tous les segments y ont un
traitement final). C'est la différence entre *avoir un plan* et *avoir un plan
exécutable* — et c'est ce qui autorise la composition : on ne branche un
mécanisme d'inférence que derrière une porte de vérification qui garantit que le
plan produit est complet.

## 8. La safety bound — arrêter une découverte divergente

La deuxième capacité de sécurité d'un système composable : **ne jamais tourner à
l'infini**. On branche un cerveau *pathologique* qui découpe toujours, sans
produire de traitement final — le parcours ne s'épuise jamais. Le squelette doit
l'**arrêter** à la borne d'appels, au lieu de laisser l'agent s'épuiser.

In [10]:
{
    /// Representation commune d'un morceau de donnee (reutilisee de Compression-07b).
    ///
    /// L'objet que la boucle percoit et represente : `id` stable, `data` = octets,
    /// `genre` = qualification, `parent`/`profondeur` = lien vers l'origine.
    #[derive(Clone)]
    struct Segment {
        id: String,
        data: Vec<u8>,
        genre: String,
        parent: Option<String>,
        profondeur: i64,
    }

    /// Signature de structure deduite du CONTENU — la perception du segment.
    ///
    /// C'est ce que le cerveau regarde pour decider. Deduite, pas etiquettee :
    /// un fichier nomme `.json` qui ne parse pas n'est pas du JSON.
    struct Info {
        octets: usize,
        texte: bool,
        csv: bool,
        blocs_md: usize,
        json: bool,
        json_obj: bool,
    }

    /// Les booleens Rust s'impriment `true`/`false` ; le jumeau Python impose
    /// `True`/`False`. Toute comparaison de texte passe par cette fonction.
    fn py(b: bool) -> &'static str {
        if b { "True" } else { "False" }
    }

    /// Repr Python d'une liste de chaines : `['a', 'b']`.
    fn py_list_str(items: &[&str]) -> String {
        let mut s = String::from("[");
        for (i, it) in items.iter().enumerate() {
            if i > 0 { s.push_str(", "); }
            s.push('\'');
            s.push_str(it);
            s.push('\'');
        }
        s.push(']');
        s
    }

    /// JSON : mini-dumpeur et mini-parseur maison (`std` seule, aucune crate).
    ///
    /// L'ordre d'insertion des cles d'un objet est PRESERVE (`Vec<(String, Jv)>`) :
    /// le catalogue amorti s'appuie sur cet ordre, un `BTreeMap` le casserait.
    #[derive(Clone)]
    enum Jv {
        Null,
        Bool(bool),
        Int(i64),
        Str(String),
        List(Vec<Jv>),
        Obj(Vec<(String, Jv)>),
    }

    fn json_echapper(s: &str, out: &mut String) {
        out.push('"');
        for c in s.chars() {
            match c {
                '"' => { out.push('\\'); out.push('"'); }
                '\\' => { out.push('\\'); out.push('\\'); }
                '\n' => out.push_str("\\n"),
                '\r' => out.push_str("\\r"),
                '\t' => out.push_str("\\t"),
                c if (c as u32) < 0x20 => out.push_str(&format!("\\u{:04x}", c as u32)),
                c => out.push(c),
            }
        }
        out.push('"');
    }

    /// Dumps equivalent a `json.dumps(indent=2, ensure_ascii=False)`.
    fn json_dump(v: &Jv, level: usize, out: &mut String) {
        let pad = "  ".repeat(level);
        let pad_in = "  ".repeat(level + 1);
        match v {
            Jv::Null => out.push_str("null"),
            Jv::Bool(b) => out.push_str(if *b { "true" } else { "false" }),
            Jv::Int(i) => out.push_str(&i.to_string()),
            Jv::Str(s) => json_echapper(s, out),
            Jv::List(items) => {
                if items.is_empty() { out.push_str("[]"); return; }
                out.push_str("[\n");
                for (i, it) in items.iter().enumerate() {
                    out.push_str(&pad_in);
                    json_dump(it, level + 1, out);
                    if i + 1 < items.len() { out.push(','); }
                    out.push('\n');
                }
                out.push_str(&pad);
                out.push(']');
            }
            Jv::Obj(fields) => {
                if fields.is_empty() { out.push_str("{}"); return; }
                out.push_str("{\n");
                for (i, (k, val)) in fields.iter().enumerate() {
                    out.push_str(&pad_in);
                    json_echapper(k, out);
                    out.push_str(": ");
                    json_dump(val, level + 1, out);
                    if i + 1 < fields.len() { out.push(','); }
                    out.push('\n');
                }
                out.push_str(&pad);
                out.push('}');
            }
        }
    }

    fn json_skip_ws(b: &[u8], i: &mut usize) {
        while *i < b.len() && (b[*i] as char).is_whitespace() { *i += 1; }
    }

    fn json_chaine(b: &[u8], i: &mut usize) -> Option<String> {
        if *i >= b.len() || b[*i] != b'"' { return None; }
        *i += 1;
        let mut s = String::new();
        while *i < b.len() {
            let c = b[*i];
            if c == b'"' { *i += 1; return Some(s); }
            if c == b'\\' {
                *i += 1;
                if *i >= b.len() { return None; }
                let e = b[*i];
                *i += 1;
                match e {
                    b'n' => s.push('\n'),
                    b't' => s.push('\t'),
                    b'r' => s.push('\r'),
                    b'b' => s.push('\u{8}'),
                    b'f' => s.push('\u{c}'),
                    b'/' => s.push('/'),
                    b'"' => s.push('"'),
                    b'\\' => s.push('\\'),
                    b'u' => {
                        if *i + 4 > b.len() { return None; }
                        let hex = std::str::from_utf8(&b[*i..*i + 4]).ok()?;
                        let code = u32::from_str_radix(hex, 16).ok()?;
                        *i += 4;
                        s.push(char::from_u32(code).unwrap_or('\u{fffd}'));
                    }
                    _ => return None,
                }
            } else {
                let start = *i;
                while *i < b.len() && b[*i] != b'"' && b[*i] != b'\\' { *i += 1; }
                s.push_str(std::str::from_utf8(&b[start..*i]).ok()?);
            }
        }
        None
    }

    fn json_valeur(b: &[u8], i: &mut usize) -> Option<Jv> {
        json_skip_ws(b, i);
        if *i >= b.len() { return None; }
        match b[*i] {
            b'{' => {
                *i += 1;
                let mut fields: Vec<(String, Jv)> = Vec::new();
                json_skip_ws(b, i);
                if *i < b.len() && b[*i] == b'}' { *i += 1; return Some(Jv::Obj(fields)); }
                loop {
                    json_skip_ws(b, i);
                    if *i >= b.len() || b[*i] != b'"' { return None; }
                    let k = json_chaine(b, i)?;
                    json_skip_ws(b, i);
                    if *i >= b.len() || b[*i] != b':' { return None; }
                    *i += 1;
                    let v = json_valeur(b, i)?;
                    fields.push((k, v));
                    json_skip_ws(b, i);
                    if *i >= b.len() { return None; }
                    match b[*i] {
                        b',' => { *i += 1; }
                        b'}' => { *i += 1; return Some(Jv::Obj(fields)); }
                        _ => return None,
                    }
                }
            }
            b'[' => {
                *i += 1;
                let mut items: Vec<Jv> = Vec::new();
                json_skip_ws(b, i);
                if *i < b.len() && b[*i] == b']' { *i += 1; return Some(Jv::List(items)); }
                loop {
                    let v = json_valeur(b, i)?;
                    items.push(v);
                    json_skip_ws(b, i);
                    if *i >= b.len() { return None; }
                    match b[*i] {
                        b',' => { *i += 1; }
                        b']' => { *i += 1; return Some(Jv::List(items)); }
                        _ => return None,
                    }
                }
            }
            b'"' => Some(Jv::Str(json_chaine(b, i)?)),
            b't' => {
                if b.len() >= *i + 4 && &b[*i..*i + 4] == b"true" { *i += 4; return Some(Jv::Bool(true)); }
                None
            }
            b'f' => {
                if b.len() >= *i + 5 && &b[*i..*i + 5] == b"false" { *i += 5; return Some(Jv::Bool(false)); }
                None
            }
            b'n' => {
                if b.len() >= *i + 4 && &b[*i..*i + 4] == b"null" { *i += 4; return Some(Jv::Null); }
                None
            }
            _ => {
                let start = *i;
                if b[*i] == b'-' { *i += 1; }
                while *i < b.len() && b[*i].is_ascii_digit() { *i += 1; }
                if start == *i { return None; }
                let txt = std::str::from_utf8(&b[start..*i]).ok()?;
                txt.parse::<i64>().ok().map(Jv::Int)
            }
        }
    }

    /// Parse un document JSON complet ; `None` si le texte n'est pas du JSON.
    fn json_parse(texte: &str) -> Option<Jv> {
        let b = texte.as_bytes();
        let mut i = 0usize;
        let v = json_valeur(b, &mut i)?;
        json_skip_ws(b, &mut i);
        if i != b.len() { return None; }
        Some(v)
    }

    /// Positions (en octets) des titres de markdown : meme regex que le jumeau
    /// Python (`^##? ` en mode multiligne), donc memes decoupes.
    fn positions_titres(texte: &str) -> Vec<usize> {
        let mut out = Vec::new();
        let mut off = 0usize;
        for l in texte.split_inclusive('\n') {
            let t = l.trim_end_matches('\n').trim_end_matches('\r');
            if t.starts_with("# ") || t.starts_with("## ") { out.push(off); }
            off += l.len();
        }
        out
    }

    /// Perception : la signature de structure deduite du contenu.
    fn structure_interne(seg: &Segment) -> Info {
        let mut info = Info {
            octets: seg.data.len(),
            texte: false,
            csv: false,
            blocs_md: 0,
            json: false,
            json_obj: false,
        };
        let texte = match String::from_utf8(seg.data.clone()) {
            Ok(t) => t,
            Err(_) => return info,
        };
        info.texte = true;
        let t = texte.trim();
        if t.starts_with('{') || t.starts_with('[') {
            if let Some(v) = json_parse(&texte) {
                info.json_obj = matches!(v, Jv::Obj(_));
                info.json = true;
                return info;
            }
        }
        let lignes: Vec<&str> = texte.lines().filter(|l| !l.is_empty()).collect();
        if !lignes.is_empty() && lignes[0].contains(',') {
            let n = lignes[0].matches(',').count();
            info.csv = lignes.iter().all(|l| l.matches(',').count() == n);
        }
        info.blocs_md = positions_titres(&texte).len();
        info
    }

    /// Classe de forme d'un segment — la cle du catalogue amorti.
    ///
    /// Distingue l'objet JSON (decoupable par ses cles) du tableau JSON
    /// (compresse tel quel) : deux formes, donc deux decisions distinctes.
    fn signature(s: &Info) -> String {
        if s.json {
            return if s.json_obj { "json_obj".to_string() } else { "json".to_string() };
        }
        if s.csv { return "csv".to_string(); }
        if s.blocs_md > 0 { return "markdown".to_string(); }
        if s.texte { "texte".to_string() } else { "binaire".to_string() }
    }

    /// Un petit corpus heterogene, genere deterministiquement : meme entree a
    /// chaque execution, donc resultat reproductible.
    fn corpus_demo() -> Vec<Segment> {
        let ventes = "nom,quantite,prix\ncafe,10,2.50\nthe,20,1.80\nsucre,5,0.90\nlait,8,1.10\npain,12,0.80\n";
        let clients_obj = Jv::Obj(vec![
            ("id".to_string(), Jv::Int(7)),
            ("panier".to_string(), Jv::List(vec![
                Jv::Obj(vec![
                    ("sku".to_string(), Jv::Str("cafe".to_string())),
                    ("q".to_string(), Jv::Int(10)),
                ]),
                Jv::Obj(vec![
                    ("sku".to_string(), Jv::Str("the".to_string())),
                    ("q".to_string(), Jv::Int(20)),
                ]),
            ])),
        ]);
        let mut clients = String::new();
        json_dump(&clients_obj, 0, &mut clients);
        clients.push('\n');
        let notes = "# Rapport\n\n## Chiffre\n12 kEUR, en hausse.\n\n## Risque\nFournisseur unique, lead time 6 semaines.\n";
        let journal: String = (0..40)
            .map(|i| format!("10:{:02}:00 INFO requete {} ok\n", i, i))
            .collect();
        let mut binaire: Vec<u8> = Vec::with_capacity(1024);
        for _ in 0..4 {
            for b in 0u16..256 { binaire.push(b as u8); }
        }
        vec![
            Segment {
                id: "ventes.csv#0".to_string(),
                data: ventes.as_bytes().to_vec(),
                genre: "donnees/csv".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "clients.json#0".to_string(),
                data: clients.into_bytes(),
                genre: "donnees/json".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "notes.md#0".to_string(),
                data: notes.as_bytes().to_vec(),
                genre: "texte/markdown".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "journal.log#0".to_string(),
                data: journal.into_bytes(),
                genre: "texte/log".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "brut.bin#0".to_string(),
                data: binaire,
                genre: "binaire".to_string(),
                parent: None,
                profondeur: 0,
            },
        ]
    }

    /// La decision d'un cerveau : decouper (reduire) ou compresser.
    ///
    /// Un `enum` remplace le `dict` du jumeau Python : les deux cas sont
    /// nommes, donc le compilateur refuse le cas forget.
    #[derive(Clone)]
    enum Decision {
        Decouper { mode: String },
        Compresser { codec: String },
    }

    impl Decision {
        fn op(&self) -> &'static str {
            match self {
                Decision::Decouper { .. } => "decouper",
                Decision::Compresser { .. } => "compresser",
            }
        }
        fn quoi(&self) -> &str {
            match self {
                Decision::Decouper { mode } => mode,
                Decision::Compresser { codec } => codec,
            }
        }
    }

    /// Point de composition : le contrat d'un mecanisme d'inference.
    ///
    /// La boucle agentique n'appelle JAMAIS un cerveau en particulier, seulement
    /// ce trait. C'est ce contrat unique qui rend le cerveau interchangeable
    /// (deterministe, LLM, hybride). Chaque consultation compte comme une inference.
    trait Cerveau {
        fn inferer(&mut self, seg: &Segment) -> Result<Decision, String>;
    }

    /// La recette : le traitement final retenu pour un segment.
    ///
    /// Un segment est couvert s'il est compresse, ou decoupe en enfants tous
    /// couverts — c'est l'invariant que `verifier_couverture` fait respecter.
    enum Recette {
        Decouper { mode: String, enfants: Vec<String> },
        Compresser { codec: String },
    }

    /// Le squelette agentique : percevoir -> representer -> inferer -> decider
    /// -> verifier -> executer -> observer.
    ///
    /// Compose la representation (Segment), un cerveau (point de composition),
    /// l'invariant de couverture (verification) et l'application de la recette
    /// (execution). L'inference est isolee derriere le trait `Cerveau`.
    struct AgentTriage<'a> {
        cerveau: &'a mut dyn Cerveau,
        borne: i64,
        catalogue: Vec<(String, Decision)>,
        recette: Vec<(String, Recette)>,
        appel_decouverte: i64,
    }

    impl<'a> AgentTriage<'a> {
        /// Apprend le catalogue forme->decision en parcourant le corpus.
        ///
        /// Chaque consultation du cerveau compte. Si le parcours atteint la borne
        /// sans epuiser la file, la decouverte ECHOUE (safety bound) : on ne
        /// laisse jamais l'agent tourner a l'infini.
        fn decouvrir(&mut self, segments: &[Segment]) -> Result<(), String> {
            self.catalogue = Vec::new();
            self.recette = Vec::new();
            self.appel_decouverte = 0;
            let mut file: Vec<Segment> = segments.to_vec();
            while !file.is_empty() {
                // La borne est verifiee AVANT le retrait : c'est elle qui
                // distingue une decouverte bornee d'une decouverte divergente.
                if self.appel_decouverte >= self.borne {
                    return Err(format!(
                        "decouverte interrompue a {} inférences, {} segment(s) en attente — safety bound.",
                        self.appel_decouverte,
                        file.len()
                    ));
                }
                let seg = file.remove(0);
                let dec = self.cerveau.inferer(&seg)?; // inferer + decider (compte)
                self.appel_decouverte += 1;
                let forme = signature(&structure_interne(&seg));
                self.cat_set(&forme, dec.clone());
                if let Decision::Decouper { mode } = &dec {
                    let enfants = decouper(&seg, mode);
                    // La recette porte les ENFANTS reels, pas seulement la
                    // decision : c'est cette liste que l'invariant parcourt.
                    let ids: Vec<String> = enfants.iter().map(|e| e.id.clone()).collect();
                    self.recette.push((
                        seg.id.clone(),
                        Recette::Decouper { mode: mode.clone(), enfants: ids },
                    ));
                    file.extend(enfants);
                } else {
                    self.recette.push((seg.id.clone(), Recette::Compresser { codec: dec.quoi().to_string() }));
                }
            }
            let racines: Vec<&str> = segments.iter().map(|s| s.id.as_str()).collect();
            verifier_couverture(&racines, &self.recette)?; // verifier
            Ok(())
        }

        /// Le catalogue garde l'ordre de PREMIERE insertion : une forme deja
        /// connue est remplacee en place, pas appendue.
        fn cat_set(&mut self, forme: &str, dec: Decision) {
            for (k, v) in self.catalogue.iter_mut() {
                if k == forme {
                    *v = dec;
                    return;
                }
            }
            self.catalogue.push((forme.to_string(), dec));
        }

        /// Applique le catalogue appris a de nouveaux segments, SANS re-inferer.
        ///
        /// C'est l'amortissement : la decouverte a paye le catalogue, le rejeu le
        /// consomme a cout d'inference NUL.
        fn rejouer(&self, segments: &[Segment]) -> Vec<(String, Decision)> {
            segments
                .iter()
                .map(|seg| {
                    let forme = signature(&structure_interne(seg));
                    let dec = self
                        .catalogue
                        .iter()
                        .find(|(k, _)| *k == forme)
                        .map(|(_, v)| v.clone())
                        .unwrap_or(Decision::Compresser { codec: "zlib".to_string() });
                    (seg.id.clone(), dec)
                })
                .collect()
        }
    }

    /// Un segment est couvert s'il est compresse, ou decoupe en enfants tous
    /// couverts. Le cycle est detecte : la recette ne doit pas boucler.
    fn couverture(
        sid: &str,
        recette: &[(String, Recette)],
        vus: &mut Vec<String>,
    ) -> Result<Vec<String>, String> {
        if vus.iter().any(|v| v == sid) {
            return Err(format!("cycle dans la recette autour de {}", sid));
        }
        vus.push(sid.to_string());
        let entree = recette.iter().find(|(k, _)| k == sid).map(|(_, v)| v);
        let res = match entree {
            None => Ok(vec![sid.to_string()]),
            Some(Recette::Compresser { .. }) => Ok(Vec::new()),
            Some(Recette::Decouper { enfants, .. }) => {
                let mut manquants: Vec<String> = Vec::new();
                for enfant in enfants {
                    match couverture(enfant, recette, vus) {
                        Ok(abs) => manquants.extend(abs),
                        Err(msg) => {
                            vus.pop();
                            return Err(msg);
                        }
                    }
                }
                Ok(manquants)
            }
        };
        vus.pop();
        res
    }

    /// Invariant : leve en nommant tout segment original non traite.
    fn verifier_couverture(racines: &[&str], recette: &[(String, Recette)]) -> Result<(), String> {
        let mut manquants: Vec<String> = Vec::new();
        for sid in racines {
            match couverture(sid, recette, &mut Vec::new()) {
                Ok(abs) => manquants.extend(abs),
                Err(msg) => return Err(msg),
            }
        }
        if !manquants.is_empty() {
            let liste: Vec<&str> = manquants.iter().take(6).map(|s| s.as_str()).collect();
            return Err(format!(
                "{} segment(s) sans traitement final : {}",
                manquants.len(),
                py_list_str(&liste)
            ));
        }
        Ok(())
    }

    /// Operation inversible : un segment macro -> des fragments qualifies.
    fn decouper(seg: &Segment, mode: &str) -> Vec<Segment> {
        let texte = match String::from_utf8(seg.data.clone()) {
            Ok(t) => t,
            Err(_) => return Vec::new(),
        };
        let mut kids: Vec<Segment> = Vec::new();
        match mode {
            "blocs_markdown" => {
                let mut bornes: Vec<usize> = vec![0];
                bornes.extend(positions_titres(&texte));
                bornes.push(texte.len());
                for i in 0..bornes.len() - 1 {
                    let m = &texte[bornes[i]..bornes[i + 1]];
                    if !m.trim().is_empty() {
                        kids.push(Segment {
                            id: format!("{}.b{}", seg.id, kids.len()),
                            data: m.as_bytes().to_vec(),
                            genre: format!("{}/bloc", seg.genre),
                            parent: Some(seg.id.clone()),
                            profondeur: seg.profondeur + 1,
                        });
                    }
                }
            }
            "colonnes_csv" => {
                let lignes: Vec<&str> = texte.lines().filter(|l| !l.is_empty()).collect();
                if lignes.is_empty() { return kids; }
                let cols: Vec<&str> = lignes[0].split(',').collect();
                let mat: Vec<Vec<&str>> = lignes.iter().map(|l| l.split(',').collect()).collect();
                for j in 0..cols.len() {
                    let mut s = String::new();
                    for (r, row) in mat.iter().enumerate() {
                        if r > 0 { s.push('\n'); }
                        s.push_str(row[j]);
                    }
                    kids.push(Segment {
                        id: format!("{}.c{}", seg.id, j),
                        data: s.into_bytes(),
                        genre: format!("{}/col:{}", seg.genre, cols[j]),
                        parent: Some(seg.id.clone()),
                        profondeur: seg.profondeur + 1,
                    });
                }
            }
            "cles_json" => {
                let champs = match json_parse(&texte) {
                    Some(Jv::Obj(f)) => f,
                    _ => return kids,
                };
                for (k, v) in &champs {
                    let mut s = String::new();
                    json_dump(v, 0, &mut s);
                    kids.push(Segment {
                        id: format!("{}.k{}", seg.id, kids.len()),
                        data: s.into_bytes(),
                        genre: format!("{}/cle:{}", seg.genre, k),
                        parent: Some(seg.id.clone()),
                        profondeur: seg.profondeur + 1,
                    });
                }
            }
            _ => {}
        }
        kids
    }

    /// L'inverse d'un decoupage : reconstruit les octets du segment parent.
    fn recoller(mode: &str, enfants: &[Segment]) -> Vec<u8> {
        match mode {
            "blocs_markdown" => {
                let mut out = Vec::new();
                for e in enfants { out.extend_from_slice(&e.data); }
                out
            }
            "colonnes_csv" => {
                let mat: Vec<Vec<String>> = enfants
                    .iter()
                    .map(|e| String::from_utf8_lossy(&e.data).split('\n').map(|s| s.to_string()).collect())
                    .collect();
                if mat.is_empty() { return Vec::new(); }
                let n = mat[0].len();
                let mut lignes: Vec<String> = Vec::new();
                for row in 0..n {
                    let parts: Vec<&str> = mat.iter().map(|c| c[row].as_str()).collect();
                    lignes.push(parts.join(","));
                }
                let mut s = lignes.join("\n");
                s.push('\n');
                s.into_bytes()
            }
            "cles_json" => {
                let mut champs: Vec<(String, Jv)> = Vec::new();
                for e in enfants {
                    let k = e.genre.rsplit(':').next().unwrap_or("").to_string();
                    let v = json_parse(&String::from_utf8_lossy(&e.data)).unwrap_or(Jv::Null);
                    champs.push((k, v));
                }
                let mut s = String::new();
                json_dump(&Jv::Obj(champs), 0, &mut s);
                s.push('\n');
                s.into_bytes()
            }
            _ => Vec::new(),
        }
    }

    /// Cerveau pathologique : decoupe TOUJOURS, ne produit jamais de feu final.
    ///
    /// Le parcours ne s'epuise jamais : c'est precisement le cas que la safety
    /// bound doit arreter, au lieu de laisser l'agent tourner a l'infini.
    struct CerveauDivergent { appels: i64 }

    impl Cerveau for CerveauDivergent {
        fn inferer(&mut self, _seg: &Segment) -> Result<Decision, String> {
            self.appels += 1;
            Ok(Decision::Decouper { mode: "blocs_markdown".to_string() })
        }
    }

    let mut cerveau = CerveauDivergent { appels: 0 };
    let mut divergent = AgentTriage {
        cerveau: &mut cerveau,
        borne: 8,
        catalogue: Vec::new(),
        recette: Vec::new(),
        appel_decouverte: 0,
    };
    // Texte SANS en-tete : blocs_markdown ne produit qu'un seul bloc = le
    // segment lui-meme, re-decoupe a l'identique -> le parcours ne s'epuise
    // jamais.
    let boucle = vec![Segment {
        id: "boucle.txt#0".to_string(),
        data: b"texte sans en-tete, a decouper a l'infini\n".to_vec(),
        genre: "texte".to_string(),
        parent: None,
        profondeur: 0,
    }];
    match divergent.decouvrir(&boucle) {
        Ok(()) => println!("BUG : la divergence n'a PAS ete arretee"),
        Err(e) => println!("SAFETY BOUND : la decouverte divergente est arretee -> {}", e),
    }
};

SAFETY BOUND : la decouverte divergente est arretee -> decouverte interrompue a 8 inférences, 1 segment(s) en attente — safety bound.


### Lecture du résultat

Le cerveau divergent découpe indéfiniment (un bloc sans en-tête se « découpe » en
un seul bloc identique) : sans borne, l'agent tournerait pour toujours. Le
squelette l'**arrête à 8 inférences** et signale l'échec de la découverte. C'est
la contrepartie de la composition : on accepte de brancher n'importe quel
mécanisme d'inférence *parce que* le squelette porte les deux garde-fous — la
**vérification** de complétude (section 7) et la **safety bound** (ici). Sans
eux, « composable » ne serait que « instable ».

## 9. Le second cerveau — le LLM, point de composition

On montre maintenant concrètement le **point de composition** : le *même*
`AgentTriage` accepte un cerveau **LLM** (Semantic Kernel, tel que
Compression-07b l'emploie) aussi bien que le cerveau déterministe, parce qu'il ne
voit que l'interface `Cerveau`. Le cerveau LLM est **déclaré, pas invoqué** ici —
le carnet reste reproductible sans clé API — et c'est exactement le propos : le
cerveau est une *pièce* amovible, pas le squelette.

In [11]:
{
    /// Representation commune d'un morceau de donnee (reutilisee de Compression-07b).
    ///
    /// L'objet que la boucle percoit et represente : `id` stable, `data` = octets,
    /// `genre` = qualification, `parent`/`profondeur` = lien vers l'origine.
    #[derive(Clone)]
    struct Segment {
        id: String,
        data: Vec<u8>,
        genre: String,
        parent: Option<String>,
        profondeur: i64,
    }

    /// Signature de structure deduite du CONTENU — la perception du segment.
    ///
    /// C'est ce que le cerveau regarde pour decider. Deduite, pas etiquettee :
    /// un fichier nomme `.json` qui ne parse pas n'est pas du JSON.
    struct Info {
        octets: usize,
        texte: bool,
        csv: bool,
        blocs_md: usize,
        json: bool,
        json_obj: bool,
    }

    /// Les booleens Rust s'impriment `true`/`false` ; le jumeau Python impose
    /// `True`/`False`. Toute comparaison de texte passe par cette fonction.
    fn py(b: bool) -> &'static str {
        if b { "True" } else { "False" }
    }

    /// JSON : mini-dumpeur et mini-parseur maison (`std` seule, aucune crate).
    ///
    /// L'ordre d'insertion des cles d'un objet est PRESERVE (`Vec<(String, Jv)>`) :
    /// le catalogue amorti s'appuie sur cet ordre, un `BTreeMap` le casserait.
    #[derive(Clone)]
    enum Jv {
        Null,
        Bool(bool),
        Int(i64),
        Str(String),
        List(Vec<Jv>),
        Obj(Vec<(String, Jv)>),
    }

    fn json_echapper(s: &str, out: &mut String) {
        out.push('"');
        for c in s.chars() {
            match c {
                '"' => { out.push('\\'); out.push('"'); }
                '\\' => { out.push('\\'); out.push('\\'); }
                '\n' => out.push_str("\\n"),
                '\r' => out.push_str("\\r"),
                '\t' => out.push_str("\\t"),
                c if (c as u32) < 0x20 => out.push_str(&format!("\\u{:04x}", c as u32)),
                c => out.push(c),
            }
        }
        out.push('"');
    }

    /// Dumps equivalent a `json.dumps(indent=2, ensure_ascii=False)`.
    fn json_dump(v: &Jv, level: usize, out: &mut String) {
        let pad = "  ".repeat(level);
        let pad_in = "  ".repeat(level + 1);
        match v {
            Jv::Null => out.push_str("null"),
            Jv::Bool(b) => out.push_str(if *b { "true" } else { "false" }),
            Jv::Int(i) => out.push_str(&i.to_string()),
            Jv::Str(s) => json_echapper(s, out),
            Jv::List(items) => {
                if items.is_empty() { out.push_str("[]"); return; }
                out.push_str("[\n");
                for (i, it) in items.iter().enumerate() {
                    out.push_str(&pad_in);
                    json_dump(it, level + 1, out);
                    if i + 1 < items.len() { out.push(','); }
                    out.push('\n');
                }
                out.push_str(&pad);
                out.push(']');
            }
            Jv::Obj(fields) => {
                if fields.is_empty() { out.push_str("{}"); return; }
                out.push_str("{\n");
                for (i, (k, val)) in fields.iter().enumerate() {
                    out.push_str(&pad_in);
                    json_echapper(k, out);
                    out.push_str(": ");
                    json_dump(val, level + 1, out);
                    if i + 1 < fields.len() { out.push(','); }
                    out.push('\n');
                }
                out.push_str(&pad);
                out.push('}');
            }
        }
    }

    fn json_skip_ws(b: &[u8], i: &mut usize) {
        while *i < b.len() && (b[*i] as char).is_whitespace() { *i += 1; }
    }

    fn json_chaine(b: &[u8], i: &mut usize) -> Option<String> {
        if *i >= b.len() || b[*i] != b'"' { return None; }
        *i += 1;
        let mut s = String::new();
        while *i < b.len() {
            let c = b[*i];
            if c == b'"' { *i += 1; return Some(s); }
            if c == b'\\' {
                *i += 1;
                if *i >= b.len() { return None; }
                let e = b[*i];
                *i += 1;
                match e {
                    b'n' => s.push('\n'),
                    b't' => s.push('\t'),
                    b'r' => s.push('\r'),
                    b'b' => s.push('\u{8}'),
                    b'f' => s.push('\u{c}'),
                    b'/' => s.push('/'),
                    b'"' => s.push('"'),
                    b'\\' => s.push('\\'),
                    b'u' => {
                        if *i + 4 > b.len() { return None; }
                        let hex = std::str::from_utf8(&b[*i..*i + 4]).ok()?;
                        let code = u32::from_str_radix(hex, 16).ok()?;
                        *i += 4;
                        s.push(char::from_u32(code).unwrap_or('\u{fffd}'));
                    }
                    _ => return None,
                }
            } else {
                let start = *i;
                while *i < b.len() && b[*i] != b'"' && b[*i] != b'\\' { *i += 1; }
                s.push_str(std::str::from_utf8(&b[start..*i]).ok()?);
            }
        }
        None
    }

    fn json_valeur(b: &[u8], i: &mut usize) -> Option<Jv> {
        json_skip_ws(b, i);
        if *i >= b.len() { return None; }
        match b[*i] {
            b'{' => {
                *i += 1;
                let mut fields: Vec<(String, Jv)> = Vec::new();
                json_skip_ws(b, i);
                if *i < b.len() && b[*i] == b'}' { *i += 1; return Some(Jv::Obj(fields)); }
                loop {
                    json_skip_ws(b, i);
                    if *i >= b.len() || b[*i] != b'"' { return None; }
                    let k = json_chaine(b, i)?;
                    json_skip_ws(b, i);
                    if *i >= b.len() || b[*i] != b':' { return None; }
                    *i += 1;
                    let v = json_valeur(b, i)?;
                    fields.push((k, v));
                    json_skip_ws(b, i);
                    if *i >= b.len() { return None; }
                    match b[*i] {
                        b',' => { *i += 1; }
                        b'}' => { *i += 1; return Some(Jv::Obj(fields)); }
                        _ => return None,
                    }
                }
            }
            b'[' => {
                *i += 1;
                let mut items: Vec<Jv> = Vec::new();
                json_skip_ws(b, i);
                if *i < b.len() && b[*i] == b']' { *i += 1; return Some(Jv::List(items)); }
                loop {
                    let v = json_valeur(b, i)?;
                    items.push(v);
                    json_skip_ws(b, i);
                    if *i >= b.len() { return None; }
                    match b[*i] {
                        b',' => { *i += 1; }
                        b']' => { *i += 1; return Some(Jv::List(items)); }
                        _ => return None,
                    }
                }
            }
            b'"' => Some(Jv::Str(json_chaine(b, i)?)),
            b't' => {
                if b.len() >= *i + 4 && &b[*i..*i + 4] == b"true" { *i += 4; return Some(Jv::Bool(true)); }
                None
            }
            b'f' => {
                if b.len() >= *i + 5 && &b[*i..*i + 5] == b"false" { *i += 5; return Some(Jv::Bool(false)); }
                None
            }
            b'n' => {
                if b.len() >= *i + 4 && &b[*i..*i + 4] == b"null" { *i += 4; return Some(Jv::Null); }
                None
            }
            _ => {
                let start = *i;
                if b[*i] == b'-' { *i += 1; }
                while *i < b.len() && b[*i].is_ascii_digit() { *i += 1; }
                if start == *i { return None; }
                let txt = std::str::from_utf8(&b[start..*i]).ok()?;
                txt.parse::<i64>().ok().map(Jv::Int)
            }
        }
    }

    /// Parse un document JSON complet ; `None` si le texte n'est pas du JSON.
    fn json_parse(texte: &str) -> Option<Jv> {
        let b = texte.as_bytes();
        let mut i = 0usize;
        let v = json_valeur(b, &mut i)?;
        json_skip_ws(b, &mut i);
        if i != b.len() { return None; }
        Some(v)
    }

    /// Positions (en octets) des titres de markdown : meme regex que le jumeau
    /// Python (`^##? ` en mode multiligne), donc memes decoupes.
    fn positions_titres(texte: &str) -> Vec<usize> {
        let mut out = Vec::new();
        let mut off = 0usize;
        for l in texte.split_inclusive('\n') {
            let t = l.trim_end_matches('\n').trim_end_matches('\r');
            if t.starts_with("# ") || t.starts_with("## ") { out.push(off); }
            off += l.len();
        }
        out
    }

    /// Perception : la signature de structure deduite du contenu.
    fn structure_interne(seg: &Segment) -> Info {
        let mut info = Info {
            octets: seg.data.len(),
            texte: false,
            csv: false,
            blocs_md: 0,
            json: false,
            json_obj: false,
        };
        let texte = match String::from_utf8(seg.data.clone()) {
            Ok(t) => t,
            Err(_) => return info,
        };
        info.texte = true;
        let t = texte.trim();
        if t.starts_with('{') || t.starts_with('[') {
            if let Some(v) = json_parse(&texte) {
                info.json_obj = matches!(v, Jv::Obj(_));
                info.json = true;
                return info;
            }
        }
        let lignes: Vec<&str> = texte.lines().filter(|l| !l.is_empty()).collect();
        if !lignes.is_empty() && lignes[0].contains(',') {
            let n = lignes[0].matches(',').count();
            info.csv = lignes.iter().all(|l| l.matches(',').count() == n);
        }
        info.blocs_md = positions_titres(&texte).len();
        info
    }

    /// Classe de forme d'un segment — la cle du catalogue amorti.
    ///
    /// Distingue l'objet JSON (decoupable par ses cles) du tableau JSON
    /// (compresse tel quel) : deux formes, donc deux decisions distinctes.
    fn signature(s: &Info) -> String {
        if s.json {
            return if s.json_obj { "json_obj".to_string() } else { "json".to_string() };
        }
        if s.csv { return "csv".to_string(); }
        if s.blocs_md > 0 { return "markdown".to_string(); }
        if s.texte { "texte".to_string() } else { "binaire".to_string() }
    }

    /// Un petit corpus heterogene, genere deterministiquement : meme entree a
    /// chaque execution, donc resultat reproductible.
    fn corpus_demo() -> Vec<Segment> {
        let ventes = "nom,quantite,prix\ncafe,10,2.50\nthe,20,1.80\nsucre,5,0.90\nlait,8,1.10\npain,12,0.80\n";
        let clients_obj = Jv::Obj(vec![
            ("id".to_string(), Jv::Int(7)),
            ("panier".to_string(), Jv::List(vec![
                Jv::Obj(vec![
                    ("sku".to_string(), Jv::Str("cafe".to_string())),
                    ("q".to_string(), Jv::Int(10)),
                ]),
                Jv::Obj(vec![
                    ("sku".to_string(), Jv::Str("the".to_string())),
                    ("q".to_string(), Jv::Int(20)),
                ]),
            ])),
        ]);
        let mut clients = String::new();
        json_dump(&clients_obj, 0, &mut clients);
        clients.push('\n');
        let notes = "# Rapport\n\n## Chiffre\n12 kEUR, en hausse.\n\n## Risque\nFournisseur unique, lead time 6 semaines.\n";
        let journal: String = (0..40)
            .map(|i| format!("10:{:02}:00 INFO requete {} ok\n", i, i))
            .collect();
        let mut binaire: Vec<u8> = Vec::with_capacity(1024);
        for _ in 0..4 {
            for b in 0u16..256 { binaire.push(b as u8); }
        }
        vec![
            Segment {
                id: "ventes.csv#0".to_string(),
                data: ventes.as_bytes().to_vec(),
                genre: "donnees/csv".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "clients.json#0".to_string(),
                data: clients.into_bytes(),
                genre: "donnees/json".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "notes.md#0".to_string(),
                data: notes.as_bytes().to_vec(),
                genre: "texte/markdown".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "journal.log#0".to_string(),
                data: journal.into_bytes(),
                genre: "texte/log".to_string(),
                parent: None,
                profondeur: 0,
            },
            Segment {
                id: "brut.bin#0".to_string(),
                data: binaire,
                genre: "binaire".to_string(),
                parent: None,
                profondeur: 0,
            },
        ]
    }

    /// La decision d'un cerveau : decouper (reduire) ou compresser.
    ///
    /// Un `enum` remplace le `dict` du jumeau Python : les deux cas sont
    /// nommes, donc le compilateur refuse le cas forget.
    #[derive(Clone)]
    enum Decision {
        Decouper { mode: String },
        Compresser { codec: String },
    }

    impl Decision {
        fn op(&self) -> &'static str {
            match self {
                Decision::Decouper { .. } => "decouper",
                Decision::Compresser { .. } => "compresser",
            }
        }
        fn quoi(&self) -> &str {
            match self {
                Decision::Decouper { mode } => mode,
                Decision::Compresser { codec } => codec,
            }
        }
    }

    /// Point de composition : le contrat d'un mecanisme d'inference.
    ///
    /// La boucle agentique n'appelle JAMAIS un cerveau en particulier, seulement
    /// ce trait. C'est ce contrat unique qui rend le cerveau interchangeable
    /// (deterministe, LLM, hybride). Chaque consultation compte comme une inference.
    trait Cerveau {
        fn inferer(&mut self, seg: &Segment) -> Result<Decision, String>;
    }

    /// Point de composition : un LLM derriere la MEME interface `Cerveau`.
    ///
    /// En production, `inferer` construirait le kernel Semantic Kernel (un
    /// kernel neuf par decision, cf Compression-07b) et lirait l'appel de
    /// fonction compresser/decouper. Ici il est DECLARE, pas invoque : le
    /// carnet reste reproductible sans cle, et c'est precisement le point
    /// illustre — le cerveau est une piece, pas le squelette.
    struct CerveauLLM { appels: i64 }

    impl Cerveau for CerveauLLM {
        fn inferer(&mut self, _seg: &Segment) -> Result<Decision, String> {
            Err("CerveauLLM.inferer n'est pas cablé dans ce carnet (pas de cle API)".to_string())
        }
    }

    /// Le cerveau par defaut, pour la comparaison des deux cotes.
    struct CerveauDeterministe { appels: i64 }

    impl Cerveau for CerveauDeterministe {
        fn inferer(&mut self, seg: &Segment) -> Result<Decision, String> {
            self.appels += 1;
            let s = structure_interne(seg);
            if seg.data.len() < 64 {
                return Ok(Decision::Compresser {
                    codec: if s.texte { "zlib".to_string() } else { "lzma".to_string() },
                });
            }
            let mode = match signature(&s).as_str() {
                "json_obj" => "cles_json",
                "csv" => "colonnes_csv",
                "markdown" => "blocs_markdown",
                _ => {
                    return Ok(Decision::Compresser {
                        codec: if s.texte { "zlib".to_string() } else { "lzma".to_string() },
                    })
                }
            };
            Ok(Decision::Decouper { mode: mode.to_string() })
        }
    }

    /// Le test du point de composition : une fonction qui accepte n'importe
    /// quel `&mut dyn Cerveau` ne peut etre appelee QUE par un cerveau qui
    /// implemente le trait — le test de la version Python (`isinstance`)
    /// devient une contrainte de type.
    fn accepte_cerveau(_c: &mut dyn Cerveau) -> bool { true }

    // Le point de composition, concret : le meme squelette accepte les deux
    // cerveaux, car il ne teste que l'interface Cerveau.
    let mut det = CerveauDeterministe { appels: 0 };
    let mut llm = CerveauLLM { appels: 0 };
    println!(
        "   AgentTriage accepte un cerveau {:14} : {}",
        "deterministe",
        py(accepte_cerveau(&mut det))
    );
    println!(
        "   AgentTriage accepte un cerveau {:14} : {}",
        "LLM",
        py(accepte_cerveau(&mut llm))
    );
    println!();
    println!("la boucle est agnostique du mecanisme d'inference : c'est la composition.");
    println!("le resultat experimental ci-dessus a ete produit par le cerveau determine ;");
    println!("brancher CerveauLLM changerait le cout, pas le squelette ni les garde-fous.");
};

   AgentTriage accepte un cerveau deterministe   : True


   AgentTriage accepte un cerveau LLM            : True


la boucle est agnostique du mecanisme d'inference : c'est la composition.


le resultat experimental ci-dessus a ete produit par le cerveau determine ;


brancher CerveauLLM changerait le cout, pas le squelette ni les garde-fous.


### Lecture du résultat

Le même squelette accepte les deux cerveaux — le test d'acceptation passe pour les
deux, et c'est tout ce que la boucle exige. En Rust, ce test est une **contrainte
de type** : une fonction qui attend un `&mut dyn Cerveau` ne peut être appelée que
par un cerveau qui implémente le trait. La composition est
réelle : **brancher un LLM change le coût d'inférence, pas le squelette, pas la
vérification, pas la safety bound.** Le résultat expérimental de ce carnet a été
produit par le cerveau déterministe (reproductible, sans clé) ; en production on
y brancherait `CerveauLLM`, et les sections 6 à 8 garderaient leur sens à
l'identique.

## 10. Exercices

Trois exercices, un par capacité du squelette : étendre la **perception**, ajouter
une **opération inversible**, et exploiter le **point de composition**.

### Exercice 1 — étendre la perception

Le corpus ne détecte que le CSV (virgules). Ajoutez la détection du **TSV**
(colonnes séparées par des tabulateurs) dans `structure_interne`, puis une
signature `"tsv"` dans `signature()` (avant `"csv"`).

- `# Etape 1` : dans `structure_interne`, poser `info.tsv` à `true` quand chaque ligne non vide partage le même nombre de tabulations que la première.
- `# Etape 2` : dans `signature`, renvoyer `"tsv"` avant le test `csv`.
- `# Indice` : un TSV a des tabulations, un CSV a des `','` — même logique, autre séparateur.

In [12]:
{
    /// Representation commune d'un morceau de donnee (reutilisee de Compression-07b).
    ///
    /// L'objet que la boucle percoit et represente : `id` stable, `data` = octets,
    /// `genre` = qualification, `parent`/`profondeur` = lien vers l'origine.
    #[derive(Clone)]
    struct Segment {
        id: String,
        data: Vec<u8>,
        genre: String,
        parent: Option<String>,
        profondeur: i64,
    }

    /// Signature de structure deduite du CONTENU — la perception du segment.
    ///
    /// C'est ce que le cerveau regarde pour decider. Deduite, pas etiquettee :
    /// un fichier nomme `.json` qui ne parse pas n'est pas du JSON.
    struct Info {
        octets: usize,
        texte: bool,
        csv: bool,
        blocs_md: usize,
        json: bool,
        json_obj: bool,
    }

    // TODO etudiant : renvoyer une signature de structure qui detecte le TSV
    // (colonnes separees par des tabulateurs), comme structure_interne le
    // fait pour le CSV. Doit renvoyer une Info avec au moins le champ
    // "tsv" (bool).
    let structure_tsv = |_seg: &Segment| -> Option<Info> { None }; // TODO etudiant
    let segment = Segment {
        id: "ventes.csv#0".to_string(),
        data: Vec::new(),
        genre: "donnees/csv".to_string(),
        parent: None,
        profondeur: 0,
    };

    if structure_tsv(&segment).is_none() {
        println!("Exercice 1 a completer : detection du TSV dans la perception");
    }
};

Exercice 1 a completer : detection du TSV dans la perception


### Exercice 2 — une opération inversible

Écrivez un codec **RLE** (run-length : `(octet, compte)`) et enregistrez-le dans
la table des codecs. Le contrat d'inversibilité doit tenir :
`decompresser(compresser(d)) == d`.

- `# Etape 1` : `compresser_rle` parcourt `donnees` et émet des paires `(octet, compte)` pour chaque run consécutive.
- `# Etape 2` : `decompresser_rle` lit les paires et répète chaque octet `compte` fois.
- `# Indice` : `"aaabbb"` → `[(a,3),(b,3)]` ; attention au run de longueur 1.

In [13]:
{
    // TODO etudiant : encoder les runs consecutives en paires (octet, compte)
    let compresser_rle = |_donnees: &[u8]| -> Option<Vec<u8>> { None }; // TODO etudiant
    // TODO etudiant : reconstruire les octets d'origine depuis les paires
    let decompresser_rle = |_donnees: &[u8]| -> Option<Vec<u8>> { None }; // TODO etudiant

    if compresser_rle(&b"aaab"[..]).is_none() {
        println!("Exercice 2 a completer : codec RLE inversible");
    }
};

Exercice 2 a completer : codec RLE inversible


### Exercice 3 — le point de composition

Écrivez un `CerveauHybride` qui **route** : les petits segments (moins d'un seuil
d'octets) sont confiés au cerveau déterministe, les gros renvoient une escalade
(à traiter par un LLM en production). Comptez les appels comme les autres cerveaux.

- `# Etape 1` : dans `inferer`, si la taille du segment est sous le seuil, déléguer à `self.petit.inferer(seg)`.
- `# Etape 2` : sinon renvoyer la décision d'escalade et compter l'appel.
- `# Indice` : le hybride *contient* un cerveau — c'est une composition de compositions.

In [14]:
{
    // CerveauHybride : stub d'exercice.
    // TODO etudiant : ecrire un cerveau qui route selon la taille du segment —
    // les petits segments sont confies au cerveau deterministe, les gros
    // renvoient une escalade (a traiter par un LLM en production). Comptez
    // les appels comme les autres cerveaux.
    println!("Exercice 3 a completer : cerveau hybride (routing petit/gros)");
};

Exercice 3 a completer : cerveau hybride (routing petit/gros)


## 11. À retenir

- **Un agent composable = un squelette + des pièces.** La boucle
  percevoir→représenter→inférer→décider→vérifier→exécuter→observer est le
  squelette ; l'inférence est une *pièce* derrière une interface.
- **L'inférence est un point de composition.** Déterministe, LLM, hybride : le
  squelette, la vérification et la safety bound ne changent pas — seul le **coût**
  d'inférence change.
- **La connaissance s'amortit.** Découvrir une politique `forme → décision` coûte des
  inférences une fois ; la rejouer en coûte zéro. La connaissance est un actif.
- **Composer exige des garde-fous.** Un mécanisme d'inférence ne se branche que
  derrière une **vérification de complétude** (l'invariant de couverture rejette un
  plan à moitié formé) et une **safety bound** (on n'arrête pas une découverte
  divergente).
- **Les primitives se réutilisent.** La représentation, l'invariant et la boucle
  bornée viennent de Compression-07b ; ce carnet les *compose*, il ne les réinvente
  pas.

## Vérifiez votre compréhension

1. Pourquoi le stade **Vérifier** est-il placé *dans* la découverte, et non après
   l'exécution ? *(Réponse repliée ci-dessous.)*
2. Que mesure exactement le compteur `appel_decouverte`, et pourquoi est-il de 0
   pendant `rejouer` ?
3. Quel est le risque précis que la **safety bound** empêche, et qu'arriverait-il
   sans elle sur un cerveau divergent ?

<details><summary>Réponse 1</summary>
Parce qu'un plan incomplet ne doit **jamais** être exécuté : la vérification est
la *porte d'admission* du plan. Si elle était après l'exécution, on aurait déjà
transformé des données avec un plan à moitié formé — trop tard pour rejeter.
</details>

<details><summary>Réponse 2</summary>
Il compte les **consultations du cerveau** (les véritables inférences). Pendant
`rejouer`, le catalogue `forme → décision` est *consulté* (une table), pas le
cerveau — d'où 0. C'est tout l'amortissement.
</details>

<details><summary>Réponse 3</summary>
Un cerveau qui découpe toujours sans produire de traitement final fait croître la
file indéfiniment : sans borne, l'agent **tourne à l'infini** (temps et coût sans
fin). La safety bound arrête la découverte et signale l'échec.
</details>

## Pour aller plus loin

- [`Compression-07b`](../../InformationCalcul/Compression/Compression-07b-Triage-LLM-Semantic-Kernel-Python.ipynb) —
  l'organe dont ce carnet réutilise les primitives (le triage LLM complet, le
  rejeu sans LLM, la réfutation « le triage bat-il l'indifférencié ? »).
- Le jumeau Python de ce carnet :
  [`AgentSystems-01-Perception-Vers-Action-Python`](AgentSystems-01-Perception-Vers-Action-Python.ipynb).
- L'EPIC #26930 — la suite : extraire ces primitives en **module importable**
  (le pas suivant nommé), puis composer un deuxième système (perception → action
  sur un autre domaine) pour montrer que le squelette est vraiment réutilisable.